In [1]:
import sys
print(sys.executable)

C:\Users\suraj\poppy_env\Scripts\python.exe


In [2]:
import mediapipe as mp

mp_hands = mp.solutions.hands
print("Working ✅")

Working ✅


In [ ]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands()

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        print("Hand detected")

    cv2.imshow("cam", frame)
    if cv2.waitKey(1) == 27:
        break

Hand detected
Hand detected
Hand detected
Hand detected


In [ ]:
def is_fist(landmarks):
    return landmarks.landmark[8].y > landmarks.landmark[6].y

In [ ]:
import cv2
import mediapipe as mp
from pypot.dynamixel import DxlIO

# Motor setup
dxl = DxlIO('COM4')
print(dxl.scan())

# Vision setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands()

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            lm = hand.landmark

            # Simple logic
            if lm[8].y > lm[6].y:
                print("FIST")
                dxl.set_goal_position({54: 0})
            else:
                print("OPEN")
                dxl.set_goal_position({54: 90})

    cv2.imshow("cam", frame)

    if cv2.waitKey(1) == 27:
        break

[11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
OPEN
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST
FIST


In [ ]:
import cv2
import mediapipe as mp
import time
from pypot.dynamixel import DxlIO

# ================= MOTOR SETUP =================
dxl = DxlIO('COM4', baudrate=1000000)
ids = dxl.scan()
print("Detected motors:", ids)

if not ids:
    print("No motors found! Check connection.")
    exit()

dxl.enable_torque(ids)

# ================= MEDIAPIPE =================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7)

cap = cv2.VideoCapture(0)

last_gesture = None
last_time = 0

# ================= GESTURE FUNCTION =================
def detect_gesture(lm):
    fingers = []

    # Thumb
    fingers.append(lm[4].x < lm[3].x)

    # Other fingers
    fingers.append(lm[8].y < lm[6].y)
    fingers.append(lm[12].y < lm[10].y)
    fingers.append(lm[16].y < lm[14].y)
    fingers.append(lm[20].y < lm[18].y)

    total = fingers.count(True)

    if total == 0:
        return "FIST"
    elif total == 5:
        return "OPEN"
    elif total == 2:
        return "TWO"

    # Direction-based
    if lm[8].x > 0.7:
        return "RIGHT"
    elif lm[8].x < 0.3:
        return "LEFT"

    return "UNKNOWN"

# ================= ACTIONS =================

def stop():
    print("STOP")
    dxl.set_goal_position({
        52: 0, 54: -20,
        42: 0, 44: -20
    })

def move():
    print("MOVE")
    dxl.set_goal_position({
        52: 30,
        51: -20
    })

def wave_right():
    print("WAVE RIGHT")
    for _ in range(3):
        dxl.set_goal_position({52: 30, 54: -40})
        time.sleep(0.3)
        dxl.set_goal_position({52: -30, 54: -20})
        time.sleep(0.3)

def wave_left():
    print("WAVE LEFT")
    for _ in range(3):
        dxl.set_goal_position({42: -30, 44: -40})
        time.sleep(0.3)
        dxl.set_goal_position({42: 30, 44: -20})
        time.sleep(0.3)

def dance():
    print("DANCE")
    for _ in range(3):
        dxl.set_goal_position({52: 30, 42: -30})
        time.sleep(0.3)
        dxl.set_goal_position({52: -30, 42: 30})
        time.sleep(0.3)

# ================= MAIN LOOP =================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            lm = hand.landmark

            gesture = detect_gesture(lm)

            # Prevent repeated triggering
            if gesture != last_gesture and time.time() - last_time > 1:
                last_gesture = gesture
                last_time = time.time()

                print("Gesture:", gesture)

                if gesture == "OPEN":
                    stop()
                elif gesture == "FIST":
                    move()
                elif gesture == "RIGHT":
                    wave_right()
                elif gesture == "LEFT":
                    wave_left()
                elif gesture == "TWO":
                    dance()

    cv2.imshow("Poppy Gesture Controller", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

Detected motors: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
Gesture: OPEN
STOP
Gesture: TWO
DANCE
Gesture: FIST
MOVE
Gesture: TWO
DANCE
Gesture: LEFT
WAVE LEFT
Gesture: TWO
DANCE
Gesture: LEFT
WAVE LEFT
Gesture: FIST
MOVE
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: TWO
DANCE
Gesture: FIST
MOVE
Gesture: UNKNOWN
Gesture: TWO
DANCE
Gesture: LEFT
WAVE LEFT
Gesture: TWO
DANCE
Gesture: LEFT
WAVE LEFT
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: TWO
DANCE
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: TWO
DANCE
Gesture: LEFT
WAVE LEFT
Gesture: UNKNOWN
Gesture: LEFT
WAVE LEFT
Gesture: UNKNOWN
Gesture: TWO
DANCE
Gesture: RIGHT
WAVE RIGHT
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: OPEN
STOP
Gesture: UNKNOWN
Gesture: FIST
MOVE
Gesture: TWO
DANCE
Gesture: FIST
MOVE
Gesture: RIGHT
WAVE RIGHT
Gesture: TWO
DANCE
Gesture: RIGHT
WAVE RIGH

In [ ]:
import cv2
import mediapipe as mp
import time
from pypot.dynamixel import DxlIO

# ================= CONFIG =================
PORT = 'COM4'          # change only if needed
BAUDRATE = 1000000

# ================= CONNECT MOTORS =================
dxl = DxlIO(PORT, baudrate=BAUDRATE)
ids = dxl.scan()
print("Detected motors:", ids)

if not ids:
    print("❌ No motors detected. Check connection.")
    exit()

dxl.enable_torque(ids)

# ================= MEDIAPIPE SETUP =================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

# ================= HELPER FUNCTIONS =================
def map_range(val, in_min=0, in_max=1, out_min=-60, out_max=60):
    return (val - in_min) * (out_max - out_min) / (in_max - in_min) + out_min

def clamp(val, min_v, max_v):
    return max(min_v, min(max_v, val))

# ================= SMOOTHING =================
prev_x, prev_y = 0.5, 0.5
alpha = 0.7  # smoothing factor

# ================= MAIN LOOP =================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        hand = result.multi_hand_landmarks[0]
        lm = hand.landmark

        # ================= HAND POSITION =================
        x = lm[8].x   # index finger tip
        y = lm[8].y

        # ================= SMOOTHING =================
        x = alpha * x + (1 - alpha) * prev_x
        y = alpha * y + (1 - alpha) * prev_y
        prev_x, prev_y = x, y

        # ================= ARM CONTROL =================
        shoulder_x = map_range(x, 0, 1, -60, 60)
        shoulder_y = map_range(y, 0, 1, -60, 60)
        elbow = map_range(y, 0, 1, -20, -80)
        arm_z = shoulder_x / 2

        # ================= SAFETY CLAMP =================
        shoulder_x = clamp(shoulder_x, -60, 60)
        shoulder_y = clamp(shoulder_y, -60, 60)
        elbow = clamp(elbow, -90, -10)

        # ================= SEND TO MOTORS =================
        dxl.set_goal_position({
            52: int(shoulder_x),   # r_shoulder_x
            51: int(shoulder_y),   # r_shoulder_y
            53: int(arm_z),        # r_arm_z
            54: int(elbow)         # r_elbow
        })

    # ================= DISPLAY =================
    cv2.imshow("Right Arm Follow", frame)

    if cv2.waitKey(1) == 27:
        break

    # ================= SMOOTH DELAY =================
    time.sleep(0.03)

# ================= CLEANUP =================
cap.release()
cv2.destroyAllWindows()

Detected motors: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]


In [ ]:
"""
Poppy Humanoid — Optimised Hand-Tracking Right Arm Control
──────────────────────────────────────────────────────────
• Tracks palm centre (avg of all landmarks) — far stabler than index tip alone
• Adaptive EMA smoothing — fast when hand moves fast, smooth when still
• Threaded motor writes — camera loop never blocks on serial I/O
• Dead-zone filter — kills micro-jitter without adding lag
• Full overlay HUD with live angle readouts
"""

import cv2
import mediapipe as mp
import threading
import time
import math
import numpy as np
from pypot.dynamixel import DxlIO

# ══════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════
PORT        = 'COM4'
BAUDRATE    = 1000000

# Smoothing: base alpha (0 = max smooth/laggy, 1 = no smooth/reactive)
ALPHA_BASE  = 0.35      # gentler base for stability
ALPHA_FAST  = 0.75      # kicks in when hand moves quickly
SPEED_THRESH = 0.04     # normalised-coord speed to trigger fast alpha

# Dead-zone: ignore changes smaller than this (degrees)
DEAD_ZONE   = 1.5

# Motor angle limits (degrees)
LIMITS = {
    51: (-90,  90),   # r_shoulder_y
    52: (-90,  90),   # r_shoulder_x
    53: (-60,  60),   # r_arm_z
    54: (-90, -10),   # r_elbow_y
}

# ══════════════════════════════════════════
#  CONNECT
# ══════════════════════════════════════════
dxl = DxlIO(PORT, baudrate=BAUDRATE)
ids = dxl.scan()
print("Detected motors:", ids)
if not ids:
    print("❌ No motors detected. Check connection.")
    exit()

dxl.enable_torque(ids)
# Pre-set a comfortable speed so motors don't slam
dxl.set_moving_speed({i: 300 for i in ids})

# ══════════════════════════════════════════
#  MEDIAPIPE  (static_image_mode=False  ←  fastest live-feed mode)
# ══════════════════════════════════════════
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=0,          # 0 = fastest; 1 = more accurate
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6,
)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)   # CAP_DSHOW = faster init on Windows
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 60)             # request 60 fps if camera supports it
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)       # ← key: always grab the latest frame

# ══════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════
def map_range(val, in_min, in_max, out_min, out_max):
    return (val - in_min) / (in_max - in_min) * (out_max - out_min) + out_min

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

def palm_centre(lm):
    """Average of wrist + 5 MCP joints = stable palm anchor."""
    indices = [0, 1, 5, 9, 13, 17]   # wrist + all MCPs
    xs = [lm[i].x for i in indices]
    ys = [lm[i].y for i in indices]
    return sum(xs) / len(xs), sum(ys) / len(ys)

# ══════════════════════════════════════════
#  THREADED MOTOR WRITER
#  Camera loop deposits target angles here;
#  writer thread drains at its own pace.
# ══════════════════════════════════════════
_motor_lock   = threading.Lock()
_motor_target = {}          # {motor_id: angle}
_motor_current= {}          # last sent angles for dead-zone comparison
_writer_alive = True

def _motor_writer():
    while _writer_alive:
        with _motor_lock:
            target = dict(_motor_target)

        if target:
            # Apply dead-zone: only include motors that changed enough
            cmd = {}
            for mid, ang in target.items():
                prev = _motor_current.get(mid, None)
                if prev is None or abs(ang - prev) >= DEAD_ZONE:
                    cmd[mid] = ang
                    _motor_current[mid] = ang

            if cmd:
                try:
                    dxl.set_goal_position(cmd)
                except Exception as e:
                    print(f"Motor write error: {e}")

        time.sleep(0.015)   # ~66 Hz motor update ceiling

writer_thread = threading.Thread(target=_motor_writer, daemon=True)
writer_thread.start()

def send_angles(pose: dict):
    """Non-blocking — just updates the shared target dict."""
    with _motor_lock:
        _motor_target.update(pose)

# ══════════════════════════════════════════
#  STATE
# ══════════════════════════════════════════
smooth_x, smooth_y = 0.5, 0.5
prev_x,   prev_y   = 0.5, 0.5
angles = {51: 0, 52: 0, 53: 0, 54: -50}   # display cache

# ══════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════
fps_counter, fps_t0, fps_display = 0, time.time(), 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False          # save a copy — speeds up mediapipe
    result = hands.process(rgb)
    rgb.flags.writeable = True

    hand_detected = False

    if result.multi_hand_landmarks:
        hand_detected = True
        lm = result.multi_hand_landmarks[0].landmark

        # ── Palm centre (stable tracking point) ──────────────────
        px, py = palm_centre(lm)

        # ── Adaptive EMA smoothing ────────────────────────────────
        speed = math.hypot(px - prev_x, py - prev_y)
        alpha = ALPHA_FAST if speed > SPEED_THRESH else ALPHA_BASE
        smooth_x = alpha * px + (1 - alpha) * smooth_x
        smooth_y = alpha * py + (1 - alpha) * smooth_y
        prev_x, prev_y = px, py

        # ── Wrist roll (z-axis twist via index–pinky distance) ────
        dx = lm[8].x - lm[20].x        # index tip → pinky tip
        dy = lm[8].y - lm[20].y
        roll_norm = math.atan2(dy, dx) / math.pi   # –1 … +1

        # ── Map to motor angles ───────────────────────────────────
        s_x  = map_range(smooth_x, 0.05, 0.95, -80,  80)   # r_shoulder_x: left/right
        s_y  = map_range(smooth_y, 0.05, 0.95, -80,  80)   # r_shoulder_y: up/down
        elbow= map_range(smooth_y, 0.05, 0.95, -15, -85)   # elbow bends as arm rises
        arm_z= map_range(roll_norm, -1,   1,   -50,  50)   # r_arm_z: wrist roll

        # ── Safety clamp to per-motor limits ─────────────────────
        s_x   = clamp(s_x,  *LIMITS[52])
        s_y   = clamp(s_y,  *LIMITS[51])
        arm_z = clamp(arm_z,*LIMITS[53])
        elbow = clamp(elbow,*LIMITS[54])

        angles = {51: round(s_y), 52: round(s_x),
                  53: round(arm_z), 54: round(elbow)}
        send_angles(angles)   # non-blocking

        # ── Draw landmarks ────────────────────────────────────────
        mp_drawing.draw_landmarks(
            frame,
            result.multi_hand_landmarks[0],
            mp_hands.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0,230,255), thickness=2, circle_radius=3),
            mp_drawing.DrawingSpec(color=(255,255,255), thickness=1),
        )

        # Palm centre dot
        h, w = frame.shape[:2]
        cv2.circle(frame, (int(smooth_x*w), int(smooth_y*h)), 10, (0,255,100), -1)

    # ── HUD overlay ───────────────────────────────────────────────
    status_col = (0,230,80) if hand_detected else (80,80,255)
    status_txt = "TRACKING" if hand_detected else "NO HAND"

    cv2.rectangle(frame, (0,0), (260, 170), (20,20,20), -1)
    cv2.putText(frame, f"Status : {status_txt}", (8, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, status_col, 1)
    cv2.putText(frame, f"r_shoulder_y : {angles[51]:>5}°", (8, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
    cv2.putText(frame, f"r_shoulder_x : {angles[52]:>5}°", (8, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
    cv2.putText(frame, f"r_arm_z      : {angles[53]:>5}°", (8,105),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
    cv2.putText(frame, f"r_elbow_y    : {angles[54]:>5}°", (8,130),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
    cv2.putText(frame, f"FPS: {fps_display:.0f}", (8, 158),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100,180,255), 1)

    cv2.imshow("Poppy — Right Arm Follow  [ESC to quit]", frame)

    # FPS counter
    fps_counter += 1
    if time.time() - fps_t0 >= 1.0:
        fps_display = fps_counter / (time.time() - fps_t0)
        fps_counter, fps_t0 = 0, time.time()

    if cv2.waitKey(1) == 27:
        break

# ══════════════════════════════════════════
#  CLEANUP
# ══════════════════════════════════════════
_writer_alive = False
cap.release()
cv2.destroyAllWindows()
dxl.disable_torque(ids)
dxl.close()
print("✅ Shutdown complete.")

Detected motors: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]


In [1]:
"""
╔══════════════════════════════════════════════════════════════════╗
║          POPPY HUMANOID — FULL BODY MIMIC SYSTEM                ║
║  MediaPipe Pose (33 world-space landmarks) → 24 Poppy motors    ║
║  Arms · Legs · Torso · Head — all joints, real-time             ║
╚══════════════════════════════════════════════════════════════════╝

Requirements:
    pip install opencv-python mediapipe numpy pypot

Usage:
    Stand 2–3 m from camera, fully visible.
    Press ESC to quit and return robot to balanced stand.

Tune the CONFIG section only — do not change the math below.
"""

import cv2
import mediapipe as mp
import numpy as np
import time
from collections import deque
from pypot.dynamixel import DxlIO

# ══════════════════════════════════════════════════════════════════
#  CONFIG  — only edit this block
# ══════════════════════════════════════════════════════════════════
PORT        = 'COM4'
BAUDRATE    = 1_000_000
CAM_INDEX   = 0
CAM_W       = 640
CAM_H       = 480

# Movement scale — reduce to 0.5 for first test, raise toward 1.0 as you gain confidence
SCALE       = 0.5         # 0.0 = frozen  /  1.0 = full range

# Smoothing — lower = smoother but laggier, higher = snappier
EMA_ALPHA   = 0.20
EMA_WARMUP  = 5           # frames before EMA kicks in

# Dead-zone — skip motor write if change is smaller (reduces servo hunting)
DEAD_ZONE   = 1.5          # degrees

# Motor speed for mimic
MOVE_SPEED  = 100          # 0–1023

# MediaPipe confidence
MIN_CONF    = 0.65

# Minimum landmark visibility to trust a joint (0–1)
VIS_THRESH  = 0.75

# ══════════════════════════════════════════════════════════════════
#  MOTOR DEFINITIONS
#  id : (name,  lower_limit°, upper_limit°)
# ══════════════════════════════════════════════════════════════════
MOTORS = {
    # ── Left Leg ──────────────────────────────────────────────────
    11: ('l_hip_x',      -35,  35),
    12: ('l_hip_z',      -35,  35),
    13: ('l_hip_y',      -60,  60),
    14: ('l_knee_y',       0, 120),
    15: ('l_ankle_y',    -30,  30),
    # ── Right Leg ─────────────────────────────────────────────────
    21: ('r_hip_x',      -35,  35),
    22: ('r_hip_z',      -35,  35),
    23: ('r_hip_y',      -60,  60),
    24: ('r_knee_y',    -120,   0),
    25: ('r_ankle_y',    -30,  30),
    # ── Torso ─────────────────────────────────────────────────────
    31: ('abs_y',        -30,  30),
    32: ('abs_x',        -30,  30),
    33: ('abs_z',        -30,  30),
    34: ('bust_y',       -45,  30),
    35: ('bust_x',       -30,  30),
    36: ('head_z',       -40,  40),
    37: ('head_y',       -40,  20),
    # ── Left Arm ──────────────────────────────────────────────────
    41: ('l_shoulder_y', -150, 150),
    42: ('l_shoulder_x', -150,  10),
    43: ('l_arm_z',       -90,  90),
    44: ('l_elbow_y',    -130,   0),
    # ── Right Arm ─────────────────────────────────────────────────
    51: ('r_shoulder_y', -150, 150),
    52: ('r_shoulder_x',  -10, 150),
    53: ('r_arm_z',       -90,  90),
    54: ('r_elbow_y',       0, 130),
}

# Your calibrated balanced-stand pose (used as rest/return position)
STAND = {
    11: -1.98,  12: -3.74,  13:  4.88,  14: 13.32,  15: -16.40,
    21:-22.81,  22:  6.64,  23:  0.48,  24:-10.24,  25:   6.55,
    31:  6.02,  32: -4.70,  33: -0.48,  34:-32.13,  35:   4.44,
    36: -3.08,  37:-26.83,
    41: 60.70,  42: 66.33,  43:-25.45,  44:   0.48,
    51:101.32,  52: 80.92,  53:104.13,  54:   7.34,
}

ALL_IDS = list(MOTORS.keys())

# ══════════════════════════════════════════════════════════════════
#  CONNECT MOTORS
# ══════════════════════════════════════════════════════════════════
print("Connecting to Poppy ...")
dxl    = DxlIO(PORT, baudrate=BAUDRATE)
found  = dxl.scan()
print(f"Found motors: {found}")
if not found:
    raise RuntimeError("No motors detected — check port / cable / power.")

active = [m for m in ALL_IDS if m in found]
dxl.enable_torque(active)
dxl.set_moving_speed({m: MOVE_SPEED for m in active})
print(f"Active motors ({len(active)}): {active}")

# ══════════════════════════════════════════════════════════════════
#  MEDIAPIPE POSE
#  model_complexity=2 for best accuracy; use 1 if framerate is too low
# ══════════════════════════════════════════════════════════════════
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

pose_tracker = mp_pose.Pose(
    static_image_mode        = False,
    model_complexity         = 2,
    smooth_landmarks         = True,
    enable_segmentation      = False,
    min_detection_confidence = MIN_CONF,
    min_tracking_confidence  = MIN_CONF,
)

# MediaPipe landmark indices used
# 0:nose 7:l_ear 8:r_ear
# 11:l_shoulder 12:r_shoulder 13:l_elbow 14:r_elbow 15:l_wrist 16:r_wrist
# 23:l_hip 24:r_hip 25:l_knee 26:r_knee 27:l_ankle 28:r_ankle
# 29:l_heel 30:r_heel

# ══════════════════════════════════════════════════════════════════
#  MATH UTILITIES
# ══════════════════════════════════════════════════════════════════
def v3(lm, i):
    """World-landmark → numpy float32 vector (x=right, y=up, z=toward-viewer)."""
    p = lm[i]
    return np.array([p.x, p.y, p.z], dtype=np.float32)

def vis(lm, *indices):
    """True if all specified landmarks exceed visibility threshold."""
    return all(lm[i].visibility >= VIS_THRESH for i in indices)

def norm(v):
    return v / (np.linalg.norm(v) + 1e-7)

def angle3(a, b, c):
    """Angle in degrees at joint b (vectors b→a and b→c)."""
    return float(np.degrees(
        np.arccos(np.clip(np.dot(norm(a - b), norm(c - b)), -1.0, 1.0))
    ))

def signed_angle(v1, v2, axis):
    """Signed angle in degrees from v1 to v2 around axis."""
    c  = np.cross(v1, v2)
    s  = np.dot(c, axis) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-7)
    co = np.dot(v1, v2)  / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-7)
    return float(np.degrees(np.arctan2(s, co)))

def clamp_motor(mid, val):
    _, lo, hi = MOTORS[mid]
    return float(np.clip(val, lo, hi))

def scale_from_stand(mid, raw):
    """Blend raw angle toward STAND value by SCALE factor."""
    return STAND[mid] + SCALE * (raw - STAND[mid])

# ══════════════════════════════════════════════════════════════════
#  BODY FRAME
#  World landmarks: Y=up, X=right, Z=toward viewer
# ══════════════════════════════════════════════════════════════════
WORLD_UP = np.array([0., 1., 0.], dtype=np.float32)

def body_frame(lm):
    """
    Returns orthonormal (right, up, forward) body axes,
    plus hip_mid and shoulder_mid positions.
    """
    l_sh, r_sh = v3(lm, 11), v3(lm, 12)
    l_hp, r_hp = v3(lm, 23), v3(lm, 24)

    sh_mid = (l_sh + r_sh) * 0.5
    hp_mid = (l_hp + r_hp) * 0.5

    right   = norm(r_sh - l_sh)           # shoulder line → body right
    spine   = norm(sh_mid - hp_mid)        # pelvis → chest → body up
    forward = norm(np.cross(right, spine)) # perpendicular → forward
    up      = norm(np.cross(forward, right))

    return right, up, forward, hp_mid, sh_mid

# ══════════════════════════════════════════════════════════════════
#  ARM ANGLES
#  shoulder_y  : flexion / extension (forward / backward swing)
#  shoulder_x  : abduction / adduction (side raise)
#  arm_z       : axial rotation of upper arm
#  elbow_y     : elbow flexion (0=straight)
# ══════════════════════════════════════════════════════════════════
def arm_angles(lm, R, U, F, side='right'):
    if side == 'right':
        sh, el, wr = v3(lm,12), v3(lm,14), v3(lm,16)
        if not vis(lm, 12, 14, 16): return None
        lateral_sign = 1
    else:
        sh, el, wr = v3(lm,11), v3(lm,13), v3(lm,15)
        if not vis(lm, 11, 13, 15): return None
        lateral_sign = -1

    ua = norm(el - sh)    # upper-arm direction (shoulder → elbow)

    # Project upper arm into body frame
    ua_lat  = float(np.dot(ua, R))   # lateral (+ = toward right)
    ua_up   = float(np.dot(ua, U))   # vertical (+ = upward)
    ua_fwd  = float(np.dot(ua, F))   # fore-aft (+ = forward)

    # shoulder_y: forward/back in sagittal plane
    # 0 = arm down, +90 = arm forward, -90 = arm back
    sh_y = np.degrees(np.arctan2(ua_fwd, -ua_up))

    # shoulder_x: side raise in frontal plane
    # For RIGHT: +90 = arm raised to right side
    # For LEFT:  mirrored via lateral_sign
    sh_x = lateral_sign * np.degrees(np.arctan2(lateral_sign * ua_lat, -ua_up))

    # elbow: joint angle → 0 = straight, + = bent
    elbow_bend = 180.0 - angle3(sh, el, wr)

    # arm_z: axial twist — dihedral between "body-up plane" and "forearm plane"
    fa          = norm(wr - el)
    fa_perp     = norm(fa - np.dot(fa, ua) * ua)              # forearm ⊥ upper arm
    ref         = norm(U - np.dot(U, ua) * ua)                # body-up ⊥ upper arm
    arm_z_val   = signed_angle(ref, fa_perp, ua)

    return float(sh_y), float(sh_x), float(arm_z_val), float(elbow_bend)

# ══════════════════════════════════════════════════════════════════
#  LEG ANGLES
#  hip_y  : forward / backward swing (sagittal)
#  hip_x  : side swing (frontal, abduction)
#  hip_z  : axial rotation
#  knee_y : knee flexion (0=straight)
#  ankle_y: ankle dorsiflexion
# ══════════════════════════════════════════════════════════════════
def leg_angles(lm, R, U, F, side='right'):
    if side == 'right':
        hp, kn, an = v3(lm,24), v3(lm,26), v3(lm,28)
        heel        = v3(lm, 30)
        if not vis(lm, 24, 26, 28): return None
        lat_sign = 1
    else:
        hp, kn, an = v3(lm,23), v3(lm,25), v3(lm,27)
        heel        = v3(lm, 29)
        if not vis(lm, 23, 25, 27): return None
        lat_sign = -1

    th = norm(kn - hp)    # thigh direction (hip → knee)

    th_lat = float(np.dot(th, R))
    th_up  = float(np.dot(th, U))
    th_fwd = float(np.dot(th, F))

    # hip_y: forward/backward swing
    hip_y = np.degrees(np.arctan2(th_fwd, -th_up))

    # hip_x: side swing / abduction
    hip_x = lat_sign * np.degrees(np.arctan2(lat_sign * th_lat, -th_up))

    # knee_y: flexion angle
    knee_bend = 180.0 - angle3(hp, kn, an)

    # hip_z: axial rotation — dihedral of shin relative to thigh
    sh_leg     = norm(an - kn)
    sh_perp    = norm(sh_leg - np.dot(sh_leg, th) * th)
    ref        = norm(F - np.dot(F, th) * th)               # forward ⊥ thigh
    hip_z_val  = signed_angle(ref, sh_perp, th)

    # ankle_y: use heel to measure plantarflexion
    try:
        ankle_angle = angle3(kn, an, heel)
        ankle_y     = ankle_angle - 90.0
    except Exception:
        ankle_y = 0.0

    return (float(hip_y), float(hip_x), float(hip_z_val),
            float(knee_bend), float(ankle_y))

# ══════════════════════════════════════════════════════════════════
#  TORSO & HEAD ANGLES
# ══════════════════════════════════════════════════════════════════
def torso_head_angles(lm, R, U, F, hp_mid, sh_mid):
    l_hp, r_hp = v3(lm, 23), v3(lm, 24)
    l_sh, r_sh = v3(lm, 11), v3(lm, 12)
    nose        = v3(lm,  0)
    l_ear, r_ear = v3(lm, 7), v3(lm, 8)

    spine = norm(sh_mid - hp_mid)

    # ── abs (lower spine): tilt vs world up ───────────────────────
    abs_y = float(np.degrees(np.arctan2(np.dot(spine, F), np.dot(spine, WORLD_UP))))
    abs_x = float(np.degrees(np.arctan2(np.dot(spine, R), np.dot(spine, WORLD_UP))))

    # ── abs_z: torso axial rotation — angle between hip line and shoulder line
    hip_line = norm(r_hp - l_hp)
    sh_line  = norm(r_sh - l_sh)
    abs_z    = float(np.degrees(
        np.arctan2(np.dot(np.cross(hip_line, sh_line), WORLD_UP),
                   np.dot(hip_line, sh_line))
    ))

    # ── bust (upper spine): neck direction ────────────────────────
    neck_mid  = (l_ear + r_ear) * 0.5
    bust_dir  = norm(neck_mid - sh_mid)
    bust_y    = float(np.degrees(np.arctan2(np.dot(bust_dir, F), np.dot(bust_dir, WORLD_UP))))
    bust_x    = float(np.degrees(np.arctan2(np.dot(bust_dir, R), np.dot(bust_dir, WORLD_UP))))

    # ── head: nose direction from neck ────────────────────────────
    head_dir  = norm(nose - neck_mid)
    head_y    = float(np.degrees(np.arctan2(np.dot(head_dir, F), np.dot(head_dir, WORLD_UP))))
    head_z    = float(np.degrees(np.arctan2(np.dot(head_dir, R), np.dot(head_dir, WORLD_UP))))

    return abs_y, abs_x, abs_z, bust_y, bust_x, head_y, head_z

# ══════════════════════════════════════════════════════════════════
#  EMA SMOOTHER  (per-motor exponential moving average)
# ══════════════════════════════════════════════════════════════════
class MotorSmoother:
    def __init__(self, ids, alpha=EMA_ALPHA, warmup=EMA_WARMUP):
        self.alpha  = alpha
        self.bufs   = {m: deque(maxlen=warmup) for m in ids}
        self.values = {m: None              for m in ids}

    def update(self, raw: dict) -> dict:
        out = {}
        for m, v in raw.items():
            self.bufs[m].append(v)
            b    = self.bufs[m]
            prev = self.values[m]
            if len(b) < b.maxlen or prev is None:
                s = float(np.mean(b))
            else:
                s = self.alpha * v + (1.0 - self.alpha) * prev
            self.values[m] = s
            out[m] = s
        return out

smoother   = MotorSmoother(active)
prev_cmds  = {m: None for m in active}

# ══════════════════════════════════════════════════════════════════
#  SEND COMMAND  (clamp → scale → smooth → dead-zone → write)
# ══════════════════════════════════════════════════════════════════
def send(raw: dict):
    scaled   = {m: scale_from_stand(m, v) for m, v in raw.items() if m in active}
    clamped  = {m: clamp_motor(m, v)      for m, v in scaled.items()}
    smoothed = smoother.update(clamped)

    cmds = {}
    for m, val in smoothed.items():
        t = int(round(val))
        if prev_cmds[m] is None or abs(t - prev_cmds[m]) >= DEAD_ZONE:
            cmds[m]        = t
            prev_cmds[m]   = t
    if cmds:
        dxl.set_goal_position(cmds)

# ══════════════════════════════════════════════════════════════════
#  CAMERA SETUP
# ══════════════════════════════════════════════════════════════════
cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAM_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
cap.set(cv2.CAP_PROP_FPS,          30)
cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)   # always read the latest frame

print("Camera ready. Stand 2–3 m from camera. Keep full body visible. ESC = quit.")

# ══════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════
fps_t = time.perf_counter()
fc    = 0
fps   = 0.0
tracking_parts = []

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)

    rgb               = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    res               = pose_tracker.process(rgb)
    rgb.flags.writeable = True

    tracking_parts = []

    if res.pose_world_landmarks:
        wlm = res.pose_world_landmarks.landmark
        try:
            # ── Body frame ─────────────────────────────────────
            R, U, F, hp_mid, sh_mid = body_frame(wlm)

            cmd = {}   # will hold {motor_id: angle}

            # ── RIGHT ARM ──────────────────────────────────────
            ra = arm_angles(wlm, R, U, F, 'right')
            if ra:
                rsh_y, rsh_x, rarm_z, relbow = ra
                cmd.update({
                    51: rsh_y,
                    52: rsh_x,
                    53: rarm_z,
                    54: relbow,
                })
                tracking_parts.append("R-Arm")

            # ── LEFT ARM ───────────────────────────────────────
            la = arm_angles(wlm, R, U, F, 'left')
            if la:
                lsh_y, lsh_x, larm_z, lelbow = la
                cmd.update({
                    41: lsh_y,
                    42: lsh_x,
                    43: larm_z,
                    44: -lelbow,   # left elbow motor is inverted on Poppy
                })
                tracking_parts.append("L-Arm")

            # ── RIGHT LEG ──────────────────────────────────────
            rl = leg_angles(wlm, R, U, F, 'right')
            if rl:
                rhy, rhx, rhz, rkn, ran = rl
                cmd.update({
                    23: rhy,
                    21: rhx,
                    22: rhz,
                    24: -rkn,   # r_knee_y is negative when bent
                    25: ran,
                })
                tracking_parts.append("R-Leg")

            # ── LEFT LEG ───────────────────────────────────────
            ll = leg_angles(wlm, R, U, F, 'left')
            if ll:
                lhy, lhx, lhz, lkn, lan = ll
                cmd.update({
                    13: lhy,
                    11: lhx,
                    12: lhz,
                    14: lkn,
                    15: lan,
                })
                tracking_parts.append("L-Leg")

            # ── TORSO + HEAD ───────────────────────────────────
            if vis(wlm, 11, 12, 23, 24, 0, 7, 8):
                aby, abx, abz, bsy, bsx, hdy, hdz = torso_head_angles(
                    wlm, R, U, F, hp_mid, sh_mid
                )
                cmd.update({
                    31: aby, 32: abx, 33: abz,
                    34: bsy, 35: bsx,
                    36: hdz, 37: hdy,
                })
                tracking_parts.append("Torso")
                tracking_parts.append("Head")

            if cmd:
                send(cmd)

        except Exception as e:
            pass   # skip frame on transient numerical errors

    # ── Draw skeleton ───────────────────────────────────────────
    if res.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame,
            res.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_styles.get_default_pose_landmarks_style(),
        )

    # ── FPS counter ─────────────────────────────────────────────
    fc  += 1
    now  = time.perf_counter()
    if now - fps_t >= 1.0:
        fps   = fc / (now - fps_t)
        fps_t = now
        fc    = 0

    # ── HUD ─────────────────────────────────────────────────────
    detected = bool(tracking_parts)
    status   = "  ".join(tracking_parts) if detected else "SEARCHING ..."
    color    = (0, 210, 60) if detected else (0, 60, 220)

    cv2.rectangle(frame, (0, 0), (CAM_W, 30), (20, 20, 20), -1)
    cv2.putText(frame, status, (8, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, color, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS:{fps:.0f}  SCALE:{SCALE:.1f}", (CAM_W - 130, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (150, 150, 150), 1, cv2.LINE_AA)

    cv2.imshow("Poppy Full Body Mimic", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:    # ESC
        break
    elif key == ord('+') or key == ord('='):
        SCALE = min(1.0, round(SCALE + 0.1, 1))
        print(f"SCALE → {SCALE}")
    elif key == ord('-'):
        SCALE = max(0.0, round(SCALE - 0.1, 1))
        print(f"SCALE → {SCALE}")

# ══════════════════════════════════════════════════════════════════
#  CLEANUP
# ══════════════════════════════════════════════════════════════════
cap.release()
cv2.destroyAllWindows()
pose_tracker.close()

print("Returning to balanced stand ...")
dxl.set_moving_speed({m: 60 for m in active})
dxl.set_goal_position({m: int(v) for m, v in STAND.items() if m in active})
time.sleep(3.0)
dxl.disable_torque(active)
dxl.close()
print("Done.")

Connecting to Poppy ...
Found motors: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
Active motors (25): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
Camera ready. Stand 2–3 m from camera. Keep full body visible. ESC = quit.
Returning to balanced stand ...
Done.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         POPPY HUMANOID — PRODUCTION FULL-BODY MIMIC SYSTEM  v3.1           ║
║                                                                              ║
║  VERIFIED 25-MOTOR LAYOUT:                                                  ║
║    Left  Leg  (5): 11  12  13  14  15                                       ║
║    Right Leg  (5): 21  22  23  24  25                                       ║
║    Torso      (5): 31  32  33  34  35   ← abs_y/x/z + bust_y/x             ║
║    Head       (2): 36  37              ← head_z, head_y                    ║
║    Left  Arm  (4): 41  42  43  44                                           ║
║    Right Arm  (4): 51  52  53  54                                           ║
║    TOTAL         : 5+5+5+2+4+4 = 25  ✓                                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

Requirements:
    pip install opencv-python mediapipe numpy pypot

Camera:
    Stand 2–3 m from camera. Ensure full body is visible in frame.

Controls (live):
    ESC        → quit and return robot to balanced stand
    + / =      → increase movement SCALE by 0.1
    -          → decrease movement SCALE by 0.1
    SPACE      → toggle freeze (hold current position)
    R          → re-trigger soft-start from current robot position
"""

import cv2
import mediapipe as mp
import numpy as np
import time
import logging
from collections import deque
from typing import Dict, List, Optional, Tuple
from pypot.dynamixel import DxlIO

# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "[%(levelname)s %(asctime)s] %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("PoppyMimic")

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  — Edit only this block
# ══════════════════════════════════════════════════════════════════════════════
class Config:
    PORT     : str   = "COM4"
    BAUDRATE : int   = 1_000_000

    CAM_INDEX: int   = 0
    CAM_W    : int   = 640
    CAM_H    : int   = 480
    CAM_FPS  : int   = 30

    # Movement scale: 0.0 = frozen, 1.0 = full human range → START AT 0.5
    SCALE    : float = 0.7

    # Stage-1 EMA — applied to raw MediaPipe XYZ before angle computation
    # Lower alpha = smoother/laggier   |   Higher alpha = snappier
    LM_EMA_ALPHA  : float = 0.30
    LM_EMA_WARMUP : int   = 8

    # Stage-2 EMA — applied to computed motor angles
    MTR_EMA_ALPHA  : float = 0.40
    MTR_EMA_WARMUP : int   = 5

    # Skip motor write if |Δangle| < DEAD_ZONE degrees (stops servo hunting)
    DEAD_ZONE: float = 1.2

    # Motor speed during mimic mode (0–1023)
    MIMIC_SPEED: int = 180

    # Soft-start: robot interpolates from current → tracked pose over this time
    SOFT_START_SECS : float = 2.5
    SOFT_START_STEPS: int   = 50

    # MediaPipe model quality: 0=fast / 1=balanced / 2=accurate
    MP_COMPLEXITY: int = 2

    # Minimum landmark visibility to trust a joint
    VIS_THRESH: float = 0.55

    # MediaPipe detection / tracking confidence
    MP_CONF: float = 0.65


CFG = Config()

# ══════════════════════════════════════════════════════════════════════════════
#  25-MOTOR MAP  (verified against user calibration data)
#  id → ( name, lower_limit°, upper_limit°, balanced_stand° )
# ══════════════════════════════════════════════════════════════════════════════
MOTOR_MAP: Dict[int, Tuple[str, float, float, float]] = {

    # ── Left Leg  (5 motors) ─────────────────────────────────────────────────
    11: ("l_hip_x",      -40,   40,   -1.98),
    12: ("l_hip_z",      -40,   40,   -3.74),
    13: ("l_hip_y",      -70,   70,    4.88),
    14: ("l_knee_y",       0,  130,   13.32),
    15: ("l_ankle_y",    -35,   35,  -16.40),

    # ── Right Leg (5 motors) ─────────────────────────────────────────────────
    21: ("r_hip_x",      -40,   40,  -22.81),
    22: ("r_hip_z",      -40,   40,    6.64),
    23: ("r_hip_y",      -70,   70,    0.48),
    24: ("r_knee_y",    -130,    0,  -10.24),
    25: ("r_ankle_y",    -35,   35,    6.55),

    # ── Torso     (5 motors) ─────────────────────────────────────────────────
    31: ("abs_y",        -30,   30,    6.02),   # lower-spine fore/aft tilt
    32: ("abs_x",        -30,   30,   -4.70),   # lower-spine lateral tilt
    33: ("abs_z",        -30,   30,   -0.48),   # lower-spine axial twist
    34: ("bust_y",       -45,   30,  -32.13),   # upper-spine fore/aft tilt
    35: ("bust_x",       -30,   30,    4.44),   # upper-spine lateral tilt

    # ── Head      (2 motors) ─────────────────────────────────────────────────
    36: ("head_z",       -45,   45,   -3.08),   # head left / right yaw
    37: ("head_y",       -40,   20,  -26.83),   # head up / down nod

    # ── Left Arm  (4 motors) ─────────────────────────────────────────────────
    41: ("l_shoulder_y", -150, 150,   60.70),
    42: ("l_shoulder_x", -150,  10,   66.33),
    43: ("l_arm_z",       -90,  90,  -25.45),
    44: ("l_elbow_y",    -130,   0,    0.48),

    # ── Right Arm (4 motors) ─────────────────────────────────────────────────
    51: ("r_shoulder_y", -150, 150,  101.32),
    52: ("r_shoulder_x",  -10, 150,   80.92),
    53: ("r_arm_z",       -90,  90,  104.13),
    54: ("r_elbow_y",       0, 130,    7.34),
}

# Sanity-check at import time
assert len(MOTOR_MAP) == 25, f"BUG: motor map has {len(MOTOR_MAP)} entries, expected 25"

ALL_IDS    = list(MOTOR_MAP.keys())
STAND_POSE = {mid: MOTOR_MAP[mid][3] for mid in ALL_IDS}

# Group sets used for last-known-good fallback
_GRP_RARM  = [51, 52, 53, 54]
_GRP_LARM  = [41, 42, 43, 44]
_GRP_RLEG  = [21, 22, 23, 24, 25]
_GRP_LLEG  = [11, 12, 13, 14, 15]
_GRP_TORSO = [31, 32, 33, 34, 35]
_GRP_HEAD  = [36, 37]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE-1 FILTER — Landmark EMA smoother (XYZ space)
#  Smooths raw MediaPipe world coordinates BEFORE any angle maths.
#  This prevents trig functions (arctan2 / arccos) from amplifying
#  per-frame detection noise into large angle swings.
# ══════════════════════════════════════════════════════════════════════════════
class LandmarkSmoother:
    def __init__(self, n: int = 33,
                 alpha: float = CFG.LM_EMA_ALPHA,
                 warmup: int  = CFG.LM_EMA_WARMUP):
        self.alpha  = alpha
        self.warmup = warmup
        self.n      = n
        # shape (n, 3) ring buffers
        self.bufs   = [[deque(maxlen=warmup) for _ in range(3)] for _ in range(n)]
        self.smooth = np.zeros((n, 3), dtype=np.float64)
        self.ready  = False   # True once first warmup period is complete

    def update(self, lm_list) -> np.ndarray:
        raw = np.array([[p.x, p.y, p.z] for p in lm_list], dtype=np.float64)
        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].append(raw[i, c])
                buf = self.bufs[i][c]
                if len(buf) < self.warmup:
                    self.smooth[i, c] = float(np.mean(buf))
                else:
                    self.ready        = True
                    self.smooth[i, c] = (self.alpha * raw[i, c]
                                         + (1.0 - self.alpha) * self.smooth[i, c])
        return self.smooth.copy()

    def reset(self):
        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].clear()
        self.smooth[:] = 0.0
        self.ready = False


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE-2 FILTER — Motor angle EMA smoother
#  Second EMA pass in the degree domain removes residual noise that
#  coordinate smoothing alone cannot fix (e.g., near gimbal-lock).
# ══════════════════════════════════════════════════════════════════════════════
class MotorSmoother:
    def __init__(self, ids: List[int],
                 alpha: float = CFG.MTR_EMA_ALPHA,
                 warmup: int  = CFG.MTR_EMA_WARMUP):
        self.alpha  = alpha
        self.bufs   = {m: deque(maxlen=warmup) for m in ids}
        self.values = {m: None               for m in ids}

    def update(self, raw: Dict[int, float]) -> Dict[int, float]:
        out = {}
        for m, v in raw.items():
            if m not in self.bufs:
                continue
            self.bufs[m].append(v)
            prev = self.values[m]
            if len(self.bufs[m]) < self.bufs[m].maxlen or prev is None:
                s = float(np.mean(self.bufs[m]))
            else:
                s = self.alpha * v + (1.0 - self.alpha) * prev
            self.values[m] = s
            out[m] = s
        return out

    def inject(self, pose: Dict[int, float]):
        """Pre-seed with a known pose — avoids transient snap after soft-start."""
        for m, v in pose.items():
            if m in self.values:
                self.values[m] = v
                for _ in range(self.bufs[m].maxlen):
                    self.bufs[m].append(v)


# ══════════════════════════════════════════════════════════════════════════════
#  ERROR HANDLING — Last-Known-Good store
#  On visibility loss a group of motors holds its last safe position
#  instead of snapping to zero or stand pose.
# ══════════════════════════════════════════════════════════════════════════════
class LastKnownGood:
    def __init__(self, seed: Dict[int, float]):
        self._pos: Dict[int, float] = dict(seed)

    def update(self, d: Dict[int, float]):
        self._pos.update(d)

    def get(self, mid: int) -> float:
        return self._pos.get(mid, STAND_POSE.get(mid, 0.0))

    def group(self, ids: List[int]) -> Dict[int, float]:
        return {m: self.get(m) for m in ids}

    def all(self) -> Dict[int, float]:
        return dict(self._pos)


# ══════════════════════════════════════════════════════════════════════════════
#  VECTOR MATH HELPERS
# ══════════════════════════════════════════════════════════════════════════════
_WORLD_UP = np.array([0., 1., 0.], dtype=np.float64)

def _v(pts: np.ndarray, i: int) -> np.ndarray:
    return pts[i].astype(np.float64)

def _norm(v: np.ndarray) -> np.ndarray:
    return v / (np.linalg.norm(v) + 1e-9)

def _angle3(a, b, c) -> float:
    """Interior angle in degrees at vertex b."""
    return float(np.degrees(
        np.arccos(np.clip(np.dot(_norm(a - b), _norm(c - b)), -1.0, 1.0))
    ))

def _signed_angle(v1, v2, axis) -> float:
    """Signed angle in degrees from v1 to v2 around axis."""
    v1n, v2n = _norm(v1), _norm(v2)
    s = np.dot(np.cross(v1n, v2n), _norm(axis))
    c = np.dot(v1n, v2n)
    return float(np.degrees(np.arctan2(s, c)))

def _clamp(mid: int, val: float) -> float:
    _, lo, hi, _ = MOTOR_MAP[mid]
    return float(np.clip(val, lo, hi))

def _scale(mid: int, raw: float) -> float:
    """Blend raw angle toward stand by CFG.SCALE factor."""
    return STAND_POSE[mid] + CFG.SCALE * (raw - STAND_POSE[mid])

def _sc(mid: int, raw: float) -> float:
    """Scale then clamp."""
    return _clamp(mid, _scale(mid, raw))

def _vis(lm_raw, *indices: int) -> bool:
    return all(lm_raw[i].visibility >= CFG.VIS_THRESH for i in indices)

def _body_frame(pts: np.ndarray):
    """
    Derive orthonormal body frame (right, up, forward) from hip + shoulder landmarks.
    MediaPipe world space: X = right, Y = up, Z = toward viewer
    Returns: right, up, forward (3,), hp_mid (3,), sh_mid (3,)
    """
    l_sh, r_sh = _v(pts, 11), _v(pts, 12)
    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    sh_mid = (l_sh + r_sh) * 0.5
    hp_mid = (l_hp + r_hp) * 0.5
    right   = _norm(r_sh - l_sh)
    spine   = _norm(sh_mid - hp_mid)
    forward = _norm(np.cross(right, spine))
    up      = _norm(np.cross(forward, right))
    return right, up, forward, hp_mid, sh_mid


# ══════════════════════════════════════════════════════════════════════════════
#  ANGLE EXTRACTORS
#  Each function returns a {motor_id: angle_deg} dict, or None if invisible.
#  All angles come out of geometric decomposition in body-local coordinates.
# ══════════════════════════════════════════════════════════════════════════════

# ── ARM (right or left) ──────────────────────────────────────────────────────
def extract_arm_ik(pts, lm_raw, side):
    if side == "right":
        sh, el, wr = 12, 14, 16
        m_sy, m_sx, m_el = 51, 52, 54
    else:
        sh, el, wr = 11, 13, 15
        m_sy, m_sx, m_el = 41, 42, 44

    if not _vis(lm_raw, sh, el, wr):
        return None

    s = _v(pts, sh)
    e = _v(pts, el)
    w = _v(pts, wr)

    sy, sx, elbow = ik_arm(s, e, w)

    return {
        m_sy: _sc(m_sy, sy),
        m_sx: _sc(m_sx, sx),
        m_el: _sc(m_el, elbow),
    }


# ── LEG (right or left) ──────────────────────────────────────────────────────
def extract_leg_ik(pts, lm_raw, side):
    if side == "right":
        hp, kn, an = 24, 26, 28
        m_hy, m_hx, m_kn = 23, 21, 24
    else:
        hp, kn, an = 23, 25, 27
        m_hy, m_hx, m_kn = 13, 11, 14

    if not _vis(lm_raw, hp, kn, an):
        return None

    h = _v(pts, hp)
    k = _v(pts, kn)
    a = _v(pts, an)

    hy, hx, knee = ik_leg(h, k, a)

    return {
        m_hy: _sc(m_hy, hy),
        m_hx: _sc(m_hx, hx),
        m_kn: _sc(m_kn, knee),
    }


# ── TORSO (5 motors: 31–35) + HEAD (2 motors: 36–37) ────────────────────────
def extract_torso_head(pts: np.ndarray, lm_raw,
                       R, U, F,
                       hp_mid, sh_mid) -> Optional[Dict[int, float]]:
    """
    Torso (motors 31–35):
        abs_y  (31): lower-spine fore/aft tilt
        abs_x  (32): lower-spine lateral tilt
        abs_z  (33): pelvis axial twist (hip-line yaw)
        bust_y (34): upper-spine fore/aft tilt
        bust_x (35): upper-spine lateral tilt

    Head (motors 36–37):
        head_z (36): yaw  — left/right turn
        head_y (37): pitch — up/down nod
    """
    if not _vis(lm_raw, 11, 12, 23, 24):
        return None

    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    l_sh, r_sh = _v(pts, 11), _v(pts, 12)
    spine = _norm(sh_mid - hp_mid)

    # ── abs: lower spine inclination ─────────────────────────────────────────
    abs_y = float(np.degrees(np.arctan2(np.dot(spine, F), np.dot(spine, _WORLD_UP))))
    abs_x = float(np.degrees(np.arctan2(np.dot(spine, R), np.dot(spine, _WORLD_UP))))

    # abs_z: pelvic twist — angle between hip-line and world-X
    hip_line = _norm(r_hp - l_hp)
    world_x  = np.array([1., 0., 0.])
    abs_z    = float(np.degrees(np.arctan2(
        np.dot(np.cross(hip_line, world_x), _WORLD_UP),
        np.dot(hip_line, world_x)
    )))

    # ── bust: upper spine inclination (toward neck/ear midpoint) ─────────────
    if _vis(lm_raw, 7, 8):
        neck_mid = (_v(pts, 7) + _v(pts, 8)) * 0.5
    else:
        neck_mid = sh_mid + spine * 0.12   # fallback: extrapolate up the spine

    bust_dir = _norm(neck_mid - sh_mid)
    bust_y   = float(np.degrees(np.arctan2(np.dot(bust_dir, F), np.dot(bust_dir, _WORLD_UP))))
    bust_x   = float(np.degrees(np.arctan2(np.dot(bust_dir, R), np.dot(bust_dir, _WORLD_UP))))

    # ── head: nose direction relative to neck ────────────────────────────────
    if _vis(lm_raw, 0, 7, 8):
        nose     = _v(pts, 0)
        head_dir = _norm(nose - neck_mid)
        head_z   = float(np.degrees(np.arctan2(np.dot(head_dir, R), np.dot(head_dir, _WORLD_UP))))
        head_y   = float(np.degrees(np.arctan2(np.dot(head_dir, F), np.dot(head_dir, _WORLD_UP))))
    else:
        head_z = STAND_POSE[36]
        head_y = STAND_POSE[37]

    return {
        31: _sc(31, abs_y),
        32: _sc(32, abs_x),
        33: _sc(33, abs_z),
        34: _sc(34, bust_y),
        35: _sc(35, bust_x),
        36: _sc(36, head_z),   # head_z
        37: _sc(37, head_y),   # head_y
    }


# ══════════════════════════════════════════════════════════════════════════════
#  SOFT-START  (Safety initialisation — eliminates startup jerk)
#  Reads ACTUAL current servo positions via DxlIO.
#  Blends to first tracked target using cosine ease-in-out over N steps.
# ══════════════════════════════════════════════════════════════════════════════
def soft_start(dxl: DxlIO,
               active: List[int],
               target: Dict[int, float],
               steps: int   = CFG.SOFT_START_STEPS,
               duration: float = CFG.SOFT_START_SECS):

    log.info("Soft-start: reading current motor positions ...")
    try:
        vals    = dxl.get_present_position(active)
        current = dict(zip(active, vals))
    except Exception as exc:
        log.warning(f"Could not read positions ({exc}) — using stand pose as start.")
        current = {m: STAND_POSE.get(m, 0.0) for m in active}

    dxl.set_moving_speed({m: 80 for m in active})
    dt = duration / max(steps, 1)

    log.info(f"Soft-start: {steps} steps over {duration:.1f} s ...")
    for step in range(steps + 1):
        t    = step / steps
        ease = 0.5 * (1.0 - np.cos(np.pi * t))   # cosine ease-in-out

        cmd = {}
        for m in active:
            if m not in target:
                continue
            v0  = current.get(m, STAND_POSE.get(m, 0.0))
            v1  = target[m]
            val = _clamp(m, v0 + ease * (v1 - v0))
            cmd[m] = int(round(val))

        if cmd:
            dxl.set_goal_position(cmd)
        time.sleep(dt)

    dxl.set_moving_speed({m: CFG.MIMIC_SPEED for m in active})
    log.info("Soft-start complete — entering mimic loop.")


# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE + MEDIAPIPE + CAMERA SETUP
# ══════════════════════════════════════════════════════════════════════════════
def connect_robot() -> Tuple[DxlIO, List[int]]:
    log.info(f"Connecting on {CFG.PORT} @ {CFG.BAUDRATE} baud ...")
    dxl    = DxlIO(CFG.PORT, baudrate=CFG.BAUDRATE)
    found  = dxl.scan()
    log.info(f"Scanned motor IDs: {found}")
    if not found:
        raise RuntimeError("No motors found — check port / cable / power.")

    active  = [m for m in ALL_IDS if m in found]
    missing = [m for m in ALL_IDS if m not in found]
    if missing:
        log.warning(f"Missing motors: {[(m, MOTOR_MAP[m][0]) for m in missing]}")
    log.info(f"Active: {len(active)}/25 motors")

    dxl.enable_torque(active)
    dxl.set_moving_speed({m: CFG.MIMIC_SPEED for m in active})
    return dxl, active


def make_tracker():
    return mp.solutions.pose.Pose(
        static_image_mode        = False,
        model_complexity         = CFG.MP_COMPLEXITY,
        smooth_landmarks         = True,
        enable_segmentation      = False,
        smooth_segmentation      = False,
        min_detection_confidence = CFG.MP_CONF,
        min_tracking_confidence  = CFG.MP_CONF,
    )


def make_camera() -> cv2.VideoCapture:
    cap = cv2.VideoCapture(CFG.CAM_INDEX, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CFG.CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CFG.CAM_H)
    cap.set(cv2.CAP_PROP_FPS,          CFG.CAM_FPS)
    cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)   # discard stale frames → minimum latency
    return cap


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════
def draw_hud(frame, fps, tracking_parts, frozen, soft_starting):
    h, w = frame.shape[:2]

    cv2.rectangle(frame, (0, 0), (w, 32), (18, 18, 18), -1)

    if soft_starting:
        txt, col = "⟳ SOFT-START ...", (0, 200, 255)
    elif frozen:
        txt, col = "| | FROZEN",       (0, 165, 255)
    elif tracking_parts:
        txt, col = "● " + "  ".join(tracking_parts), (0, 210, 60)
    else:
        txt, col = "◌ Searching ...",  (30, 30, 220)

    cv2.putText(frame, txt, (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, col, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS:{fps:4.0f}  Scale:{CFG.SCALE:.1f}",
                (w - 145, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, (150, 150, 150), 1, cv2.LINE_AA)

    cv2.rectangle(frame, (0, h - 22), (w, h), (18, 18, 18), -1)
    cv2.putText(frame,
                "ESC=quit  +/-=scale  SPACE=freeze  R=soft-start",
                (8, h - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.40, (110, 110, 110), 1, cv2.LINE_AA)


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    # ── Connect ──────────────────────────────────────────────────────────────
    dxl, active = connect_robot()

    log.info("Moving to balanced stand ...")
    dxl.set_moving_speed({m: 60 for m in active})
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)

    # ── MediaPipe + camera ────────────────────────────────────────────────────
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_styles  = mp.solutions.drawing_styles
    tracker    = make_tracker()
    cap        = make_camera()

    log.info("Stand 2–3 m from camera — full body must be visible. ESC to quit.")

    # ── Filters / state ───────────────────────────────────────────────────────
    lm_smooth  = LandmarkSmoother()
    mtr_smooth = MotorSmoother(active)
    mtr_smooth.inject(STAND_POSE)
    lkg        = LastKnownGood(STAND_POSE)
    prev_cmds  : Dict[int, Optional[int]] = {m: None for m in active}

    frozen        = False
    soft_starting = False
    first_detect  = True   # triggers soft-start on first good detection

    fps_t  = time.perf_counter()
    fc     = 0
    fps    = 0.0
    parts : List[str] = []

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame, 1)

        rgb              = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res              = tracker.process(rgb)
        rgb.flags.writeable = True

        parts = []
        cmd_raw: Dict[int, float] = {}

        if res.pose_world_landmarks and not frozen:
            wlm = res.pose_world_landmarks.landmark   # visibility available here

            # ── Stage 1: smooth landmark XYZ ─────────────────────────────────
            pts = lm_smooth.update(wlm)

            try:
                R, U, F, hp_mid, sh_mid = _body_frame(pts)

                # ── Right arm ────────────────────────────────────────────────
                ra = extract_arm_ik(pts, wlm, R, U, F, "right")
                if ra:
                    cmd_raw.update(ra);  parts.append("R-Arm")
                else:
                    cmd_raw.update(lkg.group(_GRP_RARM))

                # ── Left arm ─────────────────────────────────────────────────
                la =extract_arm_ik(pts, wlm, R, U, F, "left")
                if la:
                    cmd_raw.update(la);  parts.append("L-Arm")
                else:
                    cmd_raw.update(lkg.group(_GRP_LARM))

                # ── Right leg ─────────────────────────────────────────────────
                rl = extract_leg_ik(pts, wlm, R, U, F, "right")
                if rl:
                    cmd_raw.update(rl);  parts.append("R-Leg")
                else:
                    cmd_raw.update(lkg.group(_GRP_RLEG))

                # ── Left leg ──────────────────────────────────────────────────
                ll = extract_leg(pts, wlm, R, U, F, "left")
                if ll:
                    cmd_raw.update(ll);  parts.append("L-Leg")
                else:
                    cmd_raw.update(lkg.group(_GRP_LLEG))

                # ── Torso (31–35) + Head (36, 37) ─────────────────────────────
                th = extract_torso_head(pts, wlm, R, U, F, hp_mid, sh_mid)
                if th:
                    cmd_raw.update(th);  parts.append("Torso+Head")
                else:
                    cmd_raw.update(lkg.group(_GRP_TORSO + _GRP_HEAD))

                # ── First good detection → soft-start ─────────────────────────
                if first_detect and lm_smooth.ready and cmd_raw:
                    first_detect  = False
                    soft_starting = True
                    log.info("First pose detected → soft-start.")
                    soft_start(dxl, active, cmd_raw)
                    soft_starting = False
                    mtr_smooth.inject(cmd_raw)
                    lkg.update(cmd_raw)

                # ── Stage 2: smooth motor angles ──────────────────────────────
                if not first_detect:
                    smoothed = mtr_smooth.update(cmd_raw)
                    lkg.update(smoothed)

                    # ── Dead-zone guard → write ───────────────────────────────
                    cmds: Dict[int, int] = {}
                    for m, val in smoothed.items():
                        if m not in active:
                            continue
                        t = int(round(val))
                        if prev_cmds[m] is None or abs(t - prev_cmds[m]) >= CFG.DEAD_ZONE:
                            cmds[m]       = t
                            prev_cmds[m]  = t

                    if cmds:
                        dxl.set_goal_position(cmds)

            except Exception as exc:
                log.debug(f"Frame skipped: {exc}")

        # ── Draw skeleton ─────────────────────────────────────────────────────
        if res.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                res.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp_styles.get_default_pose_landmarks_style(),
            )

        # ── FPS ───────────────────────────────────────────────────────────────
        fc += 1
        now = time.perf_counter()
        if now - fps_t >= 1.0:
            fps   = fc / (now - fps_t)
            fps_t = now
            fc    = 0

        draw_hud(frame, fps, parts, frozen, soft_starting)
        cv2.imshow("Poppy Full-Body Mimic v3.1", frame)

        # ── Key handling ──────────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == 27:                              # ESC
            break
        elif key in (ord('+'), ord('=')):
            CFG.SCALE = min(1.0, round(CFG.SCALE + 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord('-'):
            CFG.SCALE = max(0.0, round(CFG.SCALE - 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord(' '):
            frozen = not frozen
            log.info(f"FROZEN = {frozen}")
        elif key in (ord('r'), ord('R')):
            log.info("Re-triggering soft-start ...")
            soft_starting = True
            soft_start(dxl, active, lkg.all())
            soft_starting = False
            mtr_smooth.inject(lkg.all())

    # ── Shutdown ──────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    tracker.close()

    log.info("Returning to balanced stand ...")
    dxl.set_moving_speed({m: 60 for m in active})
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)
    dxl.disable_torque(active)
    dxl.close()
    log.info("All motors released. Done.")


if __name__ == "__main__":
    main()

[INFO 13:24:08] Connecting on COM4 @ 1000000 baud ...
[INFO 13:24:08] Opening port 'COM4'
[INFO 13:24:23] Scanned motor IDs: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[INFO 13:24:23] Active: 25/25 motors
[INFO 13:24:23] Moving to balanced stand ...
[INFO 13:24:28] Stand 2–3 m from camera — full body must be visible. ESC to quit.
[INFO 13:24:29] First pose detected → soft-start.
[INFO 13:24:29] Soft-start: reading current motor positions ...
[INFO 13:24:30] Soft-start: 50 steps over 2.5 s ...
[INFO 13:24:32] Soft-start complete — entering mimic loop.
[INFO 13:24:36] FROZEN = True
[INFO 13:24:44] FROZEN = False
[INFO 13:24:46] SCALE → 0.6
[INFO 13:24:48] SCALE → 0.5
[INFO 13:24:48] SCALE → 0.4
[INFO 13:24:49] SCALE → 0.3
[INFO 13:24:49] SCALE → 0.2
[INFO 13:24:50] SCALE → 0.1
[INFO 13:24:50] SCALE → 0.0
[INFO 13:24:50] SCALE → 0.0
[INFO 13:24:51] SCALE → 0.0
[INFO 13:24:51] SCALE → 0.0
[INFO 13:24:51] SCALE → 0.0
[INFO 13:24:52] 

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║       POPPY HUMANOID — ADAPTIVE JITTER-FREE MIMIC SYSTEM  v4.0             ║
║                                                                              ║
║  NEW IN v4 (zero-jitter production upgrade):                                ║
║   ● MotionDetector   — per-frame landmark velocity → motion score [0..1]   ║
║   ● AdaptiveEMA      — alpha morphs still→moving in real time              ║
║   ● DynamicDeadZone  — dead-zone widens at idle, tightens on movement      ║
║   ● PoseLock         — hard-freezes servo writes after N still frames      ║
║   ● VelocityGuard    — clamps runaway inter-frame angle jumps              ║
║                                                                              ║
║  VERIFIED 25-MOTOR LAYOUT (5+5+5+2+4+4 = 25):                              ║
║    Left Leg  11–15 | Right Leg 21–25 | Torso 31–35                         ║
║    Head      36–37 | Left Arm  41–44 | Right Arm 51–54                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

Install: pip install opencv-python mediapipe numpy pypot

Controls:
    ESC        quit → return to stand pose
    +/-        increase / decrease movement scale
    SPACE      manual freeze toggle
    R          re-trigger soft-start
    D          debug overlay toggle
"""

import cv2
import mediapipe as mp
import numpy as np
import time
import logging
from collections import deque
from enum import Enum, auto
from typing import Dict, List, Optional, Tuple
from pypot.dynamixel import DxlIO

# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "[%(levelname)s %(asctime)s] %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("PoppyMimic")

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════
class Config:
    # ── Serial ────────────────────────────────────────────────────────────────
    PORT     : str   = "COM4"
    BAUDRATE : int   = 1_000_000

    # ── Camera ────────────────────────────────────────────────────────────────
    CAM_INDEX: int   = 0
    CAM_W    : int   = 640
    CAM_H    : int   = 480
    CAM_FPS  : int   = 30

    # ── Motion scale: 0 = frozen, 1 = full human range ────────────────────────
    SCALE    : float = 0.75

    # ── MediaPipe ─────────────────────────────────────────────────────────────
    MP_COMPLEXITY: int   = 2
    MP_CONF      : float = 0.65
    VIS_THRESH   : float = 0.55

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  ADAPTIVE SMOOTHING  (Stage-1: landmark XYZ EMA)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # When perfectly still → LM_ALPHA_STILL (heavy smoothing, kills noise)
    # When fully moving   → LM_ALPHA_MOVE  (light smoothing, fast response)
    LM_ALPHA_STILL  : float = 0.06   # very slow follower at idle
    LM_ALPHA_MOVE   : float = 0.55   # near real-time during motion
    LM_WARMUP       : int   = 10     # frames of mean before EMA starts

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  ADAPTIVE SMOOTHING  (Stage-2: motor angle EMA)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    MTR_ALPHA_STILL : float = 0.08
    MTR_ALPHA_MOVE  : float = 0.48
    MTR_WARMUP      : int   = 6

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  MOTION DETECTOR
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Landmark delta thresholds (world units ≈ metres)
    MOTION_THRESH_LOW  : float = 0.003  # below → definitely still
    MOTION_THRESH_HIGH : float = 0.018  # above → definitely moving
    # Score EMA alpha — how quickly the motion score itself adapts
    MOTION_SCORE_ALPHA : float = 0.25
    # How many landmark joints to sample (top-N by delta magnitude)
    MOTION_TOP_N       : int   = 10
    # Key landmarks to weight more heavily (wrists, ankles, head)
    MOTION_KEY_LM      : List  = [0, 15, 16, 27, 28, 11, 12, 23, 24]
    MOTION_KEY_WEIGHT  : float = 2.5    # multiplier for key landmarks

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  DYNAMIC DEAD-ZONE
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    DZ_STILL : float = 3.5   # degrees — wide dead-zone kills idle servo hunting
    DZ_MOVE  : float = 0.6   # degrees — tight dead-zone for accurate tracking

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  POSE LOCK
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    LOCK_STILL_FRAMES : int   = 12    # consecutive still frames → engage lock
    LOCK_SCORE_THRESH : float = 0.04  # motion score below this = "still"
    LOCK_RELEASE_SCORE: float = 0.12  # motion score above this = release lock

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  VELOCITY GUARD (inter-frame angle jump limiter)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    MAX_DEG_PER_FRAME : float = 18.0  # clamp runaway frame-to-frame jumps

    # ── Hardware ──────────────────────────────────────────────────────────────
    MIMIC_SPEED      : int   = 180
    SOFT_START_SECS  : float = 2.5
    SOFT_START_STEPS : int   = 50


CFG = Config()

# ══════════════════════════════════════════════════════════════════════════════
#  25-MOTOR MAP   id → (name, lo°, hi°, stand°)
# ══════════════════════════════════════════════════════════════════════════════
MOTOR_MAP: Dict[int, Tuple[str, float, float, float]] = {
    # Left Leg (5)
    11: ("l_hip_x",      -40,   40,   -1.98),
    12: ("l_hip_z",      -40,   40,   -3.74),
    13: ("l_hip_y",      -70,   70,    4.88),
    14: ("l_knee_y",       0,  130,   13.32),
    15: ("l_ankle_y",    -35,   35,  -16.40),
    # Right Leg (5)
    21: ("r_hip_x",      -40,   40,  -22.81),
    22: ("r_hip_z",      -40,   40,    6.64),
    23: ("r_hip_y",      -70,   70,    0.48),
    24: ("r_knee_y",    -130,    0,  -10.24),
    25: ("r_ankle_y",    -35,   35,    6.55),
    # Torso (5)
    31: ("abs_y",        -30,   30,    6.02),
    32: ("abs_x",        -30,   30,   -4.70),
    33: ("abs_z",        -30,   30,   -0.48),
    34: ("bust_y",       -45,   30,  -32.13),
    35: ("bust_x",       -30,   30,    4.44),
    # Head (2)
    36: ("head_z",       -45,   45,   -3.08),
    37: ("head_y",       -40,   20,  -26.83),
    # Left Arm (4)
    41: ("l_shoulder_y", -150, 150,   60.70),
    42: ("l_shoulder_x", -150,  10,   66.33),
    43: ("l_arm_z",       -90,  90,  -25.45),
    44: ("l_elbow_y",    -130,   0,    0.48),
    # Right Arm (4)
    51: ("r_shoulder_y", -150, 150,  101.32),
    52: ("r_shoulder_x",  -10, 150,   80.92),
    53: ("r_arm_z",       -90,  90,  104.13),
    54: ("r_elbow_y",       0, 130,    7.34),
}


assert len(MOTOR_MAP) == 25, f"Motor count error: {len(MOTOR_MAP)}"

ALL_IDS    = list(MOTOR_MAP.keys())
STAND_POSE = {mid: MOTOR_MAP[mid][3] for mid in ALL_IDS}

_GRP_RARM  = [51, 52, 53, 54]
_GRP_LARM  = [41, 42, 43, 44]
_GRP_RLEG  = [21, 22, 23, 24, 25]
_GRP_LLEG  = [11, 12, 13, 14, 15]
_GRP_TORSO = [31, 32, 33, 34, 35]
_GRP_HEAD  = [36, 37]


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION STATE
# ══════════════════════════════════════════════════════════════════════════════
class MotionState(Enum):
    STILL  = auto()
    MOVING = auto()


# ══════════════════════════════════════════════════════════════════════════════
#  ① MOTION DETECTOR
#  Computes a normalised motion score [0..1] per frame by comparing
#  current smoothed landmark positions to the previous frame.
#  Key landmarks (wrists, ankles, head) are weighted more heavily.
# ══════════════════════════════════════════════════════════════════════════════
class MotionDetector:
    """
    Per-frame motion scoring from MediaPipe world-landmark deltas.

    score = 0.0 → user is perfectly still
    score = 1.0 → user is in rapid motion

    Internally:
      1. Compute L2 delta per landmark vs previous frame
      2. Up-weight key landmarks (wrists, ankles, head)
      3. Take mean of top-N deltas to resist outliers
      4. Normalise to [0,1] using configured thresholds
      5. EMA-smooth the score itself for hysteresis
    """

    def __init__(self):
        self.prev_pts : Optional[np.ndarray] = None   # (33, 3) last frame
        self.score    : float                = 0.0    # current motion score
        self._weights : np.ndarray           = self._build_weights()

    @staticmethod
    def _build_weights() -> np.ndarray:
        w = np.ones(33, dtype=np.float32)
        for i in CFG.MOTION_KEY_LM:
            w[i] = CFG.MOTION_KEY_WEIGHT
        return w

    def update(self, pts: np.ndarray) -> float:
        """
        pts: (33, 3) smoothed world landmarks.
        Returns: motion score in [0, 1].
        """
        if self.prev_pts is None:
            self.prev_pts = pts.copy()
            return 0.0

        # Per-landmark L2 delta
        deltas = np.linalg.norm(pts - self.prev_pts, axis=1)   # (33,)
        self.prev_pts = pts.copy()

        # Weight key landmarks
        deltas_w = deltas * self._weights

        # Top-N mean (robust against single-landmark glitches)
        n = min(CFG.MOTION_TOP_N, len(deltas_w))
        top_n_mean = float(np.mean(np.partition(deltas_w, -n)[-n:]))

        # Normalise to [0, 1]
        lo, hi = CFG.MOTION_THRESH_LOW, CFG.MOTION_THRESH_HIGH
        raw_score = float(np.clip((top_n_mean - lo) / (hi - lo + 1e-9), 0.0, 1.0))

        # Temporal smoothing on the score itself
        a = CFG.MOTION_SCORE_ALPHA
        self.score = a * raw_score + (1.0 - a) * self.score
        return self.score

    @property
    def state(self) -> MotionState:
        return MotionState.STILL if self.score < CFG.LOCK_RELEASE_SCORE else MotionState.MOVING

    def reset(self):
        self.prev_pts = None
        self.score    = 0.0


# ══════════════════════════════════════════════════════════════════════════════
#  ② STAGE-1 FILTER — Adaptive landmark EMA
#  Alpha is continuously morphed between STILL and MOVE values
#  using the current motion score. No binary switching = no discontinuities.
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveLandmarkSmoother:
    """
    Exponential Moving Average over MediaPipe world landmark XYZ coordinates.
    Alpha adapts every frame based on the live motion score.

    still (score≈0) → alpha = LM_ALPHA_STILL  → very heavy smoothing
    moving(score≈1) → alpha = LM_ALPHA_MOVE   → near pass-through
    """

    def __init__(self, n: int = 33):
        self.n      = n
        self.bufs   = [[deque(maxlen=CFG.LM_WARMUP) for _ in range(3)] for _ in range(n)]
        self.smooth = np.zeros((n, 3), dtype=np.float64)
        self.ready  = False

    def _alpha(self, score: float) -> float:
        """Linearly interpolate alpha between still and move values."""
        return CFG.LM_ALPHA_STILL + score * (CFG.LM_ALPHA_MOVE - CFG.LM_ALPHA_STILL)

    def update(self, lm_list, motion_score: float) -> np.ndarray:
        raw = np.array([[p.x, p.y, p.z] for p in lm_list], dtype=np.float64)
        a   = self._alpha(motion_score)

        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].append(raw[i, c])
                buf = self.bufs[i][c]
                if len(buf) < CFG.LM_WARMUP:
                    self.smooth[i, c] = float(np.mean(buf))
                else:
                    self.ready        = True
                    self.smooth[i, c] = a * raw[i, c] + (1.0 - a) * self.smooth[i, c]

        return self.smooth.copy()

    def reset(self):
        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].clear()
        self.smooth[:] = 0.0
        self.ready = False


# ══════════════════════════════════════════════════════════════════════════════
#  ③ STAGE-2 FILTER — Adaptive motor angle EMA + Velocity Guard
#  Per-motor EMA with motion-adaptive alpha.
#  VelocityGuard clamps runaway inter-frame jumps (bad occlusion frames).
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveMotorSmoother:
    """
    Dual-role class:
      • Adaptive EMA: alpha morphs with motion score (same interpolation as Stage-1)
      • Velocity guard: clamps |Δangle| > MAX_DEG_PER_FRAME before EMA input

    This means noisy single-frame spikes (arm fully occluded then redetected)
    are absorbed before they ever reach the EMA accumulator.
    """

    def __init__(self, ids: List[int]):
        self.bufs   = {m: deque(maxlen=CFG.MTR_WARMUP) for m in ids}
        self.values : Dict[int, Optional[float]] = {m: None for m in ids}

    def _alpha(self, score: float) -> float:
        return CFG.MTR_ALPHA_STILL + score * (CFG.MTR_ALPHA_MOVE - CFG.MTR_ALPHA_STILL)

    def update(self, raw: Dict[int, float], motion_score: float) -> Dict[int, float]:
        a   = self._alpha(motion_score)
        out = {}
        for m, v in raw.items():
            if m not in self.bufs:
                continue

            # ── Velocity guard ────────────────────────────────────────────────
            prev = self.values[m]
            if prev is not None:
                delta = v - prev
                if abs(delta) > CFG.MAX_DEG_PER_FRAME:
                    v = prev + np.sign(delta) * CFG.MAX_DEG_PER_FRAME

            # ── Adaptive EMA ──────────────────────────────────────────────────
            self.bufs[m].append(v)
            if len(self.bufs[m]) < CFG.MTR_WARMUP or prev is None:
                s = float(np.mean(self.bufs[m]))
            else:
                s = a * v + (1.0 - a) * prev

            self.values[m] = s
            out[m] = s
        return out

    def inject(self, pose: Dict[int, float]):
        """Pre-seed with a known pose (after soft-start)."""
        for m, v in pose.items():
            if m in self.values:
                self.values[m] = v
                for _ in range(CFG.MTR_WARMUP):
                    self.bufs[m].append(v)


# ══════════════════════════════════════════════════════════════════════════════
#  ④ DYNAMIC DEAD-ZONE CONTROLLER
#  The dead-zone (minimum angle change to trigger a servo write) widens
#  when the user is still and tightens during movement.
#  This is the primary mechanism that stops idle servo hunting.
# ══════════════════════════════════════════════════════════════════════════════
class DynamicDeadZone:
    """
    Computes a per-frame dead-zone threshold in degrees.

    still  (score≈0) → DZ_STILL = 3.5°  → servo only moves on real gesture
    moving (score≈1) → DZ_MOVE  = 0.6°  → fine-grained tracking

    Intermediate values are linearly interpolated, so there is no
    binary snap between modes.
    """

    def __init__(self, ids: List[int]):
        self.last_written: Dict[int, Optional[int]] = {m: None for m in ids}

    def threshold(self, motion_score: float) -> float:
        return CFG.DZ_STILL + motion_score * (CFG.DZ_MOVE - CFG.DZ_STILL)

    def filter(self, smoothed: Dict[int, float], motion_score: float) -> Dict[int, int]:
        """
        Returns only the motors whose angle has changed by more than the
        current dead-zone threshold.  Prevents unnecessary servo writes.
        """
        dz   = self.threshold(motion_score)
        cmds: Dict[int, int] = {}
        for m, val in smoothed.items():
            t = int(round(val))
            prev = self.last_written[m]
            if prev is None or abs(t - prev) >= dz:
                cmds[m]             = t
                self.last_written[m] = t
        return cmds

    def inject(self, pose: Dict[int, float]):
        for m, v in pose.items():
            if m in self.last_written:
                self.last_written[m] = int(round(v))


# ══════════════════════════════════════════════════════════════════════════════
#  ⑤ POSE LOCK
#  After LOCK_STILL_FRAMES consecutive frames with motion_score < threshold,
#  the lock engages and all servo writes are suppressed entirely.
#  The lock releases instantly on detected motion above LOCK_RELEASE_SCORE.
# ══════════════════════════════════════════════════════════════════════════════
class PoseLock:
    """
    Hard-freezes servo writes during sustained stillness.

    State machine:
        UNLOCKED → (N consecutive still frames) → LOCKED
        LOCKED   → (motion_score > RELEASE)     → UNLOCKED

    When locked, DxlIO.set_goal_position is never called, eliminating
    the servo micro-corrections that cause audible hunting noise.
    """

    def __init__(self):
        self.locked       : bool = False
        self._still_count : int  = 0

    def update(self, motion_score: float) -> bool:
        """
        Call once per frame with the current motion score.
        Returns True if writes should be suppressed (locked).
        """
        if self.locked:
            if motion_score > CFG.LOCK_RELEASE_SCORE:
                self.locked       = False
                self._still_count = 0
                log.debug("PoseLock: RELEASED")
        else:
            if motion_score < CFG.LOCK_SCORE_THRESH:
                self._still_count += 1
                if self._still_count >= CFG.LOCK_STILL_FRAMES:
                    self.locked = True
                    log.debug("PoseLock: ENGAGED")
            else:
                self._still_count = 0

        return self.locked

    def force_unlock(self):
        self.locked       = False
        self._still_count = 0


# ══════════════════════════════════════════════════════════════════════════════
#  ⑥ LAST-KNOWN-GOOD STORE
# ══════════════════════════════════════════════════════════════════════════════
class LastKnownGood:
    def __init__(self, seed: Dict[int, float]):
        self._pos: Dict[int, float] = dict(seed)

    def update(self, d: Dict[int, float]):
        self._pos.update(d)

    def get(self, mid: int) -> float:
        return self._pos.get(mid, STAND_POSE.get(mid, 0.0))

    def group(self, ids: List[int]) -> Dict[int, float]:
        return {m: self.get(m) for m in ids}

    def all(self) -> Dict[int, float]:
        return dict(self._pos)


# ══════════════════════════════════════════════════════════════════════════════
#  VECTOR MATH HELPERS  (unchanged from v3.1)
# ══════════════════════════════════════════════════════════════════════════════
_WORLD_UP = np.array([0., 1., 0.], dtype=np.float64)


def _v(pts: np.ndarray, i: int) -> np.ndarray:
    return pts[i].astype(np.float64)

def _norm(v: np.ndarray) -> np.ndarray:
    return v / (np.linalg.norm(v) + 1e-9)

def _angle3(a, b, c) -> float:
    return float(np.degrees(
        np.arccos(np.clip(np.dot(_norm(a - b), _norm(c - b)), -1.0, 1.0))
    ))

def _signed_angle(v1, v2, axis) -> float:
    v1n, v2n = _norm(v1), _norm(v2)
    s = np.dot(np.cross(v1n, v2n), _norm(axis))
    c = np.dot(v1n, v2n)
    return float(np.degrees(np.arctan2(s, c)))

def _clamp(mid: int, val: float) -> float:
    _, lo, hi, _ = MOTOR_MAP[mid]
    return float(np.clip(val, lo, hi))

def _sc(mid: int, raw: float) -> float:
    stand = STAND_POSE[mid]
    return _clamp(mid, stand + CFG.SCALE * (raw - stand))

def _vis(lm_raw, *indices: int) -> bool:
    return all(lm_raw[i].visibility >= CFG.VIS_THRESH for i in indices)

def _body_frame(pts: np.ndarray):
    l_sh, r_sh = _v(pts, 11), _v(pts, 12)
    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    sh_mid  = (l_sh + r_sh) * 0.5
    hp_mid  = (l_hp + r_hp) * 0.5
    right   = _norm(r_sh - l_sh)
    spine   = _norm(sh_mid - hp_mid)
    forward = _norm(np.cross(right, spine))
    up      = _norm(np.cross(forward, right))
    return right, up, forward, hp_mid, sh_mid


# ══════════════════════════════════════════════════════════════════════════════
#  ANGLE EXTRACTORS  (unchanged kinematics from v3.1)
# ══════════════════════════════════════════════════════════════════════════════

def extract_arm_ik(pts, lm_raw, R, U, F, side: str) -> Optional[Dict[int, float]]:
    if side == "right":
        sh_i, el_i, wr_i       = 12, 14, 16
        m_sy, m_sx, m_az, m_el = 51, 52, 53, 54
        lat_sign, elbow_sign   = 1.0, 1.0
    else:
        sh_i, el_i, wr_i       = 11, 13, 15
        m_sy, m_sx, m_az, m_el = 41, 42, 43, 44
        lat_sign, elbow_sign   = -1.0, -1.0

    if not _vis(lm_raw, sh_i, el_i, wr_i):
        return None

    sh, el, wr = _v(pts, sh_i), _v(pts, el_i), _v(pts, wr_i)
    ua = _norm(el - sh)

    ua_up  = float(np.dot(ua, U))
    ua_fwd = float(np.dot(ua, F))
    ua_lat = float(np.dot(ua, R))

    sh_y = float(np.degrees(np.arctan2(ua_fwd, -ua_up)))
    sh_x = lat_sign * float(np.degrees(np.arctan2(lat_sign * ua_lat, -ua_up)))

    fa       = _norm(wr - el)
    fa_perp  = _norm(fa - np.dot(fa, ua) * ua)
    ref_perp = _norm(U  - np.dot(U,  ua) * ua)
    arm_z    = _signed_angle(ref_perp, fa_perp, ua)

    elbow_bend = 180.0 - _angle3(sh, el, wr)
    elbow_out  = elbow_sign * elbow_bend

    return {
        m_sy: _sc(m_sy, sh_y),
        m_sx: _sc(m_sx, sh_x),
        m_az: _sc(m_az, arm_z),
        m_el: _sc(m_el, elbow_out),
    }


def extract_leg(pts, lm_raw, R, U, F, side: str) -> Optional[Dict[int, float]]:
    if side == "right":
        hp_i, kn_i, an_i, heel_i     = 24, 26, 28, 30
        m_hy, m_hx, m_hz, m_kn, m_an = 23, 21, 22, 24, 25
        lat_sign, knee_sign           = 1.0, -1.0
    else:
        hp_i, kn_i, an_i, heel_i     = 23, 25, 27, 29
        m_hy, m_hx, m_hz, m_kn, m_an = 13, 11, 12, 14, 15
        lat_sign, knee_sign           = -1.0, 1.0

    if not _vis(lm_raw, hp_i, kn_i, an_i):
        return None

    hp, kn, an = _v(pts, hp_i), _v(pts, kn_i), _v(pts, an_i)
    heel        = _v(pts, heel_i)
    th = _norm(kn - hp)

    th_up  = float(np.dot(th, U))
    th_fwd = float(np.dot(th, F))
    th_lat = float(np.dot(th, R))

    hip_y = float(np.degrees(np.arctan2(th_fwd, -th_up)))
    hip_x = lat_sign * float(np.degrees(np.arctan2(lat_sign * th_lat, -th_up)))

    sh_leg   = _norm(an - kn)
    sh_perp  = _norm(sh_leg - np.dot(sh_leg, th) * th)
    ref_perp = _norm(F      - np.dot(F,       th) * th)
    hip_z    = _signed_angle(ref_perp, sh_perp, th)

    knee_out  = knee_sign * (180.0 - _angle3(hp, kn, an))
    ankle_out = _angle3(kn, an, heel) - 90.0

    return {
        m_hy: _sc(m_hy, hip_y),
        m_hx: _sc(m_hx, hip_x),
        m_hz: _sc(m_hz, hip_z),
        m_kn: _sc(m_kn, knee_out),
        m_an: _sc(m_an, ankle_out),
    }


def extract_torso_head(pts, lm_raw, R, U, F,
                       hp_mid, sh_mid) -> Optional[Dict[int, float]]:
    if not _vis(lm_raw, 11, 12, 23, 24):
        return None

    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    spine = _norm(sh_mid - hp_mid)

    abs_y = float(np.degrees(np.arctan2(np.dot(spine, F), np.dot(spine, _WORLD_UP))))
    abs_x = float(np.degrees(np.arctan2(np.dot(spine, R), np.dot(spine, _WORLD_UP))))

    hip_line = _norm(r_hp - l_hp)
    world_x  = np.array([1., 0., 0.])
    abs_z    = float(np.degrees(np.arctan2(
        np.dot(np.cross(hip_line, world_x), _WORLD_UP),
        np.dot(hip_line, world_x)
    )))

    if _vis(lm_raw, 7, 8):
        neck_mid = (_v(pts, 7) + _v(pts, 8)) * 0.5
    else:
        neck_mid = sh_mid + spine * 0.12

    bust_dir = _norm(neck_mid - sh_mid)
    bust_y   = float(np.degrees(np.arctan2(np.dot(bust_dir, F), np.dot(bust_dir, _WORLD_UP))))
    bust_x   = float(np.degrees(np.arctan2(np.dot(bust_dir, R), np.dot(bust_dir, _WORLD_UP))))

    if _vis(lm_raw, 0, 7, 8):
        head_dir = _norm(_v(pts, 0) - neck_mid)
        head_z   = float(np.degrees(np.arctan2(np.dot(head_dir, R), np.dot(head_dir, _WORLD_UP))))
        head_y   = float(np.degrees(np.arctan2(np.dot(head_dir, F), np.dot(head_dir, _WORLD_UP))))
    else:
        head_z, head_y = STAND_POSE[36], STAND_POSE[37]

    return {
        31: _sc(31, abs_y),  32: _sc(32, abs_x),  33: _sc(33, abs_z),
        34: _sc(34, bust_y), 35: _sc(35, bust_x),
        36: _sc(36, head_z), 37: _sc(37, head_y),
    }


# ══════════════════════════════════════════════════════════════════════════════
#  SOFT-START  (cosine ease-in-out, reads actual servo positions)
# ══════════════════════════════════════════════════════════════════════════════
def soft_start(dxl: DxlIO, active: List[int], target: Dict[int, float]):
    log.info("Soft-start: reading current positions ...")
    try:
        vals    = dxl.get_present_position(active)
        current = dict(zip(active, vals))
    except Exception as exc:
        log.warning(f"Could not read positions ({exc}) — using stand.")
        current = dict(STAND_POSE)

    dxl.set_moving_speed({m: 80 for m in active})
    dt = CFG.SOFT_START_SECS / CFG.SOFT_START_STEPS

    for step in range(CFG.SOFT_START_STEPS + 1):
        ease = 0.5 * (1.0 - np.cos(np.pi * step / CFG.SOFT_START_STEPS))
        cmd  = {
            m: int(round(_clamp(m, current.get(m, STAND_POSE[m]) + ease * (target[m] - current.get(m, STAND_POSE[m])))))
            for m in active if m in target
        }
        if cmd:
            dxl.set_goal_position(cmd)
        time.sleep(dt)

    dxl.set_moving_speed({m: CFG.MIMIC_SPEED for m in active})
    log.info("Soft-start complete.")


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════
def _bar(val: float, width: int = 60) -> str:
    filled = int(val * width)
    return "█" * filled + "░" * (width - filled)

def draw_hud(frame, fps: float, score: float,
             locked: bool, frozen: bool, soft_starting: bool,
             parts: List[str], debug: bool):
    h, w = frame.shape[:2]

    # Top status bar
    cv2.rectangle(frame, (0, 0), (w, 30), (18, 18, 18), -1)
    if soft_starting:
        txt, col = "SOFT-START ...", (0, 200, 255)
    elif frozen:
        txt, col = "FROZEN (manual)", (0, 165, 255)
    elif locked:
        txt, col = "POSE LOCK [still]", (200, 140, 0)
    elif parts:
        txt, col = "TRACKING: " + "  ".join(parts), (0, 210, 60)
    else:
        txt, col = "SEARCHING ...", (30, 30, 220)

    cv2.putText(frame, txt, (8, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, col, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS:{fps:4.0f}  Scale:{CFG.SCALE:.1f}",
                (w - 145, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (150, 150, 150), 1, cv2.LINE_AA)

    # Debug panel
    if debug:
        panel_x, panel_y = 8, 38
        overlay = frame.copy()
        cv2.rectangle(overlay, (panel_x - 4, panel_y - 4),
                      (panel_x + 220, panel_y + 72), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

        dz = CFG.DZ_STILL + score * (CFG.DZ_MOVE - CFG.DZ_STILL)
        lm_a = CFG.LM_ALPHA_STILL + score * (CFG.LM_ALPHA_MOVE - CFG.LM_ALPHA_STILL)
        mt_a = CFG.MTR_ALPHA_STILL + score * (CFG.MTR_ALPHA_MOVE - CFG.MTR_ALPHA_STILL)

        lines = [
            (f"Motion: {score:.3f}", (0, 210, 60) if score > 0.15 else (200, 140, 0)),
            (f"LM alpha: {lm_a:.3f}  Mtr alpha: {mt_a:.3f}", (180, 180, 180)),
            (f"Dead-zone: {dz:.2f} deg", (180, 180, 180)),
            (f"Lock: {'ON' if locked else 'off'}  Frozen: {'ON' if frozen else 'off'}", (180, 180, 180)),
        ]
        for i, (line, c) in enumerate(lines):
            cv2.putText(frame, line, (panel_x, panel_y + 16 * i),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, c, 1, cv2.LINE_AA)

        # Motion bar
        bar_y = panel_y + 68
        cv2.rectangle(frame, (panel_x, bar_y), (panel_x + 210, bar_y + 6), (50, 50, 50), -1)
        bar_w = int(score * 210)
        bar_c = (0, int(210 * (1 - score)), int(210 * score))
        cv2.rectangle(frame, (panel_x, bar_y), (panel_x + bar_w, bar_y + 6), bar_c, -1)

    # Bottom legend
    cv2.rectangle(frame, (0, h - 22), (w, h), (18, 18, 18), -1)
    cv2.putText(frame,
                "ESC=quit  +/-=scale  SPACE=freeze  R=soft-start  D=debug",
                (8, h - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (110, 110, 110), 1, cv2.LINE_AA)

    #--------CHANGED CODE--------

    import math

def ik_arm(shoulder, elbow, wrist):
    upper = np.linalg.norm(elbow - shoulder)
    lower = np.linalg.norm(wrist - elbow)
    total = np.linalg.norm(wrist - shoulder)

    total = np.clip(total, 1e-6, upper + lower - 1e-6)

    cos_elbow = (upper**2 + lower**2 - total**2) / (2 * upper * lower)
    elbow_angle = np.degrees(np.arccos(np.clip(cos_elbow, -1, 1)))

    dir_vec = (wrist - shoulder)
    dir_vec = dir_vec / (np.linalg.norm(dir_vec) + 1e-9)

    shoulder_y = np.degrees(np.arctan2(dir_vec[2], -dir_vec[1]))
    shoulder_x = np.degrees(np.arctan2(dir_vec[0], -dir_vec[1]))

    return shoulder_y, shoulder_x, elbow_angle


def ik_leg(hip, knee, ankle):
    upper = np.linalg.norm(knee - hip)
    lower = np.linalg.norm(ankle - knee)
    total = np.linalg.norm(ankle - hip)

    total = np.clip(total, 1e-6, upper + lower - 1e-6)

    cos_knee = (upper**2 + lower**2 - total**2) / (2 * upper * lower)
    knee_angle = np.degrees(np.arccos(np.clip(cos_knee, -1, 1)))

    dir_vec = (ankle - hip)
    dir_vec = dir_vec / (np.linalg.norm(dir_vec) + 1e-9)

    hip_y = np.degrees(np.arctan2(dir_vec[2], -dir_vec[1]))
    hip_x = np.degrees(np.arctan2(dir_vec[0], -dir_vec[1]))

    return hip_y, hip_x, knee_angle


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    # ── Hardware ─────────────────────────────────────────────────────────────
    log.info(f"Connecting on {CFG.PORT} @ {CFG.BAUDRATE} ...")
    dxl   = DxlIO(CFG.PORT, baudrate=CFG.BAUDRATE)
    found = dxl.scan()
    log.info(f"Found: {found}")
    if not found:
        raise RuntimeError("No motors found.")

    active  = [m for m in ALL_IDS if m in found]
    missing = [m for m in ALL_IDS if m not in found]
    if missing:
        log.warning(f"Missing motors: {[(m, MOTOR_MAP[m][0]) for m in missing]}")

    dxl.enable_torque(active)
    dxl.set_moving_speed({m: 60 for m in active})
    log.info(f"Active: {len(active)}/25 motors — moving to stand pose ...")
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)

    # ── MediaPipe ─────────────────────────────────────────────────────────────
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_styles  = mp.solutions.drawing_styles

    tracker = mp_pose.Pose(
        static_image_mode        = False,
        model_complexity         = CFG.MP_COMPLEXITY,
        smooth_landmarks         = True,
        enable_segmentation      = False,
        min_detection_confidence = CFG.MP_CONF,
        min_tracking_confidence  = CFG.MP_CONF,
    )

    # ── Camera ────────────────────────────────────────────────────────────────
    cap = cv2.VideoCapture(CFG.CAM_INDEX, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CFG.CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CFG.CAM_H)
    cap.set(cv2.CAP_PROP_FPS,          CFG.CAM_FPS)
    cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)

    log.info("Camera ready. Stand 2–3 m away, full body visible. ESC = quit.")

    # ── Adaptive pipeline ─────────────────────────────────────────────────────
    motion_det  = MotionDetector()
    lm_smooth   = AdaptiveLandmarkSmoother()
    mtr_smooth  = AdaptiveMotorSmoother(active)
    dz_ctrl     = DynamicDeadZone(active)
    pose_lock   = PoseLock()
    lkg         = LastKnownGood(STAND_POSE)

    # Pre-seed smoothers with stand pose
    mtr_smooth.inject(STAND_POSE)
    dz_ctrl.inject(STAND_POSE)

    # ── Runtime state ─────────────────────────────────────────────────────────
    frozen        = False
    debug         = False
    soft_starting = False
    first_detect  = True

    fps_t  = time.perf_counter()
    fc     = 0
    fps    = 0.0
    score  = 0.0
    parts : List[str] = []
    locked = False

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame, 1)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res   = tracker.process(rgb)
        rgb.flags.writeable = True

        parts = []

        if res.pose_world_landmarks and not frozen:
            wlm = res.pose_world_landmarks.landmark

            # ── Step 1: Motion score (raw landmarks) ──────────────────────────
            raw_pts = np.array([[p.x, p.y, p.z] for p in wlm], dtype=np.float64)
            score   = motion_det.update(raw_pts)

            # ── Step 2: Adaptive landmark smoothing ───────────────────────────
            pts = lm_smooth.update(wlm, score)

            # ── Step 3: Pose lock check ────────────────────────────────────────
            locked = pose_lock.update(score)

            try:
                R, U, F, hp_mid, sh_mid = _body_frame(pts)
                cmd_raw: Dict[int, float] = {}

                # ── Step 4: Extract segment angles ────────────────────────────
                def _ext(fn, grp, *args):
                    result = fn(pts, wlm, *args)
                    if result:
                        cmd_raw.update(result)
                        return True
                    cmd_raw.update(lkg.group(grp))
                    return False

                if _ext(extract_arm_ik,  _GRP_RARM,  R, U, F, "right"): parts.append("R-Arm")
                if _ext(extract_arm_ik,  _GRP_LARM,  R, U, F, "left"):  parts.append("L-Arm")
                if _ext(extract_leg,  _GRP_RLEG,  R, U, F, "right"): parts.append("R-Leg")
                if _ext(extract_leg,  _GRP_LLEG,  R, U, F, "left"):  parts.append("L-Leg")

                th = extract_torso_head(pts, wlm, R, U, F, hp_mid, sh_mid)
                if th:
                    cmd_raw.update(th)
                    parts.append("Torso+Head")
                else:
                    cmd_raw.update(lkg.group(_GRP_TORSO + _GRP_HEAD))

                # ── Step 5: Soft-start on first good detection ─────────────────
                if first_detect and lm_smooth.ready and cmd_raw:
                    first_detect  = False
                    soft_starting = True
                    soft_start(dxl, active, cmd_raw)
                    soft_starting = False
                    mtr_smooth.inject(cmd_raw)
                    dz_ctrl.inject(cmd_raw)
                    lkg.update(cmd_raw)

                # ── Step 6: Adaptive motor smoothing ──────────────────────────
                if not first_detect:
                    smoothed = mtr_smooth.update(cmd_raw, score)
                    lkg.update(smoothed)

                    # ── Step 7: Pose lock + dynamic dead-zone → write ──────────
                    if not locked:
                        cmds = dz_ctrl.filter(
                            {m: v for m, v in smoothed.items() if m in active},
                            score
                        )
                        if cmds:
                            dxl.set_goal_position(cmds)

            except Exception as exc:
                log.debug(f"Frame skipped: {exc}")

        elif frozen:
            locked = pose_lock.update(0.0)   # keep lock counter ticking

        # ── Skeleton overlay ──────────────────────────────────────────────────
        if res.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                res.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp_styles.get_default_pose_landmarks_style(),
            )

        # ── FPS ───────────────────────────────────────────────────────────────
        fc += 1
        now = time.perf_counter()
        if now - fps_t >= 1.0:
            fps   = fc / (now - fps_t)
            fps_t = now
            fc    = 0

        draw_hud(frame, fps, score, locked, frozen, soft_starting, parts, debug)
        cv2.imshow("Poppy Adaptive Mimic v4", frame)

        # ── Key handling ──────────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == 27:                              # ESC
            break
        elif key in (ord('+'), ord('=')):
            CFG.SCALE = min(1.0, round(CFG.SCALE + 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord('-'):
            CFG.SCALE = max(0.0, round(CFG.SCALE - 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord(' '):
            frozen = not frozen
            if frozen:
                pose_lock.force_unlock()
            log.info(f"Manual freeze: {frozen}")
        elif key in (ord('r'), ord('R')):
            log.info("Re-triggering soft-start ...")
            soft_starting = True
            pose_lock.force_unlock()
            soft_start(dxl, active, lkg.all())
            soft_starting = False
            mtr_smooth.inject(lkg.all())
            dz_ctrl.inject(lkg.all())
        elif key in (ord('d'), ord('D')):
            debug = not debug

    # ── Shutdown ──────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    tracker.close()

    log.info("Returning to stand ...")
    dxl.set_moving_speed({m: 60 for m in active})
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)
    dxl.disable_torque(active)
    dxl.close()
    log.info("Done.")


if __name__ == "__main__":
    main()

[INFO 15:21:26] Connecting on COM4 @ 1000000 ...
[INFO 15:21:26] Opening port 'COM4'
[INFO 15:21:41] Found: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[INFO 15:21:41] Active: 25/25 motors — moving to stand pose ...
[INFO 15:21:46] Camera ready. Stand 2–3 m away, full body visible. ESC = quit.
[INFO 15:21:48] Soft-start: reading current positions ...
[INFO 15:21:51] Soft-start complete.
[INFO 15:22:41] Manual freeze: True


In [ ]:
 """
╔══════════════════════════════════════════════════════════════════════════════╗
║       POPPY HUMANOID — ADAPTIVE MIMIC SYSTEM  v4.1                         ║
║                                                                              ║
║  ROOT FIX (v4.1):                                                           ║
║   ● Reference Pose Calibration — user stands still for 2 s at startup.     ║
║     MediaPipe angles at that moment become the zero-delta reference.        ║
║     Robot holds STAND_POSE exactly while you stand still. Every            ║
║     subsequent movement is a DELTA applied on top of STAND_POSE.           ║
║     This fixes wrong initial position AND wrong arm/leg tracking.          ║
║                                                                              ║
║  Retained from v4:                                                          ║
║   ● MotionDetector  → motion score [0..1]                                  ║
║   ● AdaptiveEMA     → alpha morphs still ↔ moving                          ║
║   ● DynamicDeadZone → dead-zone widens at idle                              ║
║   ● PoseLock        → hard-freeze writes after N still frames              ║
║   ● VelocityGuard   → clamps runaway inter-frame angle jumps               ║
║                                                                              ║
║  25 motors verified: 5+5+5+2+4+4 = 25                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

Install:  pip install opencv-python mediapipe numpy pypot

STARTUP SEQUENCE:
  1. Robot moves to STAND_POSE (slow, safe)
  2. "Stand still — calibrating" countdown appears (2 s, ~60 frames)
  3. Your natural standing angles are recorded as the reference
  4. Mimic begins — your movements drive motor deltas from STAND_POSE

Controls:
    ESC    quit → return to stand
    +/-    increase / decrease movement scale
    SPACE  manual freeze
    R      re-trigger soft-start
    C      re-run calibration (re-stand and hold still)
    D      debug overlay
"""

import cv2
import mediapipe as mp
import numpy as np
import time
import logging
from collections import deque
from typing import Dict, List, Optional, Tuple
from pypot.dynamixel import DxlIO

logging.basicConfig(
    level   = logging.INFO,
    format  = "[%(levelname)s %(asctime)s] %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("PoppyMimic")

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════
class Config:
    PORT     : str   = "COM4"
    BAUDRATE : int   = 1_000_000
    CAM_INDEX: int   = 0
    CAM_W    : int   = 640
    CAM_H    : int   = 480
    CAM_FPS  : int   = 30

    # Scale: 0 = frozen, 1 = full human range of motion
    SCALE    : float = 0.85

    # MediaPipe
    MP_COMPLEXITY: int   = 2
    MP_CONF      : float = 0.65
    VIS_THRESH   : float = 0.55

    # ── Reference calibration ─────────────────────────────────────────────────
    # Number of frames averaged to build the reference pose.
    # User must stand still during this window.
    CALIB_FRAMES : int = 80   # ~2.7 s at 30 fps

    # ── Adaptive landmark EMA (Stage 1) ───────────────────────────────────────
    LM_ALPHA_STILL  : float = 0.06
    LM_ALPHA_MOVE   : float = 0.55
    LM_WARMUP       : int   = 10

    # ── Adaptive motor EMA (Stage 2) ──────────────────────────────────────────
    MTR_ALPHA_STILL : float = 0.08
    MTR_ALPHA_MOVE  : float = 0.48
    MTR_WARMUP      : int   = 6

    # ── Motion detector ───────────────────────────────────────────────────────
    MOTION_THRESH_LOW  : float = 0.003
    MOTION_THRESH_HIGH : float = 0.018
    MOTION_SCORE_ALPHA : float = 0.25
    MOTION_TOP_N       : int   = 10
    MOTION_KEY_LM      : list  = None   # set below after class def
    MOTION_KEY_WEIGHT  : float = 2.5

    # ── Dynamic dead-zone ─────────────────────────────────────────────────────
    DZ_STILL : float = 3.5
    DZ_MOVE  : float = 0.6

    # ── Pose lock ─────────────────────────────────────────────────────────────
    LOCK_STILL_FRAMES  : int   = 12
    LOCK_SCORE_THRESH  : float = 0.04
    LOCK_RELEASE_SCORE : float = 0.12

    # ── Velocity guard ────────────────────────────────────────────────────────
    MAX_DEG_PER_FRAME : float = 18.0

    # ── Hardware ──────────────────────────────────────────────────────────────
    MIMIC_SPEED      : int   = 180
    SOFT_START_SECS  : float = 2.5
    SOFT_START_STEPS : int   = 50


CFG = Config()
CFG.MOTION_KEY_LM = [0, 15, 16, 27, 28, 11, 12, 23, 24]

# ══════════════════════════════════════════════════════════════════════════════
#  25-MOTOR MAP   id → (name, lo°, hi°, stand°)
# ══════════════════════════════════════════════════════════════════════════════
MOTOR_MAP: Dict[int, Tuple[str, float, float, float]] = {
    # Left Leg (5)
    11: ("l_hip_x",      -40,   40,   -1.98),
    12: ("l_hip_z",      -40,   40,   -3.74),
    13: ("l_hip_y",      -70,   70,    4.88),
    14: ("l_knee_y",       0,  130,   13.32),
    15: ("l_ankle_y",    -35,   35,  -16.40),
    # Right Leg (5)
    21: ("r_hip_x",      -40,   40,  -22.81),
    22: ("r_hip_z",      -40,   40,    6.64),
    23: ("r_hip_y",      -70,   70,    0.48),
    24: ("r_knee_y",    -130,    0,  -10.24),
    25: ("r_ankle_y",    -35,   35,    6.55),
    # Torso (5)
    31: ("abs_y",        -30,   30,    6.02),
    32: ("abs_x",        -30,   30,   -4.70),
    33: ("abs_z",        -30,   30,   -0.48),
    34: ("bust_y",       -45,   30,  -32.13),
    35: ("bust_x",       -30,   30,    4.44),
    # Head (2)
    36: ("head_z",       -45,   45,   -3.08),
    37: ("head_y",       -40,   20,  -26.83),
    # Left Arm (4)
    41: ("l_shoulder_y", -150, 150,   60.70),
    42: ("l_shoulder_x", -150,  10,   66.33),
    43: ("l_arm_z",       -90,  90,  -25.45),
    44: ("l_elbow_y",    -130,   0,    0.48),
    # Right Arm (4)
    51: ("r_shoulder_y", -150, 150,  101.32),
    52: ("r_shoulder_x",  -10, 150,   80.92),
    53: ("r_arm_z",       -90,  90,  104.13),
    54: ("r_elbow_y",       0, 130,    7.34),
}
assert len(MOTOR_MAP) == 25

ALL_IDS    = list(MOTOR_MAP.keys())
STAND_POSE = {mid: MOTOR_MAP[mid][3] for mid in ALL_IDS}

_GRP_RARM  = [51, 52, 53, 54]
_GRP_LARM  = [41, 42, 43, 44]
_GRP_RLEG  = [21, 22, 23, 24, 25]
_GRP_LLEG  = [11, 12, 13, 14, 15]
_GRP_TORSO = [31, 32, 33, 34, 35]
_GRP_HEAD  = [36, 37]
_ALL_GRPS  = _GRP_RARM + _GRP_LARM + _GRP_RLEG + _GRP_LLEG + _GRP_TORSO + _GRP_HEAD

# ══════════════════════════════════════════════════════════════════════════════
#  REFERENCE POSE CALIBRATOR  ← the key fix
#
#  Accumulates raw extractor outputs over CALIB_FRAMES while the user
#  stands still. Produces ref_angles {motor_id: angle} — the MediaPipe
#  representation of the user's natural stand.
#
#  All subsequent motor targets use:
#      target = STAND_POSE[m] + SCALE * (current_raw[m] - ref_angles[m])
#
#  Effect:
#    • User stands still → delta = 0 → robot holds STAND_POSE exactly
#    • User raises right arm 45° → delta = +45 → robot raises arm 45°
#      from its calibrated stand, regardless of body proportions or
#      camera angle differences between human and robot.
# ══════════════════════════════════════════════════════════════════════════════
class ReferencePoseCalibrator:
    def __init__(self, n_frames: int = CFG.CALIB_FRAMES):
        self.n_frames    = n_frames
        self.buffer      : List[Dict[int, float]] = []
        self.ref_angles  : Optional[Dict[int, float]] = None
        self.done        = False

    def accumulate(self, raw_angles: Dict[int, float]):
        """Call each frame during calibration window."""
        self.buffer.append(dict(raw_angles))

    def finalize(self):
        """Average all buffered frames → ref_angles."""
        if not self.buffer:
            return
        all_ids = set()
        for d in self.buffer:
            all_ids.update(d.keys())
        self.ref_angles = {}
        for mid in all_ids:
            vals = [d[mid] for d in self.buffer if mid in d]
            if vals:
                self.ref_angles[mid] = float(np.mean(vals))
        self.done = True
        log.info(f"Reference pose captured from {len(self.buffer)} frames.")
        if log.isEnabledFor(logging.DEBUG):
            for mid, v in sorted(self.ref_angles.items()):
                log.debug(f"  ref[{mid:>3}] {MOTOR_MAP[mid][0]:<16} = {v:+.2f}")

    def reset(self):
        self.buffer     = []
        self.ref_angles = None
        self.done       = False

    @property
    def progress(self) -> float:
        return min(len(self.buffer) / self.n_frames, 1.0)

    def apply(self, raw_angles: Dict[int, float]) -> Dict[int, float]:
        """
        Convert raw extractor angles → clamped motor targets using delta mapping.
        target[m] = STAND_POSE[m] + SCALE * (raw[m] - ref[m])
        """
        assert self.done, "Calibration not finished."
        out = {}
        for mid, raw_val in raw_angles.items():
            ref_val  = self.ref_angles.get(mid, raw_val)   # fallback: no delta
            stand    = STAND_POSE.get(mid, 0.0)
            delta    = raw_val - ref_val
            target   = stand + CFG.SCALE * delta
            _, lo, hi, _ = MOTOR_MAP[mid]
            out[mid] = float(np.clip(target, lo, hi))
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION DETECTOR
# ══════════════════════════════════════════════════════════════════════════════
class MotionDetector:
    def __init__(self):
        self.prev_pts : Optional[np.ndarray] = None
        self.score    : float                = 0.0
        w = np.ones(33, dtype=np.float32)
        for i in CFG.MOTION_KEY_LM:
            w[i] = CFG.MOTION_KEY_WEIGHT
        self._weights = w

    def update(self, pts: np.ndarray) -> float:
        if self.prev_pts is None:
            self.prev_pts = pts.copy()
            return 0.0
        deltas   = np.linalg.norm(pts - self.prev_pts, axis=1)
        self.prev_pts = pts.copy()
        deltas_w = deltas * self._weights
        n        = min(CFG.MOTION_TOP_N, len(deltas_w))
        top_mean = float(np.mean(np.partition(deltas_w, -n)[-n:]))
        lo, hi   = CFG.MOTION_THRESH_LOW, CFG.MOTION_THRESH_HIGH
        raw      = float(np.clip((top_mean - lo) / (hi - lo + 1e-9), 0.0, 1.0))
        a        = CFG.MOTION_SCORE_ALPHA
        self.score = a * raw + (1.0 - a) * self.score
        return self.score

    def reset(self):
        self.prev_pts = None
        self.score    = 0.0


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE-1 FILTER — Adaptive landmark EMA
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveLandmarkSmoother:
    def __init__(self, n: int = 33):
        self.n      = n
        self.bufs   = [[deque(maxlen=CFG.LM_WARMUP) for _ in range(3)] for _ in range(n)]
        self.smooth = np.zeros((n, 3), dtype=np.float64)
        self.ready  = False

    def update(self, lm_list, score: float) -> np.ndarray:
        raw = np.array([[p.x, p.y, p.z] for p in lm_list], dtype=np.float64)
        a   = CFG.LM_ALPHA_STILL + score * (CFG.LM_ALPHA_MOVE - CFG.LM_ALPHA_STILL)
        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].append(raw[i, c])
                buf = self.bufs[i][c]
                if len(buf) < CFG.LM_WARMUP:
                    self.smooth[i, c] = float(np.mean(buf))
                else:
                    self.ready        = True
                    self.smooth[i, c] = a * raw[i, c] + (1.0 - a) * self.smooth[i, c]
        return self.smooth.copy()

    def reset(self):
        for i in range(self.n):
            for c in range(3):
                self.bufs[i][c].clear()
        self.smooth[:] = 0.0
        self.ready = False


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE-2 FILTER — Adaptive motor EMA + velocity guard
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveMotorSmoother:
    def __init__(self, ids: List[int]):
        self.bufs   = {m: deque(maxlen=CFG.MTR_WARMUP) for m in ids}
        self.values : Dict[int, Optional[float]] = {m: None for m in ids}

    def update(self, raw: Dict[int, float], score: float) -> Dict[int, float]:
        a   = CFG.MTR_ALPHA_STILL + score * (CFG.MTR_ALPHA_MOVE - CFG.MTR_ALPHA_STILL)
        out = {}
        for m, v in raw.items():
            if m not in self.bufs:
                continue
            prev = self.values[m]
            # Velocity guard
            if prev is not None:
                delta = v - prev
                if abs(delta) > CFG.MAX_DEG_PER_FRAME:
                    v = prev + np.sign(delta) * CFG.MAX_DEG_PER_FRAME
            self.bufs[m].append(v)
            if len(self.bufs[m]) < CFG.MTR_WARMUP or prev is None:
                s = float(np.mean(self.bufs[m]))
            else:
                s = a * v + (1.0 - a) * prev
            self.values[m] = s
            out[m] = s
        return out

    def inject(self, pose: Dict[int, float]):
        for m, v in pose.items():
            if m in self.values:
                self.values[m] = v
                for _ in range(CFG.MTR_WARMUP):
                    self.bufs[m].append(v)


# ══════════════════════════════════════════════════════════════════════════════
#  DYNAMIC DEAD-ZONE
# ══════════════════════════════════════════════════════════════════════════════
class DynamicDeadZone:
    def __init__(self, ids: List[int]):
        self.last: Dict[int, Optional[int]] = {m: None for m in ids}

    def threshold(self, score: float) -> float:
        return CFG.DZ_STILL + score * (CFG.DZ_MOVE - CFG.DZ_STILL)

    def filter(self, smoothed: Dict[int, float], score: float) -> Dict[int, int]:
        dz   = self.threshold(score)
        cmds : Dict[int, int] = {}
        for m, val in smoothed.items():
            t    = int(round(val))
            prev = self.last[m]
            if prev is None or abs(t - prev) >= dz:
                cmds[m]    = t
                self.last[m] = t
        return cmds

    def inject(self, pose: Dict[int, float]):
        for m, v in pose.items():
            if m in self.last:
                self.last[m] = int(round(v))


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LOCK
# ══════════════════════════════════════════════════════════════════════════════
class PoseLock:
    def __init__(self):
        self.locked       = False
        self._still_count = 0

    def update(self, score: float) -> bool:
        if self.locked:
            if score > CFG.LOCK_RELEASE_SCORE:
                self.locked = False
                self._still_count = 0
        else:
            if score < CFG.LOCK_SCORE_THRESH:
                self._still_count += 1
                if self._still_count >= CFG.LOCK_STILL_FRAMES:
                    self.locked = True
            else:
                self._still_count = 0
        return self.locked

    def force_unlock(self):
        self.locked = False
        self._still_count = 0


# ══════════════════════════════════════════════════════════════════════════════
#  LAST-KNOWN-GOOD
# ══════════════════════════════════════════════════════════════════════════════
class LastKnownGood:
    def __init__(self, seed: Dict[int, float]):
        self._pos = dict(seed)

    def update(self, d: Dict[int, float]):
        self._pos.update(d)

    def get(self, mid: int) -> float:
        return self._pos.get(mid, STAND_POSE.get(mid, 0.0))

    def group(self, ids: List[int]) -> Dict[int, float]:
        return {m: self.get(m) for m in ids}

    def all(self) -> Dict[int, float]:
        return dict(self._pos)


# ══════════════════════════════════════════════════════════════════════════════
#  VECTOR MATH
# ══════════════════════════════════════════════════════════════════════════════
_WORLD_UP = np.array([0., 1., 0.], dtype=np.float64)

def _v(pts, i):    return pts[i].astype(np.float64)
def _norm(v):      return v / (np.linalg.norm(v) + 1e-9)

def _angle3(a, b, c) -> float:
    return float(np.degrees(
        np.arccos(np.clip(np.dot(_norm(a - b), _norm(c - b)), -1.0, 1.0))
    ))

def _signed_angle(v1, v2, axis) -> float:
    v1n, v2n = _norm(v1), _norm(v2)
    return float(np.degrees(np.arctan2(
        np.dot(np.cross(v1n, v2n), _norm(axis)),
        np.dot(v1n, v2n)
    )))

def _vis(lm_raw, *indices) -> bool:
    return all(lm_raw[i].visibility >= CFG.VIS_THRESH for i in indices)

def _body_frame(pts):
    l_sh, r_sh = _v(pts, 11), _v(pts, 12)
    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    sh_mid  = (l_sh + r_sh) * 0.5
    hp_mid  = (l_hp + r_hp) * 0.5
    right   = _norm(r_sh - l_sh)
    spine   = _norm(sh_mid - hp_mid)
    forward = _norm(np.cross(right, spine))
    up      = _norm(np.cross(forward, right))
    return right, up, forward, hp_mid, sh_mid


# ══════════════════════════════════════════════════════════════════════════════
#  RAW ANGLE EXTRACTORS
#  These now return RAW body-local angles with NO stand/scale applied.
#  The calibrator's .apply() does the stand + scale * delta mapping.
# ══════════════════════════════════════════════════════════════════════════════

def extract_arm_raw(pts, lm_raw, R, U, F,
                    side: str) -> Optional[Dict[int, float]]:
    """
    Returns raw body-local angles (degrees).
    No stand or scale applied here — calibrator handles that.
    """
    if side == "right":
        sh_i, el_i, wr_i       = 12, 14, 16
        m_sy, m_sx, m_az, m_el = 51, 52, 53, 54
        lat_sign, elbow_sign   = 1.0, 1.0
    else:
        sh_i, el_i, wr_i       = 11, 13, 15
        m_sy, m_sx, m_az, m_el = 41, 42, 43, 44
        lat_sign, elbow_sign   = -1.0, -1.0

    if not _vis(lm_raw, sh_i, el_i, wr_i):
        return None

    sh, el, wr = _v(pts, sh_i), _v(pts, el_i), _v(pts, wr_i)
    ua = _norm(el - sh)

    sh_y = float(np.degrees(np.arctan2(np.dot(ua, F), -np.dot(ua, U))))
    sh_x = lat_sign * float(np.degrees(np.arctan2(lat_sign * np.dot(ua, R), -np.dot(ua, U))))

    fa       = _norm(wr - el)
    fa_perp  = _norm(fa - np.dot(fa, ua) * ua)
    ref_perp = _norm(U  - np.dot(U,  ua) * ua)
    arm_z    = _signed_angle(ref_perp, fa_perp, ua)

    elbow_out = elbow_sign * (180.0 - _angle3(sh, el, wr))

    return {m_sy: sh_y, m_sx: sh_x, m_az: arm_z, m_el: elbow_out}


def extract_leg_raw(pts, lm_raw, R, U, F,
                    side: str) -> Optional[Dict[int, float]]:
    if side == "right":
        hp_i, kn_i, an_i, heel_i     = 24, 26, 28, 30
        m_hy, m_hx, m_hz, m_kn, m_an = 23, 21, 22, 24, 25
        lat_sign, knee_sign           = 1.0, -1.0
    else:
        hp_i, kn_i, an_i, heel_i     = 23, 25, 27, 29
        m_hy, m_hx, m_hz, m_kn, m_an = 13, 11, 12, 14, 15
        lat_sign, knee_sign           = -1.0, 1.0

    if not _vis(lm_raw, hp_i, kn_i, an_i):
        return None

    hp, kn, an = _v(pts, hp_i), _v(pts, kn_i), _v(pts, an_i)
    heel        = _v(pts, heel_i)
    th = _norm(kn - hp)

    hip_y = float(np.degrees(np.arctan2(np.dot(th, F), -np.dot(th, U))))
    hip_x = lat_sign * float(np.degrees(np.arctan2(lat_sign * np.dot(th, R), -np.dot(th, U))))

    sh_leg   = _norm(an - kn)
    sh_perp  = _norm(sh_leg - np.dot(sh_leg, th) * th)
    ref_perp = _norm(F - np.dot(F, th) * th)
    hip_z    = _signed_angle(ref_perp, sh_perp, th)

    knee_out  = knee_sign * (180.0 - _angle3(hp, kn, an))
    ankle_out = _angle3(kn, an, heel) - 90.0

    return {m_hy: hip_y, m_hx: hip_x, m_hz: hip_z, m_kn: knee_out, m_an: ankle_out}


def extract_torso_head_raw(pts, lm_raw, R, U, F,
                           hp_mid, sh_mid) -> Optional[Dict[int, float]]:
    if not _vis(lm_raw, 11, 12, 23, 24):
        return None

    l_hp, r_hp = _v(pts, 23), _v(pts, 24)
    spine = _norm(sh_mid - hp_mid)

    abs_y = float(np.degrees(np.arctan2(np.dot(spine, F), np.dot(spine, _WORLD_UP))))
    abs_x = float(np.degrees(np.arctan2(np.dot(spine, R), np.dot(spine, _WORLD_UP))))

    hip_line = _norm(r_hp - l_hp)
    world_x  = np.array([1., 0., 0.])
    abs_z    = float(np.degrees(np.arctan2(
        np.dot(np.cross(hip_line, world_x), _WORLD_UP),
        np.dot(hip_line, world_x)
    )))

    if _vis(lm_raw, 7, 8):
        neck_mid = (_v(pts, 7) + _v(pts, 8)) * 0.5
    else:
        neck_mid = sh_mid + spine * 0.12

    bust_dir = _norm(neck_mid - sh_mid)
    bust_y   = float(np.degrees(np.arctan2(np.dot(bust_dir, F), np.dot(bust_dir, _WORLD_UP))))
    bust_x   = float(np.degrees(np.arctan2(np.dot(bust_dir, R), np.dot(bust_dir, _WORLD_UP))))

    if _vis(lm_raw, 0, 7, 8):
        head_dir = _norm(_v(pts, 0) - neck_mid)
        head_z   = float(np.degrees(np.arctan2(np.dot(head_dir, R), np.dot(head_dir, _WORLD_UP))))
        head_y   = float(np.degrees(np.arctan2(np.dot(head_dir, F), np.dot(head_dir, _WORLD_UP))))
    else:
        head_z = 0.0
        head_y = 0.0

    return {31: abs_y, 32: abs_x, 33: abs_z,
            34: bust_y, 35: bust_x,
            36: head_z, 37: head_y}


# ══════════════════════════════════════════════════════════════════════════════
#  SOFT-START  (cosine ease-in-out, reads actual servo positions)
# ══════════════════════════════════════════════════════════════════════════════
def soft_start(dxl: DxlIO, active: List[int], target: Dict[int, float]):
    log.info("Soft-start: reading current positions ...")
    try:
        vals    = dxl.get_present_position(active)
        current = dict(zip(active, vals))
    except Exception as exc:
        log.warning(f"Could not read positions ({exc}) — using stand.")
        current = dict(STAND_POSE)

    dxl.set_moving_speed({m: 80 for m in active})
    dt = CFG.SOFT_START_SECS / CFG.SOFT_START_STEPS

    for step in range(CFG.SOFT_START_STEPS + 1):
        ease = 0.5 * (1.0 - np.cos(np.pi * step / CFG.SOFT_START_STEPS))
        cmd  = {}
        for m in active:
            if m not in target:
                continue
            v0  = current.get(m, STAND_POSE.get(m, 0.0))
            v1  = target[m]
            _, lo, hi, _ = MOTOR_MAP[m]
            cmd[m] = int(round(np.clip(v0 + ease * (v1 - v0), lo, hi)))
        if cmd:
            dxl.set_goal_position(cmd)
        time.sleep(dt)

    dxl.set_moving_speed({m: CFG.MIMIC_SPEED for m in active})
    log.info("Soft-start complete.")


# ══════════════════════════════════════════════════════════════════════════════
#  CALIBRATION OVERLAY
# ══════════════════════════════════════════════════════════════════════════════
def draw_calibration_overlay(frame, progress: float):
    h, w = frame.shape[:2]

    # Dark overlay
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    # Progress bar (centered)
    bar_w   = 320
    bar_h   = 14
    bar_x   = (w - bar_w) // 2
    bar_y   = h // 2 + 30
    filled  = int(progress * bar_w)

    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h),
                  (80, 80, 80), -1)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + filled, bar_y + bar_h),
                  (60, 200, 100), -1)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h),
                  (120, 120, 120), 1)

    pct = int(progress * 100)
    cv2.putText(frame, f"{pct}%",
                (bar_x + bar_w + 8, bar_y + 11),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180, 180, 180), 1, cv2.LINE_AA)

    # Instruction text
    cv2.putText(frame, "Stand in your natural position",
                (w // 2 - 175, h // 2 - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.70, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(frame, "Calibrating reference pose  --  hold still",
                (w // 2 - 195, h // 2 + 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 220, 180), 1, cv2.LINE_AA)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════
def draw_hud(frame, fps, score, locked, frozen, parts, debug, calibrated):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, 0), (w, 30), (18, 18, 18), -1)

    if not calibrated:
        txt, col = "Waiting for calibration ...", (0, 200, 255)
    elif frozen:
        txt, col = "FROZEN", (0, 165, 255)
    elif locked:
        txt, col = "POSE LOCK  [still]", (200, 140, 0)
    elif parts:
        txt, col = "TRACKING: " + "  ".join(parts), (0, 210, 60)
    else:
        txt, col = "SEARCHING ...", (30, 30, 220)

    cv2.putText(frame, txt, (8, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, col, 1, cv2.LINE_AA)
    cv2.putText(frame, f"FPS:{fps:4.0f}  Scale:{CFG.SCALE:.1f}",
                (w - 150, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (150, 150, 150), 1, cv2.LINE_AA)

    if debug and calibrated:
        dz  = CFG.DZ_STILL + score * (CFG.DZ_MOVE - CFG.DZ_STILL)
        lma = CFG.LM_ALPHA_STILL + score * (CFG.LM_ALPHA_MOVE - CFG.LM_ALPHA_STILL)
        lines = [
            (f"Motion score: {score:.3f}", (0,210,60) if score > 0.15 else (200,140,0)),
            (f"LM alpha: {lma:.3f}", (180, 180, 180)),
            (f"Dead-zone: {dz:.2f} deg", (180, 180, 180)),
            (f"Lock: {'ON' if locked else 'off'}", (180, 180, 180)),
        ]
        px, py = 8, 38
        overlay = frame.copy()
        cv2.rectangle(overlay, (px-4, py-4), (px+200, py+70), (20,20,20), -1)
        cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
        for i, (ln, c) in enumerate(lines):
            cv2.putText(frame, ln, (px, py + 16*i),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, c, 1, cv2.LINE_AA)

        bar_y = py + 72
        cv2.rectangle(frame, (px, bar_y), (px+200, bar_y+5), (50,50,50), -1)
        cv2.rectangle(frame, (px, bar_y), (px+int(score*200), bar_y+5),
                      (0, int(210*(1-score)), int(210*score)), -1)

    cv2.rectangle(frame, (0, h-22), (w, h), (18, 18, 18), -1)
    cv2.putText(frame,
                "ESC=quit  +/-=scale  SPACE=freeze  R=soft-start  C=recalibrate  D=debug",
                (8, h-6), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (110,110,110), 1, cv2.LINE_AA)


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    # ── Connect robot ─────────────────────────────────────────────────────────
    log.info(f"Connecting on {CFG.PORT} @ {CFG.BAUDRATE} ...")
    dxl   = DxlIO(CFG.PORT, baudrate=CFG.BAUDRATE)
    found = dxl.scan()
    log.info(f"Found: {found}")
    if not found:
        raise RuntimeError("No motors found.")

    active  = [m for m in ALL_IDS if m in found]
    missing = [m for m in ALL_IDS if m not in found]
    if missing:
        log.warning(f"Missing: {[(m, MOTOR_MAP[m][0]) for m in missing]}")

    # ── Move to exact stand pose first ────────────────────────────────────────
    log.info("Moving to calibrated STAND_POSE ...")
    dxl.enable_torque(active)
    dxl.set_moving_speed({m: 60 for m in active})
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)
    log.info("Stand pose reached.")

    # ── MediaPipe ─────────────────────────────────────────────────────────────
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_styles  = mp.solutions.drawing_styles

    tracker = mp_pose.Pose(
        static_image_mode        = False,
        model_complexity         = CFG.MP_COMPLEXITY,
        smooth_landmarks         = True,
        enable_segmentation      = False,
        min_detection_confidence = CFG.MP_CONF,
        min_tracking_confidence  = CFG.MP_CONF,
    )

    # ── Camera ────────────────────────────────────────────────────────────────
    cap = cv2.VideoCapture(CFG.CAM_INDEX, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CFG.CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CFG.CAM_H)
    cap.set(cv2.CAP_PROP_FPS,          CFG.CAM_FPS)
    cap.set(cv2.CAP_PROP_BUFFERSIZE,   1)

    log.info("Camera ready. Stand 2–3 m away — hold still for calibration.")

    # ── Pipeline components ───────────────────────────────────────────────────
    calibrator  = ReferencePoseCalibrator()
    motion_det  = MotionDetector()
    lm_smooth   = AdaptiveLandmarkSmoother()
    mtr_smooth  = AdaptiveMotorSmoother(active)
    dz_ctrl     = DynamicDeadZone(active)
    pose_lock   = PoseLock()
    lkg         = LastKnownGood(STAND_POSE)

    mtr_smooth.inject(STAND_POSE)
    dz_ctrl.inject(STAND_POSE)

    # ── State ─────────────────────────────────────────────────────────────────
    frozen    = False
    debug     = False
    fps_t     = time.perf_counter()
    fc        = 0
    fps       = 0.0
    score     = 0.0
    locked    = False
    parts   : List[str] = []

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.flip(frame, 1)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res   = tracker.process(rgb)
        rgb.flags.writeable = True

        parts = []

        if res.pose_world_landmarks and not frozen:
            wlm     = res.pose_world_landmarks.landmark
            raw_pts = np.array([[p.x, p.y, p.z] for p in wlm], dtype=np.float64)

            # Always smooth landmarks (even during calibration)
            score = motion_det.update(raw_pts)
            pts   = lm_smooth.update(wlm, score)

            try:
                R, U, F, hp_mid, sh_mid = _body_frame(pts)

                # ── Collect raw angles ────────────────────────────────────────
                raw_angles: Dict[int, float] = {}

                ra = extract_arm_raw(pts, wlm, R, U, F, "right")
                if ra: raw_angles.update(ra)
                la = extract_arm_raw(pts, wlm, R, U, F, "left")
                if la: raw_angles.update(la)
                rl = extract_leg_raw(pts, wlm, R, U, F, "right")
                if rl: raw_angles.update(rl)
                ll = extract_leg_raw(pts, wlm, R, U, F, "left")
                if ll: raw_angles.update(ll)
                th = extract_torso_head_raw(pts, wlm, R, U, F, hp_mid, sh_mid)
                if th: raw_angles.update(th)

                # ── CALIBRATION PHASE ─────────────────────────────────────────
                if not calibrator.done:
                    if raw_angles:
                        calibrator.accumulate(raw_angles)
                        if len(calibrator.buffer) >= calibrator.n_frames:
                            calibrator.finalize()
                            log.info("Calibration done — starting mimic.")
                            # Seed motor smoother with stand pose
                            mtr_smooth.inject(STAND_POSE)
                            dz_ctrl.inject(STAND_POSE)

                    draw_calibration_overlay(frame, calibrator.progress)

                # ── MIMIC PHASE ───────────────────────────────────────────────
                else:
                    # Apply delta mapping: stand + scale * (raw - ref)
                    mapped = calibrator.apply(raw_angles)

                    # Fill missing groups from last known good
                    for grp, ids in [("R-Arm", _GRP_RARM), ("L-Arm", _GRP_LARM),
                                     ("R-Leg", _GRP_RLEG), ("L-Leg", _GRP_LLEG),
                                     ("Torso+Head", _GRP_TORSO + _GRP_HEAD)]:
                        group_ids  = ids
                        group_name = grp
                        has_data   = any(m in mapped for m in group_ids)
                        if has_data:
                            parts.append(group_name)
                        else:
                            for m in group_ids:
                                mapped[m] = lkg.get(m)

                    # Stage-2 smoothing
                    smoothed = mtr_smooth.update(mapped, score)
                    lkg.update(smoothed)

                    # Pose lock + dead-zone + write
                    locked = pose_lock.update(score)
                    if not locked:
                        cmds = dz_ctrl.filter(
                            {m: v for m, v in smoothed.items() if m in active},
                            score
                        )
                        if cmds:
                            dxl.set_goal_position(cmds)

            except Exception as exc:
                log.debug(f"Frame skipped: {exc}")

        # ── Skeleton ──────────────────────────────────────────────────────────
        if res.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, res.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp_styles.get_default_pose_landmarks_style(),
            )

        # ── FPS ───────────────────────────────────────────────────────────────
        fc += 1
        now = time.perf_counter()
        if now - fps_t >= 1.0:
            fps   = fc / (now - fps_t)
            fps_t = now
            fc    = 0

        draw_hud(frame, fps, score, locked, frozen,
                 parts, debug, calibrator.done)
        cv2.imshow("Poppy Adaptive Mimic v4.1", frame)

        # ── Keys ──────────────────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        elif key in (ord('+'), ord('=')):
            CFG.SCALE = min(1.0, round(CFG.SCALE + 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord('-'):
            CFG.SCALE = max(0.0, round(CFG.SCALE - 0.1, 1))
            log.info(f"SCALE → {CFG.SCALE:.1f}")
        elif key == ord(' '):
            frozen = not frozen
            if frozen:
                pose_lock.force_unlock()
        elif key in (ord('r'), ord('R')):
            log.info("Re-triggering soft-start ...")
            dxl.set_moving_speed({m: 60 for m in active})
            dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
            time.sleep(3.0)
            soft_start(dxl, active, lkg.all())
            mtr_smooth.inject(lkg.all())
            dz_ctrl.inject(lkg.all())
        elif key in (ord('c'), ord('C')):
            log.info("Re-running calibration ...")
            calibrator.reset()
            motion_det.reset()
            lm_smooth.reset()
            pose_lock.force_unlock()
        elif key in (ord('d'), ord('D')):
            debug = not debug

    # ── Shutdown ──────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    tracker.close()

    log.info("Returning to stand pose ...")
    dxl.set_moving_speed({m: 60 for m in active})
    dxl.set_goal_position({m: int(STAND_POSE[m]) for m in active})
    time.sleep(3.0)
    dxl.disable_torque(active)
    dxl.close()
    log.info("Done.")


if __name__ == "__main__":
    main()

[INFO 15:09:07] Connecting on COM4 @ 1000000 ...
[INFO 15:09:07] Opening port 'COM4'
[INFO 15:09:22] Found: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[INFO 15:09:22] Moving to calibrated STAND_POSE ...
[INFO 15:09:26] Stand pose reached.
[INFO 15:09:27] Camera ready. Stand 2–3 m away — hold still for calibration.
[INFO 15:09:36] Reference pose captured from 80 frames.
[INFO 15:09:36] Calibration done — starting mimic.


In [ ]:
"""
Poppy Humanoid — Gesture Recognition + Motion Execution System
==============================================================
Architecture:
  GestureDetector  → reads MediaPipe landmarks, classifies gesture
  MotionController → owns robot poses, drives motors with LERP
  Main loop        → ties everything together

Dependencies:
  pip install opencv-python mediapipe pypot numpy
"""

import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ── optional: comment out when running without real robot ──────────────────
try:
    import pypot.robot
    ROBOT_AVAILABLE = True
except ImportError:
    ROBOT_AVAILABLE = False
    print("[WARN] pypot not found — running in SIMULATION mode")
# ──────────────────────────────────────────────────────────────────────────


# ═══════════════════════════════════════════════════════════════════════════
#  MOTOR DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════

# Motor-id → (attribute_name, min_angle, max_angle)
MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",     -30,  30),
    12: ("l_hip_z",     -30,  30),
    13: ("l_hip_y",     -50,  50),
    14: ("l_knee_y",      0,  90),
    15: ("l_ankle_y",   -50,  50),
    21: ("r_hip_x",     -30,  30),
    22: ("r_hip_z",     -30,  30),
    23: ("r_hip_y",     -50,  50),
    24: ("r_knee_y",    -90,   0),
    25: ("r_ankle_y",   -50,  50),
    31: ("abs_y",       -45,  45),
    32: ("abs_x",       -45,  45),
    33: ("abs_z",       -45,  45),
    34: ("bust_y",      -45,  45),
    35: ("bust_x",      -45,  45),
    36: ("head_z",      -50,  50),
    37: ("head_y",      -50,  50),
    41: ("l_shoulder_y",-90, 180),
    42: ("l_shoulder_x",-90,  90),
    43: ("l_arm_z",    -170, 170),
    44: ("l_elbow_y",    0,  170),
    51: ("r_shoulder_y",-90, 180),
    52: ("r_shoulder_x",-90,  90),
    53: ("r_arm_z",    -170, 170),
    54: ("r_elbow_y",  -170,   0),
}

# ═══════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY  (all angles in degrees, keyed by motor-id)
# ═══════════════════════════════════════════════════════════════════════════

def _full_pose(overrides: dict[int, float]) -> dict[int, float]:
    """Start from BALANCED_STAND and apply any overrides."""
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {

    # ── default / return state ─────────────────────────────────────────────
    "BALANCED_STAND": {
        11: -1.98,  12: -3.74,  13:  4.88,  14: 13.32,  15: -16.40,
        21:-22.81,  22:  6.64,  23:  0.48,  24:-10.24,  25:   6.55,
        31:  6.02,  32: -4.70,  33: -0.48,  34:-32.13,  35:   4.44,
        36: -3.08,  37:-26.83,
        41: 60.70,  42: 66.33,  43:-25.45,  44:  0.48,
        51:101.32,  52: 80.92,  53:104.13,  54:  7.34,
    },
}

# Build gesture poses relative to BALANCED_STAND after the dict exists
POSES["RIGHT_WAVE_A"] = _full_pose(
    {51: -30.0, 52: 80.0, 53: 10.0, 54: -20.0}   # r_shoulder up, arm out
)
POSES["RIGHT_WAVE_B"] = _full_pose(
    {51: -30.0, 52: 80.0, 53: 60.0, 54: -20.0}   # rotate arm_z for wave
)

POSES["LEFT_WAVE_A"] = _full_pose(
    {41: -30.0, 42: 66.0, 43: -60.0, 44: 0.0}
)
POSES["LEFT_WAVE_B"] = _full_pose(
    {41: -30.0, 42: 66.0, 43: 10.0,  44: 0.0}
)

POSES["BOTH_HANDS_UP"] = _full_pose({
    41: -80.0, 42: 10.0, 43: 0.0, 44: 0.0,    # left arm up
    51: -80.0, 52: 10.0, 53: 0.0, 54: 0.0,    # right arm up
})

POSES["HEAD_LEFT"]  = _full_pose({36:  30.0})
POSES["HEAD_RIGHT"] = _full_pose({36: -30.0})

POSES["CROSS_ARMS"] = _full_pose({
    41:  20.0, 42: -20.0, 43:  40.0, 44:  90.0,
    51:  20.0, 52:  20.0, 53: -40.0, 54: -90.0,
})


# ═══════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR
# ═══════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Classifies a single human gesture from MediaPipe PoseLandmarks.

    Returns one of:
        "BALANCED_STAND" | "RIGHT_WAVE" | "LEFT_WAVE" |
        "BOTH_HANDS_UP"  | "HEAD_LEFT" | "HEAD_RIGHT" | "CROSS_ARMS"

    Uses a short deque to debounce — a gesture must appear in
    DEBOUNCE_FRAMES consecutive frames before it is reported.
    """

    DEBOUNCE_FRAMES = 6         # ~0.2 s at 30 fps
    HEAD_TILT_THRESHOLD = 0.05  # normalised x-distance
    WAVE_WRIST_X_HISTORY = 8    # frames to track for lateral motion

    def __init__(self) -> None:
        self._history: collections.deque[str] = collections.deque(
            maxlen=self.DEBOUNCE_FRAMES
        )
        self._wrist_x_hist: dict[str, collections.deque] = {
            "right": collections.deque(maxlen=self.WAVE_WRIST_X_HISTORY),
            "left":  collections.deque(maxlen=self.WAVE_WRIST_X_HISTORY),
        }
        self.confirmed_gesture: str = "BALANCED_STAND"

    # ── public ─────────────────────────────────────────────────────────────

    def update(self, landmarks) -> str:
        """Feed new landmarks, return the current confirmed gesture."""
        if landmarks is None:
            return self.confirmed_gesture

        lm = landmarks.landmark
        raw = self._classify(lm)
        self._history.append(raw)

        # confirm only when all recent frames agree
        if len(self._history) == self.DEBOUNCE_FRAMES and \
                len(set(self._history)) == 1:
            self.confirmed_gesture = raw

        return self.confirmed_gesture

    # ── private ────────────────────────────────────────────────────────────

    def _classify(self, lm) -> str:
        mp_pose = mp.solutions.pose.PoseLandmark

        # convenience
        def pt(idx):
            return lm[idx]

        nose          = pt(mp_pose.NOSE)
        l_shoulder    = pt(mp_pose.LEFT_SHOULDER)
        r_shoulder    = pt(mp_pose.RIGHT_SHOULDER)
        l_elbow       = pt(mp_pose.LEFT_ELBOW)
        r_elbow       = pt(mp_pose.RIGHT_ELBOW)
        l_wrist       = pt(mp_pose.LEFT_WRIST)
        r_wrist       = pt(mp_pose.RIGHT_WRIST)
        l_hip         = pt(mp_pose.LEFT_HIP)
        r_hip         = pt(mp_pose.RIGHT_HIP)

        # ── CROSS / STOP ───────────────────────────────────────────────────
        # each wrist is near the *opposite* shoulder
        l_near_r = abs(l_wrist.x - r_shoulder.x) < 0.12 and \
                   abs(l_wrist.y - r_shoulder.y) < 0.15
        r_near_l = abs(r_wrist.x - l_shoulder.x) < 0.12 and \
                   abs(r_wrist.y - l_shoulder.y) < 0.15
        if l_near_r and r_near_l:
            return "CROSS_ARMS"

        # ── HEAD TILT ──────────────────────────────────────────────────────
        shoulder_mid_x = (l_shoulder.x + r_shoulder.x) / 2
        head_offset    = nose.x - shoulder_mid_x
        if head_offset > self.HEAD_TILT_THRESHOLD:
            return "HEAD_RIGHT"
        if head_offset < -self.HEAD_TILT_THRESHOLD:
            return "HEAD_LEFT"

        # ── BOTH HANDS UP ──────────────────────────────────────────────────
        l_up = l_wrist.y < l_shoulder.y - 0.05
        r_up = r_wrist.y < r_shoulder.y - 0.05
        if l_up and r_up:
            return "BOTH_HANDS_UP"

        # ── ONE-HAND WAVES (wrist above shoulder + lateral motion) ─────────
        if r_up:
            self._wrist_x_hist["right"].append(r_wrist.x)
            if self._lateral_motion("right"):
                return "RIGHT_WAVE"
        if l_up:
            self._wrist_x_hist["left"].append(l_wrist.x)
            if self._lateral_motion("left"):
                return "LEFT_WAVE"

        # ── DEFAULT ────────────────────────────────────────────────────────
        return "BALANCED_STAND"

    def _lateral_motion(self, side: str) -> bool:
        hist = self._wrist_x_hist[side]
        if len(hist) < self.WAVE_WRIST_X_HISTORY:
            return False
        return (max(hist) - min(hist)) > 0.06   # 6 % of frame width


# ═══════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER
# ═══════════════════════════════════════════════════════════════════════════

class MotionController:
    """
    Drives the robot (or simulator) using smooth LERP interpolation.

    For each named gesture a *sequence* of pose keyframes is played.
    Repeating gestures (wave) loop until the gesture is no longer active.
    """

    # seconds to move between two adjacent keyframes
    TRANSITION_SEC = 0.25
    # update frequency for the motor-drive thread (Hz)
    DRIVE_HZ = 50

    def __init__(self, robot=None) -> None:
        self._robot = robot          # pypot robot or None
        self._lock  = threading.Lock()

        # current motor angles (start from BALANCED_STAND)
        self._current: dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._target:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        # motor speeds written to robot
        self._speed: float = 50.0   # deg/s

        self._active_gesture: str   = "BALANCED_STAND"
        self._gesture_thread: threading.Thread | None = None
        self._stop_gesture_flag      = threading.Event()

        # start the low-level drive loop
        self._drive_thread = threading.Thread(
            target=self._drive_loop, daemon=True
        )
        self._drive_thread.start()

    # ── public ─────────────────────────────────────────────────────────────

    def set_gesture(self, gesture: str) -> None:
        """Called whenever GestureDetector reports a (possibly new) gesture."""
        with self._lock:
            if gesture == self._active_gesture:
                return
            self._active_gesture = gesture

        # stop existing gesture thread cleanly
        self._stop_gesture_flag.set()
        if self._gesture_thread and self._gesture_thread.is_alive():
            self._gesture_thread.join(timeout=1.0)
        self._stop_gesture_flag.clear()

        # launch new gesture thread
        self._gesture_thread = threading.Thread(
            target=self._run_gesture, args=(gesture,), daemon=True
        )
        self._gesture_thread.start()

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._current.copy()

    # ── gesture sequences ──────────────────────────────────────────────────

    def _run_gesture(self, gesture: str) -> None:
        sequences: dict[str, list[str]] = {
            "BALANCED_STAND": ["BALANCED_STAND"],
            "RIGHT_WAVE":     ["RIGHT_WAVE_A", "RIGHT_WAVE_B"],
            "LEFT_WAVE":      ["LEFT_WAVE_A",  "LEFT_WAVE_B"],
            "BOTH_HANDS_UP":  ["BOTH_HANDS_UP"],
            "HEAD_LEFT":      ["HEAD_LEFT"],
            "HEAD_RIGHT":     ["HEAD_RIGHT"],
            "CROSS_ARMS":     ["CROSS_ARMS"],
        }

        looping = gesture in ("RIGHT_WAVE", "LEFT_WAVE")
        frames  = sequences.get(gesture, ["BALANCED_STAND"])

        while not self._stop_gesture_flag.is_set():
            for pose_name in frames:
                if self._stop_gesture_flag.is_set():
                    break
                self._lerp_to(POSES[pose_name])
            if not looping:
                break

        # always return to balanced stand when gesture ends
        if gesture != "BALANCED_STAND":
            self._lerp_to(POSES["BALANCED_STAND"])

    def _lerp_to(self, target: dict[int, float]) -> None:
        """Smoothly interpolate from current angles to target over TRANSITION_SEC."""
        steps = int(self.TRANSITION_SEC * self.DRIVE_HZ)
        steps = max(steps, 1)

        with self._lock:
            start = self._current.copy()

        for i in range(1, steps + 1):
            if self._stop_gesture_flag.is_set():
                return
            t = self._ease_in_out(i / steps)
            interpolated = {
                mid: start[mid] + t * (target[mid] - start[mid])
                for mid in target
            }
            with self._lock:
                self._target = interpolated
            time.sleep(1.0 / self.DRIVE_HZ)

    # ── drive loop (50 Hz) ─────────────────────────────────────────────────

    def _drive_loop(self) -> None:
        """Copies _target into _current and pushes to hardware."""
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                target  = self._target.copy()
                current = self._current.copy()

            # tiny alpha filter to eliminate any residual jitter
            alpha   = 0.35
            blended = {
                mid: current[mid] + alpha * (target[mid] - current[mid])
                for mid in target
            }

            with self._lock:
                self._current = blended

            self._push_to_robot(blended)
            time.sleep(dt)

    def _push_to_robot(self, angles: dict[int, float]) -> None:
        if self._robot is None:
            return   # simulation — nothing to push
        for mid, val in angles.items():
            name, lo, hi = MOTOR_MAP[mid]
            val = float(np.clip(val, lo, hi))
            try:
                motor = getattr(self._robot, name, None)
                if motor:
                    motor.goal_position = val
            except Exception:
                pass   # hardware error: swallow, keep running

    # ── math helpers ──────────────────────────────────────────────────────

    @staticmethod
    def _ease_in_out(t: float) -> float:
        """Smooth-step cubic easing: slow start, fast middle, slow end."""
        return t * t * (3 - 2 * t)


# ═══════════════════════════════════════════════════════════════════════════
#  SIMULATION VISUALISER  (shown when no robot is connected)
# ═══════════════════════════════════════════════════════════════════════════

class SimVisualiser:
    """Draws a small HUD showing current motor angles in an OpenCV window."""

    W, H = 420, 580
    GROUPS = [
        ("LEFT LEG",    [11, 12, 13, 14, 15]),
        ("RIGHT LEG",   [21, 22, 23, 24, 25]),
        ("TORSO/HEAD",  [31, 32, 33, 34, 35, 36, 37]),
        ("LEFT ARM",    [41, 42, 43, 44]),
        ("RIGHT ARM",   [51, 52, 53, 54]),
    ]

    def draw(self, angles: dict[int, float], gesture: str, frame=None) -> np.ndarray:
        canvas = np.zeros((self.H, self.W, 3), dtype=np.uint8)
        canvas[:] = (20, 20, 30)

        # gesture banner
        cv2.rectangle(canvas, (0, 0), (self.W, 36), (50, 130, 80), -1)
        cv2.putText(canvas, f"GESTURE: {gesture}", (8, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 255, 200), 2)

        y = 50
        for group_name, mids in self.GROUPS:
            cv2.putText(canvas, group_name, (8, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (120, 180, 255), 1)
            y += 16
            for mid in mids:
                name = MOTOR_MAP[mid][0]
                val  = angles.get(mid, 0.0)
                bar_w = int(np.interp(val, [-180, 180], [0, self.W - 100]))
                cv2.rectangle(canvas, (90, y - 11), (90 + bar_w, y + 1),
                               (60, 180, 100), -1)
                cv2.putText(canvas,
                            f"{name:<14} {val:+7.2f}°", (4, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.38, (220, 220, 220), 1)
                y += 17
            y += 4

        # optionally overlay the camera frame thumbnail
        if frame is not None:
            thumb = cv2.resize(frame, (140, 105))
            canvas[self.H - 110: self.H - 5,
                   self.W - 145: self.W - 5] = thumb

        return canvas


# ═══════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ═══════════════════════════════════════════════════════════════════════════

def main() -> None:
    # ── robot init ─────────────────────────────────────────────────────────
    robot = None
    if ROBOT_AVAILABLE:
        try:
            robot = pypot.robot.from_config("poppy_config.json")
            robot.start_sync()
            print("[INFO] Poppy robot connected.")
        except Exception as e:
            print(f"[WARN] Could not connect to robot: {e}  →  simulation mode")

    # ── mediapipe setup ────────────────────────────────────────────────────
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.55,
    )

    # ── subsystems ─────────────────────────────────────────────────────────
    detector    = GestureDetector()
    controller  = MotionController(robot=robot)
    visualiser  = SimVisualiser()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return

    print("[INFO] Running — press  Q  to quit.")

    prev_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)   # mirror so left/right match user's POV
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = pose_model.process(rgb)
        rgb.flags.writeable = True
        annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        # ── gesture detection ──────────────────────────────────────────────
        gesture = detector.update(results.pose_landmarks)

        # ── motion controller ──────────────────────────────────────────────
        controller.set_gesture(gesture)

        # ── draw skeleton on camera frame ──────────────────────────────────
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                annotated,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 230, 80),  thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(200, 200, 0), thickness=2),
            )

        # FPS counter
        now      = time.time()
        fps      = 1.0 / max(now - prev_time, 1e-6)
        prev_time = now
        cv2.putText(annotated, f"FPS {fps:.1f}", (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 200), 2)

        # gesture label on camera window
        cv2.putText(annotated, gesture, (8, annotated.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 200, 255), 2)

        # ── robot HUD ──────────────────────────────────────────────────────
        hud = visualiser.draw(controller.current_angles, gesture, frame)

        # ── display ────────────────────────────────────────────────────────
        cv2.imshow("Gesture Input — Poppy Control", annotated)
        cv2.imshow("Robot State", hud)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # ── cleanup ────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    pose_model.close()
    if robot:
        controller.set_gesture("BALANCED_STAND")
        time.sleep(1.5)
        robot.stop_sync()
    print("[INFO] Shutdown complete.")


if __name__ == "__main__":
    main()

[WARN] Could not connect to robot: string indices must be integers  →  simulation mode
[INFO] Running — press  Q  to quit.


In [ ]:
"""
Poppy Humanoid — Gesture Recognition + Motion Execution System
==============================================================
Architecture:
  GestureDetector  → reads MediaPipe landmarks, classifies gesture
  MotionController → owns robot poses, drives motors with LERP
  Main loop        → ties everything together

Dependencies:
  pip install opencv-python mediapipe pypot numpy
"""

import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ── optional: comment out when running without real robot ──────────────────
try:
    import pypot.robot
    ROBOT_AVAILABLE = True
except ImportError:
    ROBOT_AVAILABLE = False
    print("[WARN] pypot not found — running in SIMULATION mode")
# ──────────────────────────────────────────────────────────────────────────


# ═══════════════════════════════════════════════════════════════════════════
#  MOTOR DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════

# Motor-id → (attribute_name, min_angle, max_angle)
MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",     -30,  30),
    12: ("l_hip_z",     -30,  30),
    13: ("l_hip_y",     -50,  50),
    14: ("l_knee_y",      0,  90),
    15: ("l_ankle_y",   -50,  50),
    21: ("r_hip_x",     -30,  30),
    22: ("r_hip_z",     -30,  30),
    23: ("r_hip_y",     -50,  50),
    24: ("r_knee_y",    -90,   0),
    25: ("r_ankle_y",   -50,  50),
    31: ("abs_y",       -45,  45),
    32: ("abs_x",       -45,  45),
    33: ("abs_z",       -45,  45),
    34: ("bust_y",      -45,  45),
    35: ("bust_x",      -45,  45),
    36: ("head_z",      -50,  50),
    37: ("head_y",      -50,  50),
    41: ("l_shoulder_y",-90, 180),
    42: ("l_shoulder_x",-90,  90),
    43: ("l_arm_z",    -170, 170),
    44: ("l_elbow_y",    0,  170),
    51: ("r_shoulder_y",-90, 180),
    52: ("r_shoulder_x",-90,  90),
    53: ("r_arm_z",    -170, 170),
    54: ("r_elbow_y",  -170,   0),
}

# ═══════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY  (all angles in degrees, keyed by motor-id)
# ═══════════════════════════════════════════════════════════════════════════

def _full_pose(overrides: dict[int, float]) -> dict[int, float]:
    """Start from BALANCED_STAND and apply any overrides."""
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {

    # ── default / return state ─────────────────────────────────────────────
    "BALANCED_STAND": {
        11: -1.98,  12: -3.74,  13:  4.88,  14: 13.32,  15: -16.40,
        21:-22.81,  22:  6.64,  23:  0.48,  24:-10.24,  25:   6.55,
        31:  6.02,  32: -4.70,  33: -0.48,  34:-32.13,  35:   4.44,
        36: -3.08,  37:-26.83,
        41: 60.70,  42: 66.33,  43:-25.45,  44:  0.48,
        51:101.32,  52: 80.92,  53:104.13,  54:  7.34,
    },
}

# Build gesture poses relative to BALANCED_STAND after the dict exists
POSES["RIGHT_WAVE_A"] = _full_pose(
    {51: -30.0, 52: 80.0, 53: 10.0, 54: -20.0}   # r_shoulder up, arm out
)
POSES["RIGHT_WAVE_B"] = _full_pose(
    {51: -30.0, 52: 80.0, 53: 60.0, 54: -20.0}   # rotate arm_z for wave
)

POSES["LEFT_WAVE_A"] = _full_pose(
    {41: -30.0, 42: 66.0, 43: -60.0, 44: 0.0}
)
POSES["LEFT_WAVE_B"] = _full_pose(
    {41: -30.0, 42: 66.0, 43: 10.0,  44: 0.0}
)

POSES["BOTH_HANDS_UP"] = _full_pose({
    41: -80.0, 42: 10.0, 43: 0.0, 44: 0.0,    # left arm up
    51: -80.0, 52: 10.0, 53: 0.0, 54: 0.0,    # right arm up
})

POSES["HEAD_LEFT"]  = _full_pose({36:  30.0})
POSES["HEAD_RIGHT"] = _full_pose({36: -30.0})

POSES["CROSS_ARMS"] = _full_pose({
    41:  20.0, 42: -20.0, 43:  40.0, 44:  90.0,
    51:  20.0, 52:  20.0, 53: -40.0, 54: -90.0,
})


# ═══════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR
# ═══════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Classifies a single human gesture from MediaPipe PoseLandmarks.

    Returns one of:
        "BALANCED_STAND" | "RIGHT_WAVE" | "LEFT_WAVE" |
        "BOTH_HANDS_UP"  | "HEAD_LEFT" | "HEAD_RIGHT" | "CROSS_ARMS"

    Uses a short deque to debounce — a gesture must appear in
    DEBOUNCE_FRAMES consecutive frames before it is reported.
    """

    DEBOUNCE_FRAMES = 6         # ~0.2 s at 30 fps
    HEAD_TILT_THRESHOLD = 0.05  # normalised x-distance
    WAVE_WRIST_X_HISTORY = 8    # frames to track for lateral motion

    def __init__(self) -> None:
        self._history: collections.deque[str] = collections.deque(
            maxlen=self.DEBOUNCE_FRAMES
        )
        self._wrist_x_hist: dict[str, collections.deque] = {
            "right": collections.deque(maxlen=self.WAVE_WRIST_X_HISTORY),
            "left":  collections.deque(maxlen=self.WAVE_WRIST_X_HISTORY),
        }
        self.confirmed_gesture: str = "BALANCED_STAND"

    # ── public ─────────────────────────────────────────────────────────────

    def update(self, landmarks) -> str:
        """Feed new landmarks, return the current confirmed gesture."""
        if landmarks is None:
            return self.confirmed_gesture

        lm = landmarks.landmark
        raw = self._classify(lm)
        self._history.append(raw)

        # confirm only when all recent frames agree
        if len(self._history) == self.DEBOUNCE_FRAMES and \
                len(set(self._history)) == 1:
            self.confirmed_gesture = raw

        return self.confirmed_gesture

    # ── private ────────────────────────────────────────────────────────────

    def _classify(self, lm) -> str:
        mp_pose = mp.solutions.pose.PoseLandmark

        # convenience
        def pt(idx):
            return lm[idx]

        nose          = pt(mp_pose.NOSE)
        l_shoulder    = pt(mp_pose.LEFT_SHOULDER)
        r_shoulder    = pt(mp_pose.RIGHT_SHOULDER)
        l_elbow       = pt(mp_pose.LEFT_ELBOW)
        r_elbow       = pt(mp_pose.RIGHT_ELBOW)
        l_wrist       = pt(mp_pose.LEFT_WRIST)
        r_wrist       = pt(mp_pose.RIGHT_WRIST)
        l_hip         = pt(mp_pose.LEFT_HIP)
        r_hip         = pt(mp_pose.RIGHT_HIP)

        # ── CROSS / STOP ───────────────────────────────────────────────────
        # each wrist is near the *opposite* shoulder
        l_near_r = abs(l_wrist.x - r_shoulder.x) < 0.12 and \
                   abs(l_wrist.y - r_shoulder.y) < 0.15
        r_near_l = abs(r_wrist.x - l_shoulder.x) < 0.12 and \
                   abs(r_wrist.y - l_shoulder.y) < 0.15
        if l_near_r and r_near_l:
            return "CROSS_ARMS"

        # ── HEAD TILT ──────────────────────────────────────────────────────
        shoulder_mid_x = (l_shoulder.x + r_shoulder.x) / 2
        head_offset    = nose.x - shoulder_mid_x
        if head_offset > self.HEAD_TILT_THRESHOLD:
            return "HEAD_RIGHT"
        if head_offset < -self.HEAD_TILT_THRESHOLD:
            return "HEAD_LEFT"

        # ── BOTH HANDS UP ──────────────────────────────────────────────────
        l_up = l_wrist.y < l_shoulder.y - 0.05
        r_up = r_wrist.y < r_shoulder.y - 0.05
        if l_up and r_up:
            return "BOTH_HANDS_UP"

        # ── ONE-HAND WAVES (wrist above shoulder + lateral motion) ─────────
        if r_up:
            self._wrist_x_hist["right"].append(r_wrist.x)
            if self._lateral_motion("right"):
                return "RIGHT_WAVE"
        if l_up:
            self._wrist_x_hist["left"].append(l_wrist.x)
            if self._lateral_motion("left"):
                return "LEFT_WAVE"

        # ── DEFAULT ────────────────────────────────────────────────────────
        return "BALANCED_STAND"

    def _lateral_motion(self, side: str) -> bool:
        hist = self._wrist_x_hist[side]
        if len(hist) < self.WAVE_WRIST_X_HISTORY:
            return False
        return (max(hist) - min(hist)) > 0.06   # 6 % of frame width


# ═══════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER
# ═══════════════════════════════════════════════════════════════════════════

class MotionController:
    """
    Drives the robot (or simulator) using smooth LERP interpolation.

    For each named gesture a *sequence* of pose keyframes is played.
    Repeating gestures (wave) loop until the gesture is no longer active.
    """

    # seconds to move between two adjacent keyframes
    TRANSITION_SEC = 0.25
    # update frequency for the motor-drive thread (Hz)
    DRIVE_HZ = 50

    def __init__(self, robot=None) -> None:
        self._robot = robot          # pypot robot or None
        self._lock  = threading.Lock()

        # current motor angles (start from BALANCED_STAND)
        self._current: dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._target:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        # motor speeds written to robot
        self._speed: float = 50.0   # deg/s

        self._active_gesture: str   = "BALANCED_STAND"
        self._gesture_thread: threading.Thread | None = None
        self._stop_gesture_flag      = threading.Event()

        # start the low-level drive loop
        self._drive_thread = threading.Thread(
            target=self._drive_loop, daemon=True
        )
        self._drive_thread.start()

    # ── public ─────────────────────────────────────────────────────────────

    def set_gesture(self, gesture: str) -> None:
        """Called whenever GestureDetector reports a (possibly new) gesture."""
        with self._lock:
            if gesture == self._active_gesture:
                return
            self._active_gesture = gesture

        # stop existing gesture thread cleanly
        self._stop_gesture_flag.set()
        if self._gesture_thread and self._gesture_thread.is_alive():
            self._gesture_thread.join(timeout=1.0)
        self._stop_gesture_flag.clear()

        # launch new gesture thread
        self._gesture_thread = threading.Thread(
            target=self._run_gesture, args=(gesture,), daemon=True
        )
        self._gesture_thread.start()

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._current.copy()

    # ── gesture sequences ──────────────────────────────────────────────────

    def _run_gesture(self, gesture: str) -> None:
        sequences: dict[str, list[str]] = {
            "BALANCED_STAND": ["BALANCED_STAND"],
            "RIGHT_WAVE":     ["RIGHT_WAVE_A", "RIGHT_WAVE_B"],
            "LEFT_WAVE":      ["LEFT_WAVE_A",  "LEFT_WAVE_B"],
            "BOTH_HANDS_UP":  ["BOTH_HANDS_UP"],
            "HEAD_LEFT":      ["HEAD_LEFT"],
            "HEAD_RIGHT":     ["HEAD_RIGHT"],
            "CROSS_ARMS":     ["CROSS_ARMS"],
        }

        looping = gesture in ("RIGHT_WAVE", "LEFT_WAVE")
        frames  = sequences.get(gesture, ["BALANCED_STAND"])

        while not self._stop_gesture_flag.is_set():
            for pose_name in frames:
                if self._stop_gesture_flag.is_set():
                    break
                self._lerp_to(POSES[pose_name])
            if not looping:
                break

        # always return to balanced stand when gesture ends
        if gesture != "BALANCED_STAND":
            self._lerp_to(POSES["BALANCED_STAND"])

    def _lerp_to(self, target: dict[int, float]) -> None:
        """Smoothly interpolate from current angles to target over TRANSITION_SEC."""
        steps = int(self.TRANSITION_SEC * self.DRIVE_HZ)
        steps = max(steps, 1)

        with self._lock:
            start = self._current.copy()

        for i in range(1, steps + 1):
            if self._stop_gesture_flag.is_set():
                return
            t = self._ease_in_out(i / steps)
            interpolated = {
                mid: start[mid] + t * (target[mid] - start[mid])
                for mid in target
            }
            with self._lock:
                self._target = interpolated
            time.sleep(1.0 / self.DRIVE_HZ)

    # ── drive loop (50 Hz) ─────────────────────────────────────────────────

    def _drive_loop(self) -> None:
        """Copies _target into _current and pushes to hardware."""
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                target  = self._target.copy()
                current = self._current.copy()

            # tiny alpha filter to eliminate any residual jitter
            alpha   = 0.35
            blended = {
                mid: current[mid] + alpha * (target[mid] - current[mid])
                for mid in target
            }

            with self._lock:
                self._current = blended

            self._push_to_robot(blended)
            time.sleep(dt)

    def _push_to_robot(self, angles: dict[int, float]) -> None:
        if self._robot is None:
            return   # simulation — nothing to push
        for mid, val in angles.items():
            name, lo, hi = MOTOR_MAP[mid]
            val = float(np.clip(val, lo, hi))
            try:
                motor = getattr(self._robot, name, None)
                if motor:
                    motor.goal_position = val
            except Exception:
                pass   # hardware error: swallow, keep running

    # ── math helpers ──────────────────────────────────────────────────────

    @staticmethod
    def _ease_in_out(t: float) -> float:
        """Smooth-step cubic easing: slow start, fast middle, slow end."""
        return t * t * (3 - 2 * t)


# ═══════════════════════════════════════════════════════════════════════════
#  SIMULATION VISUALISER  (shown when no robot is connected)
# ═══════════════════════════════════════════════════════════════════════════

class SimVisualiser:
    """Draws a small HUD showing current motor angles in an OpenCV window."""

    W, H = 420, 580
    GROUPS = [
        ("LEFT LEG",    [11, 12, 13, 14, 15]),
        ("RIGHT LEG",   [21, 22, 23, 24, 25]),
        ("TORSO/HEAD",  [31, 32, 33, 34, 35, 36, 37]),
        ("LEFT ARM",    [41, 42, 43, 44]),
        ("RIGHT ARM",   [51, 52, 53, 54]),
    ]

    def draw(self, angles: dict[int, float], gesture: str, frame=None) -> np.ndarray:
        canvas = np.zeros((self.H, self.W, 3), dtype=np.uint8)
        canvas[:] = (20, 20, 30)

        # gesture banner
        cv2.rectangle(canvas, (0, 0), (self.W, 36), (50, 130, 80), -1)
        cv2.putText(canvas, f"GESTURE: {gesture}", (8, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 255, 200), 2)

        y = 50
        for group_name, mids in self.GROUPS:
            cv2.putText(canvas, group_name, (8, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (120, 180, 255), 1)
            y += 16
            for mid in mids:
                name = MOTOR_MAP[mid][0]
                val  = angles.get(mid, 0.0)
                bar_w = int(np.interp(val, [-180, 180], [0, self.W - 100]))
                cv2.rectangle(canvas, (90, y - 11), (90 + bar_w, y + 1),
                               (60, 180, 100), -1)
                cv2.putText(canvas,
                            f"{name:<14} {val:+7.2f}°", (4, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.38, (220, 220, 220), 1)
                y += 17
            y += 4

        # optionally overlay the camera frame thumbnail
        if frame is not None:
            thumb = cv2.resize(frame, (140, 105))
            canvas[self.H - 110: self.H - 5,
                   self.W - 145: self.W - 5] = thumb

        return canvas


# ═══════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ═══════════════════════════════════════════════════════════════════════════


def _connect_robot():
    """
    Try multiple pypot connection strategies in order:
      1. pypot.creatures  -- the standard Poppy package (recommended)
      2. from_config(dict) -- manual JSON config file
    Returns a robot instance or None on failure.
    """
    import os, json

    # Strategy 1: pypot.creatures  (pip install poppy-humanoid)
    try:
        from pypot.creatures import PoppyHumanoid
        robot = PoppyHumanoid()
        robot.start_sync()
        print("[INFO] Poppy robot connected via pypot.creatures.")
        return robot
    except ImportError:
        pass   # poppy-humanoid package not installed
    except Exception as e:
        print(f"[WARN] pypot.creatures failed: {e}")

    # Strategy 2: manual JSON config file
    config_path = "poppy_config.json"
    if os.path.isfile(config_path):
        try:
            with open(config_path, "r") as f:
                config_dict = json.load(f)      # parse FIRST, then pass dict
            robot = pypot.robot.from_config(config_dict)
            robot.start_sync()
            print("[INFO] Poppy robot connected via poppy_config.json.")
            return robot
        except Exception as e:
            print(f"[WARN] from_config failed: {e}")
    else:
        print(f"[WARN] '{config_path}' not found.")

    print("[INFO] No robot detected -- running in SIMULATION mode.")
    return None


def main() -> None:
    # robot init
    robot = _connect_robot() if ROBOT_AVAILABLE else None

    # ── mediapipe setup ────────────────────────────────────────────────────
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.55,
    )

    # ── subsystems ─────────────────────────────────────────────────────────
    detector    = GestureDetector()
    controller  = MotionController(robot=robot)
    visualiser  = SimVisualiser()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return

    print("[INFO] Running — press  Q  to quit.")

    prev_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)   # mirror so left/right match user's POV
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = pose_model.process(rgb)
        rgb.flags.writeable = True
        annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        # ── gesture detection ──────────────────────────────────────────────
        gesture = detector.update(results.pose_landmarks)

        # ── motion controller ──────────────────────────────────────────────
        controller.set_gesture(gesture)

        # ── draw skeleton on camera frame ──────────────────────────────────
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                annotated,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 230, 80),  thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(200, 200, 0), thickness=2),
            )

        # FPS counter
        now      = time.time()
        fps      = 1.0 / max(now - prev_time, 1e-6)
        prev_time = now
        cv2.putText(annotated, f"FPS {fps:.1f}", (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 200), 2)

        # gesture label on camera window
        cv2.putText(annotated, gesture, (8, annotated.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 200, 255), 2)

        # ── robot HUD ──────────────────────────────────────────────────────
        hud = visualiser.draw(controller.current_angles, gesture, frame)

        # ── display ────────────────────────────────────────────────────────
        cv2.imshow("Gesture Input — Poppy Control", annotated)
        cv2.imshow("Robot State", hud)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # ── cleanup ────────────────────────────────────────────────────────────
    cap.release()
    cv2.destroyAllWindows()
    pose_model.close()
    if robot:
        controller.set_gesture("BALANCED_STAND")
        time.sleep(1.5)
        robot.stop_sync()
    print("[INFO] Shutdown complete.")


if __name__ == "__main__":
    main()

[WARN] 'poppy_config.json' not found.
[INFO] No robot detected -- running in SIMULATION mode.
[INFO] Running — press  Q  to quit.


In [ ]:
"""
Poppy Humanoid -- Gesture Recognition + Motion Execution System
==============================================================
Motor bus : pypot.dynamixel.DxlIO  (direct USB/U2D2 connection)
Vision    : MediaPipe Pose
Control   : LERP + smooth-step easing, 50 Hz drive loop

Dependencies:
    pip install opencv-python mediapipe pypot numpy

Hardware:
    USB2Dynamixel or U2D2 adapter connected to all Dynamixel motors.
    Set PORT and BAUDRATE below to match your setup.
"""

import sys
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ===========================================================================
#  HARDWARE CONFIG  <-- edit these for your setup
# ===========================================================================

PORT      = "COM4"   # Windows: "COM3"  |  macOS: "/dev/tty.usbserial-*"
BAUDRATE  = 1000000          # standard Poppy baud rate
MOTOR_IDS = [
    11, 12, 13, 14, 15,      # left leg
    21, 22, 23, 24, 25,      # right leg
    31, 32, 33, 34, 35, 36, 37,   # torso + head
    41, 42, 43, 44,          # left arm
    51, 52, 53, 54,          # right arm
]

# ---------------------------------------------------------------------------
try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot.dynamixel not found -- running in SIMULATION mode")
# ---------------------------------------------------------------------------


# ===========================================================================
#  MOTOR MAP   id -> (label, min_deg, max_deg)
# ===========================================================================

MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",      -30,  30),
    12: ("l_hip_z",      -30,  30),
    13: ("l_hip_y",      -50,  50),
    14: ("l_knee_y",       0,  90),
    15: ("l_ankle_y",    -50,  50),
    21: ("r_hip_x",      -30,  30),
    22: ("r_hip_z",      -30,  30),
    23: ("r_hip_y",      -50,  50),
    24: ("r_knee_y",     -90,   0),
    25: ("r_ankle_y",    -50,  50),
    31: ("abs_y",        -45,  45),
    32: ("abs_x",        -45,  45),
    33: ("abs_z",        -45,  45),
    34: ("bust_y",       -45,  45),
    35: ("bust_x",       -45,  45),
    36: ("head_z",       -50,  50),
    37: ("head_y",       -50,  50),
    41: ("l_shoulder_y", -90, 180),
    42: ("l_shoulder_x", -90,  90),
    43: ("l_arm_z",     -170, 170),
    44: ("l_elbow_y",      0, 170),
    51: ("r_shoulder_y", -90, 180),
    52: ("r_shoulder_x", -90,  90),
    53: ("r_arm_z",     -170, 170),
    54: ("r_elbow_y",   -170,   0),
}


# ===========================================================================
#  POSE LIBRARY   (degrees, keyed by motor-id)
# ===========================================================================

def _full_pose(overrides: dict[int, float]) -> dict[int, float]:
    """Clone BALANCED_STAND and apply per-motor overrides."""
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {
    "BALANCED_STAND": {
        11: -1.98,  12: -3.74,  13:  4.88,  14: 13.32,  15: -16.40,
        21:-22.81,  22:  6.64,  23:  0.48,  24:-10.24,  25:   6.55,
        31:  6.02,  32: -4.70,  33: -0.48,  34:-32.13,  35:   4.44,
        36: -3.08,  37:-26.83,
        41: 60.70,  42: 66.33,  43:-25.45,  44:  0.48,
        51:101.32,  52: 80.92,  53:104.13,  54:  7.34,
    },
}

POSES["RIGHT_WAVE_A"]  = _full_pose({51: -30.0, 52: 80.0, 53: 10.0, 54: -20.0})
POSES["RIGHT_WAVE_B"]  = _full_pose({51: -30.0, 52: 80.0, 53: 60.0, 54: -20.0})
POSES["LEFT_WAVE_A"]   = _full_pose({41: -30.0, 42: 66.0, 43: -60.0, 44: 0.0})
POSES["LEFT_WAVE_B"]   = _full_pose({41: -30.0, 42: 66.0, 43:  10.0, 44: 0.0})
POSES["BOTH_HANDS_UP"] = _full_pose({
    41: -80.0, 42: 10.0, 43: 0.0, 44: 0.0,
    51: -80.0, 52: 10.0, 53: 0.0, 54: 0.0,
})
POSES["HEAD_LEFT"]  = _full_pose({36:  30.0})
POSES["HEAD_RIGHT"] = _full_pose({36: -30.0})
POSES["CROSS_ARMS"] = _full_pose({
    41:  20.0, 42: -20.0, 43:  40.0, 44:  90.0,
    51:  20.0, 52:  20.0, 53: -40.0, 54: -90.0,
})


# ===========================================================================
#  DXL INTERFACE
# ===========================================================================

class DxlInterface:
    """
    Thin, safe wrapper around pypot.dynamixel.DxlIO.

    * Opens the serial port at the given baud rate.
    * Scans and verifies expected motor IDs.
    * Sets a safe moving speed on every motor.
    * Exposes set_positions({id: degrees}) for the drive loop.
    * safe_shutdown() disables torque and closes the port cleanly.
    """

    MOVING_SPEED_RPM = 30    # MX-series: 0-117 RPM  |  AX-series: 0-114 RPM

    def __init__(self, port: str, baudrate: int,
                 expected_ids: list[int]) -> None:

        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} at {baudrate} baud.")

        # scan the bus -- warn about missing motors but keep going
        found   = set(self._io.scan(expected_ids))
        missing = set(expected_ids) - found
        if missing:
            print(f"[DXL] WARNING -- motors not responding: {sorted(missing)}")
        self._active: set[int] = found
        print(f"[DXL] Active motors ({len(found)}): {sorted(found)}")

        # set uniform moving speed to prevent sudden jerks
        self._io.set_moving_speed(
            {mid: self.MOVING_SPEED_RPM for mid in self._active}
        )

        # enable torque on all active motors
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ENABLED.")

    def set_positions(self, angles: dict[int, float]) -> None:
        """
        Send goal positions to the bus.
        Skips inactive IDs; clamps every angle to its safe range.
        """
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if not payload:
            return
        try:
            self._io.set_goal_position(payload)
        except Exception as e:
            print(f"[DXL] set_goal_position error: {e}")

    def safe_shutdown(self) -> None:
        """Disable torque then close the port."""
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque DISABLED.")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed.")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    """Open DxlIO on PORT/BAUDRATE. Returns None if unavailable."""
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e}")
        print("[INFO] Falling back to SIMULATION mode.")
        return None


# ===========================================================================
#  GESTURE DETECTOR
# ===========================================================================

class GestureDetector:
    """
    Classifies MediaPipe PoseLandmarks into named robot commands.

    Outputs one of:
        BALANCED_STAND | RIGHT_WAVE | LEFT_WAVE |
        BOTH_HANDS_UP  | HEAD_LEFT  | HEAD_RIGHT | CROSS_ARMS

    Debounce: gesture must hold for DEBOUNCE_FRAMES consecutive frames.
    """

    DEBOUNCE_FRAMES     = 6      # ~0.2 s at 30 fps
    HEAD_TILT_THRESHOLD = 0.05   # normalised viewport units
    WAVE_X_HISTORY      = 8      # frames buffered for lateral-motion check

    def __init__(self) -> None:
        self._history: collections.deque[str] = collections.deque(
            maxlen=self.DEBOUNCE_FRAMES
        )
        self._wrist_x: dict[str, collections.deque] = {
            "right": collections.deque(maxlen=self.WAVE_X_HISTORY),
            "left":  collections.deque(maxlen=self.WAVE_X_HISTORY),
        }
        self.confirmed: str = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._history.append(raw)
        if (len(self._history) == self.DEBOUNCE_FRAMES
                and len(set(self._history)) == 1):
            self.confirmed = raw
        return self.confirmed

    def _classify(self, lm) -> str:
        PL = mp.solutions.pose.PoseLandmark

        nose       = lm[PL.NOSE]
        l_shoulder = lm[PL.LEFT_SHOULDER]
        r_shoulder = lm[PL.RIGHT_SHOULDER]
        l_wrist    = lm[PL.LEFT_WRIST]
        r_wrist    = lm[PL.RIGHT_WRIST]

        # CROSS ARMS (highest priority -- freeze command)
        l_near_r = (abs(l_wrist.x - r_shoulder.x) < 0.12
                    and abs(l_wrist.y - r_shoulder.y) < 0.15)
        r_near_l = (abs(r_wrist.x - l_shoulder.x) < 0.12
                    and abs(r_wrist.y - l_shoulder.y) < 0.15)
        if l_near_r and r_near_l:
            return "CROSS_ARMS"

        # HEAD TILT
        mid_x  = (l_shoulder.x + r_shoulder.x) / 2.0
        offset = nose.x - mid_x
        if offset >  self.HEAD_TILT_THRESHOLD:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_TILT_THRESHOLD:
            return "HEAD_LEFT"

        # BOTH HANDS UP
        l_up = l_wrist.y < l_shoulder.y - 0.05
        r_up = r_wrist.y < r_shoulder.y - 0.05
        if l_up and r_up:
            return "BOTH_HANDS_UP"

        # SINGLE-HAND WAVES (wrist above shoulder + lateral sweep)
        if r_up:
            self._wrist_x["right"].append(r_wrist.x)
            if self._lateral_motion("right"):
                return "RIGHT_WAVE"
        if l_up:
            self._wrist_x["left"].append(l_wrist.x)
            if self._lateral_motion("left"):
                return "LEFT_WAVE"

        return "BALANCED_STAND"

    def _lateral_motion(self, side: str) -> bool:
        h = self._wrist_x[side]
        return (len(h) == self.WAVE_X_HISTORY
                and (max(h) - min(h)) > 0.06)


# ===========================================================================
#  MOTION CONTROLLER
# ===========================================================================

class MotionController:
    """
    Converts gesture names into smooth Dynamixel motor motion.

    Internals:
    - Gesture thread   : walks through keyframe sequences with LERP
    - Drive loop (50Hz): low-pass filter + DxlInterface.set_positions()
    """

    TRANSITION_SEC = 0.30   # seconds per keyframe step
    DRIVE_HZ       = 50

    _SEQUENCES: dict[str, list[str]] = {
        "BALANCED_STAND": ["BALANCED_STAND"],
        "RIGHT_WAVE":     ["RIGHT_WAVE_A", "RIGHT_WAVE_B"],
        "LEFT_WAVE":      ["LEFT_WAVE_A",  "LEFT_WAVE_B"],
        "BOTH_HANDS_UP":  ["BOTH_HANDS_UP"],
        "HEAD_LEFT":      ["HEAD_LEFT"],
        "HEAD_RIGHT":     ["HEAD_RIGHT"],
        "CROSS_ARMS":     ["CROSS_ARMS"],
    }
    _LOOPING = {"RIGHT_WAVE", "LEFT_WAVE"}

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl  = dxl
        self._lock = threading.Lock()

        self._current: dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._target:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._active_gesture = "BALANCED_STAND"
        self._gesture_thread: threading.Thread | None = None
        self._stop_flag      = threading.Event()

        threading.Thread(target=self._drive_loop, daemon=True).start()

        # move to start position
        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(0.6)

    # -- public -------------------------------------------------------------

    def set_gesture(self, gesture: str) -> None:
        with self._lock:
            if gesture == self._active_gesture:
                return
            self._active_gesture = gesture

        self._stop_flag.set()
        if self._gesture_thread and self._gesture_thread.is_alive():
            self._gesture_thread.join(timeout=1.5)
        self._stop_flag.clear()

        self._gesture_thread = threading.Thread(
            target=self._run_gesture, args=(gesture,), daemon=True
        )
        self._gesture_thread.start()

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._current.copy()

    def shutdown(self) -> None:
        self._stop_flag.set()
        time.sleep(0.3)
        if self._dxl:
            self._dxl.safe_shutdown()

    # -- gesture thread -----------------------------------------------------

    def _run_gesture(self, gesture: str) -> None:
        frames  = self._SEQUENCES.get(gesture, ["BALANCED_STAND"])
        looping = gesture in self._LOOPING

        while not self._stop_flag.is_set():
            for name in frames:
                if self._stop_flag.is_set():
                    break
                self._lerp_to(POSES[name])
            if not looping:
                break

        if gesture != "BALANCED_STAND":
            self._lerp_to(POSES["BALANCED_STAND"])

    def _lerp_to(self, target: dict[int, float]) -> None:
        steps = max(int(self.TRANSITION_SEC * self.DRIVE_HZ), 1)
        with self._lock:
            start = self._current.copy()

        for i in range(1, steps + 1):
            if self._stop_flag.is_set():
                return
            t = self._smooth_step(i / steps)
            with self._lock:
                self._target = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    # -- 50 Hz drive loop ---------------------------------------------------

    def _drive_loop(self) -> None:
        dt    = 1.0 / self.DRIVE_HZ
        alpha = 0.35    # low-pass: higher = more responsive, lower = smoother

        while True:
            with self._lock:
                target  = self._target.copy()
                current = self._current.copy()

            blended = {
                mid: current[mid] + alpha * (target[mid] - current[mid])
                for mid in target
            }

            with self._lock:
                self._current = blended

            if self._dxl:
                self._dxl.set_positions(blended)

            time.sleep(dt)

    # -- math ---------------------------------------------------------------

    @staticmethod
    def _smooth_step(t: float) -> float:
        return t * t * (3.0 - 2.0 * t)


# ===========================================================================
#  SIMULATION HUD
# ===========================================================================

class SimVisualiser:
    W, H = 430, 595
    GROUPS = [
        ("LEFT LEG",   [11, 12, 13, 14, 15]),
        ("RIGHT LEG",  [21, 22, 23, 24, 25]),
        ("TORSO/HEAD", [31, 32, 33, 34, 35, 36, 37]),
        ("LEFT ARM",   [41, 42, 43, 44]),
        ("RIGHT ARM",  [51, 52, 53, 54]),
    ]

    def draw(self, angles: dict[int, float],
             gesture: str, live: bool, frame=None) -> np.ndarray:
        canvas = np.zeros((self.H, self.W, 3), dtype=np.uint8)
        canvas[:] = (18, 18, 28)

        bar_col = (40, 160, 60) if live else (160, 60, 40)
        cv2.rectangle(canvas, (0, 0), (self.W, 38), bar_col, -1)
        mode = "LIVE  DXL" if live else "SIMULATION"
        cv2.putText(canvas, f"{mode}  |  {gesture}", (8, 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.58, (230, 255, 230), 2)

        y = 52
        for grp, mids in self.GROUPS:
            cv2.putText(canvas, grp, (8, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.40, (100, 170, 255), 1)
            y += 16
            for mid in mids:
                label = MOTOR_MAP[mid][0]
                val   = angles.get(mid, 0.0)
                bw    = max(int(np.interp(val, [-180, 180], [0, self.W - 108])), 0)
                cv2.rectangle(canvas, (94, y - 11), (94 + bw, y + 2),
                               (50, 170, 90), -1)
                cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                            (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                            0.35, (215, 215, 215), 1)
                y += 17
            y += 5

        if frame is not None:
            th = cv2.resize(frame, (140, 105))
            canvas[self.H - 110:self.H - 5,
                   self.W - 145:self.W - 5] = th
        return canvas


# ===========================================================================
#  MAIN
# ===========================================================================

def main() -> None:

    # connect Dynamixel bus
    dxl  = connect_dxl()
    live = dxl is not None

    # MediaPipe
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.60,
        min_tracking_confidence=0.55,
    )

    detector   = GestureDetector()
    controller = MotionController(dxl=dxl)
    vis        = SimVisualiser()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        controller.shutdown()
        sys.exit(1)

    print(f"[INFO] Running ({'DXL LIVE' if live else 'SIMULATION'}) -- press Q to quit.")
    prev_t = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            gesture = detector.update(results.pose_landmarks)
            controller.set_gesture(gesture)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            now   = time.time()
            fps   = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now
            cv2.putText(annotated, f"FPS {fps:.1f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)
            cv2.putText(annotated, gesture,
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 200, 255), 2)

            hud_title = "Robot State (DXL LIVE)" if live else "Robot State (SIM)"
            hud = vis.draw(controller.current_angles, gesture, live, frame)
            cv2.imshow("Gesture Input", annotated)
            cv2.imshow(hud_title, hud)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Shutting down...")
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        if live:
            controller.set_gesture("BALANCED_STAND")
            time.sleep(1.5)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[DXL] Opened COM4 at 1000000 baud.
[DXL] Active motors (25): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Torque ENABLED.
[INFO] Running (DXL LIVE) -- press Q to quit.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          POPPY HUMANOID -- GESTURE CONTROL SYSTEM  v3.0                     ║
║  Motor bus : pypot.dynamixel.DxlIO  (direct USB / U2D2)                     ║
║  Vision    : MediaPipe Pose (webcam)                                         ║
║  Control   : per-gesture LERP + smooth-step easing, 50 Hz drive loop        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL:                                                                    ║
║    pip install opencv-python mediapipe pypot numpy                           ║
║                                                                              ║
║  RUN:                                                                        ║
║    python poppy_gesture_control.py                                           ║
║    (set PORT below before first run)                                         ║
║                                                                              ║
║  GESTURES:                                                                   ║
║    Both hands at hips          -> BALANCED STAND (default / reset)          ║
║    Right wrist above shoulder  -> RIGHT WAVE  (fast, natural)               ║
║    Left wrist above shoulder   -> LEFT WAVE                                 ║
║    Both wrists above shoulders -> BOTH HANDS UP  (celebration)              ║
║    Tilt head left / right      -> HEAD TURNS that direction                 ║
║    Cross wrists at chest       -> FREEZE (stop all motion)                  ║
║    T-POSE (arms level & wide)  -> BALANCED WALK  (in-place gait)            ║
║    Deep forward lean (nose     -> BOW                                       ║
║      significantly below ears)                                               ║
║                                                                              ║
║  KEYBOARD (while windows are open):                                          ║
║    Q  -> quit cleanly (robot returns to stand, torque off)                  ║
║    W  -> toggle walk on/off                                                 ║
║    S  -> force BALANCED_STAND immediately                                   ║
║    +  -> increase walk speed                                                ║
║    -  -> decrease walk speed                                                ║
╚══════════════════════════════════════════════════════════════════════════════╝

MOTOR REFERENCE (from robot diagram):
  LEFT LEG  : l_hip_x(11)  l_hip_z(12)  l_hip_y(13)  l_knee_y(14)  l_ankle_y(15)
  RIGHT LEG : r_hip_x(21)  r_hip_z(22)  r_hip_y(23)  r_knee_y(24)  r_ankle_y(25)
  TORSO     : abs_y(31)  abs_x(32)  abs_z(33)  bust_y(34)  bust_x(35)
  HEAD      : head_z(36)  head_y(37)
  LEFT ARM  : l_shoulder_y(41)  l_shoulder_x(42)  l_arm_z(43)  l_elbow_y(44)
  RIGHT ARM : r_shoulder_y(51)  r_shoulder_x(52)  r_arm_z(53)  r_elbow_y(54)

AXIS CONVENTION (Poppy Humanoid):
  shoulder_y  = forward / backward swing  (sagittal plane)
  shoulder_x  = raise / lower arm         (frontal plane, abduction)
  arm_z       = upper-arm rotation        (axial)
  elbow_y     = elbow flex / extend
  hip_z       = leg forward / backward    (sagittal)
  hip_x       = leg inward / outward      (frontal)
  knee_y      = knee flex
  ankle_y     = ankle pitch
  abs_x       = torso lateral tilt
  abs_y       = torso forward lean
  head_z      = head left / right turn
  head_y      = head nod
"""

import sys
import math
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ===========================================================================
#  HARDWARE CONFIG  <-- EDIT BEFORE FIRST RUN
# ===========================================================================

PORT      = "COM4"   # Windows: "COM3"  |  macOS: "/dev/tty.usbserial-*"
BAUDRATE  = 1000000          # standard Poppy baud rate
MOTOR_IDS = [
    11, 12, 13, 14, 15,
    21, 22, 23, 24, 25,
    31, 32, 33, 34, 35, 36, 37,
    41, 42, 43, 44,
    51, 52, 53, 54,
]

# ===========================================================================
#  DxlIO import
# ===========================================================================
try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot.dynamixel not found -- SIMULATION mode active")


# ===========================================================================
#  MOTOR MAP   id -> (label, hard_min_deg, hard_max_deg)
#  Limits are conservative safety bounds -- tune to your robot.
# ===========================================================================
MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    # LEFT LEG
    11: ("l_hip_x",      -25,  25),
    12: ("l_hip_z",      -40,  40),
    13: ("l_hip_y",      -40,  40),
    14: ("l_knee_y",       0,  80),
    15: ("l_ankle_y",    -40,  40),
    # RIGHT LEG
    21: ("r_hip_x",      -25,  25),
    22: ("r_hip_z",      -40,  40),
    23: ("r_hip_y",      -40,  40),
    24: ("r_knee_y",     -80,   0),
    25: ("r_ankle_y",    -40,  40),
    # TORSO
    31: ("abs_y",        -35,  35),
    32: ("abs_x",        -30,  30),
    33: ("abs_z",        -30,  30),
    34: ("bust_y",       -40,  40),
    35: ("bust_x",       -30,  30),
    # HEAD
    36: ("head_z",       -45,  45),
    37: ("head_y",       -40,  20),
    # LEFT ARM
    41: ("l_shoulder_y", -80, 160),
    42: ("l_shoulder_x", -80,  80),
    43: ("l_arm_z",     -150, 150),
    44: ("l_elbow_y",    -10, 160),
    # RIGHT ARM
    51: ("r_shoulder_y", -80, 160),
    52: ("r_shoulder_x", -80,  80),
    53: ("r_arm_z",     -150, 150),
    54: ("r_elbow_y",   -160,  10),
}


# ===========================================================================
#  POSE LIBRARY
#  All angles in degrees, keyed by motor-id.
#  BALANCED_STAND is the single source of truth -- every other pose is built
#  by _p() which clones it and applies per-motor overrides.
# ===========================================================================

def _p(overrides: dict[int, float]) -> dict[int, float]:
    """Clone BALANCED_STAND and apply overrides."""
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {

    # -----------------------------------------------------------------------
    #  DEFAULT / RETURN STATE
    # -----------------------------------------------------------------------
    "BALANCED_STAND": {
        # left leg
        11: -1.98,   12: -3.74,   13:  4.88,   14: 13.32,  15: -16.40,
        # right leg
        21: -22.81,  22:  6.64,   23:  0.48,   24: -10.24, 25:   6.55,
        # torso
        31:   6.02,  32: -4.70,   33: -0.48,   34: -32.13, 35:   4.44,
        # head
        36:  -3.08,  37: -26.83,
        # left arm
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        # right arm
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ---------------------------------------------------------------------------
#  RIGHT WAVE
#  Motion: arm lifts, elbow bends, arm_z oscillates  (~2 Hz at 0.13s/step)
#  shoulder_y:  bring arm forward from 101 -> ~25 (clearly in front)
#  shoulder_x:  raise arm high  80 -> 70
#  arm_z:       rotate forearm  104 -> alternate 60 / 130  (wrist flick)
#  elbow_y:     slight bend  7 -> -55  (natural wave position)
# ---------------------------------------------------------------------------
POSES["R_WAVE_RAISE"] = _p({
    51:  20.0,   # forward
    52:  65.0,   # raised
    54: -70.0,   # elbow bent (NEGATIVE for right)
})

POSES["R_WAVE_A"] = _p({
    51:  10.0,   # move slightly inward
    52:  65.0,
    54: -80.0,   # more bend
})

POSES["R_WAVE_B"] = _p({
    51:  30.0,   # move outward
    52:  65.0,
    54: -50.0,   # less bend
})
# ---------------------------------------------------------------------------
#  LEFT WAVE
#  Mirror of right wave.
#  shoulder_y: 60 -> -20 (arm forward)
#  arm_z: -25 -> alternate -80 / 20
#  elbow_y: 0 -> 55 (bent)
# ---------------------------------------------------------------------------
POSES["L_WAVE_RAISE"] = _p({
    41: -20.0,
    42:  65.0,
    44:  70.0,   # POSITIVE for left
})

POSES["L_WAVE_A"] = _p({
    41: -10.0,
    42:  65.0,
    44:  80.0,
})

POSES["L_WAVE_B"] = _p({
    41: -30.0,
    42:  65.0,
    44:  50.0,
})
# ---------------------------------------------------------------------------
#  BOTH HANDS UP (celebration)
# ---------------------------------------------------------------------------
POSES["BOTH_HANDS_UP"] = _p({
    41: -80.0,  42: 10.0,  44: 20.0,   # left
    51: -80.0,  52: 10.0,  54: -20.0,  # right (mirror)
    34: -10.0,
})

# ---------------------------------------------------------------------------
#  HEAD TURNS
# ---------------------------------------------------------------------------
POSES["HEAD_LEFT"]  = _p({36:  38.0})
POSES["HEAD_RIGHT"] = _p({36: -38.0})

# ---------------------------------------------------------------------------
#  FREEZE / CROSS ARMS
# ---------------------------------------------------------------------------
POSES["CROSS_ARMS"] = _p({
    41:  15.0,  42: -25.0,  43:  50.0,  44: 100.0,
    51:  15.0,  52:  25.0,  53: -50.0,  54:-100.0,
})

# ---------------------------------------------------------------------------
#  BOW
# ---------------------------------------------------------------------------
POSES["BOW"] = _p({
    31:  30.0,  34:  35.0,   # forward lean torso + bust
    37:  15.0,               # head nods down
})

# ---------------------------------------------------------------------------
#  WALK -- keyframes are generated dynamically by the gait engine below
#  These are the NEUTRAL walk stance (feet slightly apart, arms ready)
# ---------------------------------------------------------------------------
POSES["WALK_NEUTRAL"] = _p({
    31:   8.0,               # abs slight forward lean
    34: -25.0,               # bust upright
})


# ===========================================================================
#  DXL INTERFACE
# ===========================================================================

class DxlInterface:
    """
    Thin, safe wrapper around pypot DxlIO.
    Handles scan, torque-enable, angle clamping, and clean shutdown.
    """

    MOVING_SPEED_RPM = 50    # MX-series: 0-117 RPM.  Increase for faster motion.

    def __init__(self, port: str, baudrate: int, expected_ids: list[int]) -> None:
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")

        found   = set(self._io.scan(expected_ids))
        missing = set(expected_ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing motors: {sorted(missing)}")
        self._active: set[int] = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")

        self._io.set_moving_speed(
            {mid: self.MOVING_SPEED_RPM for mid in self._active}
        )
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: dict[int, float]) -> None:
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed ({e}) -- SIMULATION mode")
        return None


# ===========================================================================
#  GAIT ENGINE  (sinusoidal balanced walk)
# ===========================================================================

class GaitEngine:
    """
    Generates smooth sinusoidal bipedal gait keyframes.

    The gait cycle is divided into 360 degrees (0..2pi).
    Right leg leads at phase=0, left leg leads at phase=pi.

    Motors used:
      hip_z    : forward/back leg swing   (main propulsion)
      hip_x    : inward/outward (stability)
      knee_y   : swing-phase knee lift
      ankle_y  : ankle compensation
      abs_x    : lateral weight shift
      shoulder_y: arm counter-swing
    """

    # ----- tune these for your robot ----------------------------------------
    HIP_Z_AMP    = 14.0   # deg, leg forward/back swing
    HIP_X_AMP    =  6.0   # deg, lateral hip shift
    KNEE_LIFT    = 18.0   # deg, knee bend during swing
    ANKLE_AMP    =  6.0   # deg, ankle pitch
    ABS_X_AMP    =  5.5   # deg, lateral torso lean
    ARM_SWING    = 18.0   # deg, counter-swing shoulder_y
    FORWARD_LEAN =  6.0   # deg, abs_y forward tilt while walking
    # ------------------------------------------------------------------------

    def __init__(self) -> None:
        self._phase = 0.0           # 0 .. 2*pi
        self.speed  = 1.0           # cycles per second (tune: 0.5-1.5)

    def step(self, dt: float) -> dict[int, float]:
        """Advance phase by dt seconds, return a full pose dict."""
        self._phase += 2.0 * math.pi * self.speed * dt
        if self._phase > 2.0 * math.pi:
            self._phase -= 2.0 * math.pi
        return self._pose_at(self._phase)

    def reset(self) -> None:
        self._phase = 0.0

    def _pose_at(self, p: float) -> dict[int, float]:
        base = POSES["BALANCED_STAND"].copy()

        sp = math.sin(p)
        cp = math.cos(p)

        # --- LEG SWINGS (hip_z) ---
        # right leg forward when sp > 0
        base[22] += self.HIP_Z_AMP * sp      # r_hip_z
        base[12] -= self.HIP_Z_AMP * sp      # l_hip_z  (opposite phase)

        # --- KNEE LIFT (only on swing phase, not stance) ---
        # Right knee bends when leg is forward (sp > 0)
        r_knee_lift = self.KNEE_LIFT * max(0.0, sp)
        l_knee_lift = self.KNEE_LIFT * max(0.0, -sp)
        base[24] -= r_knee_lift    # r_knee_y (negative = flex for right)
        base[14] += l_knee_lift    # l_knee_y (positive = flex for left)

        # --- ANKLE COMPENSATION ---
        base[25] += self.ANKLE_AMP * sp      # r_ankle_y
        base[15] -= self.ANKLE_AMP * sp      # l_ankle_y

        # --- LATERAL HIP SHIFT (weight transfer) ---
        # Shift weight to stance leg (opposite of swing)
        base[21] += self.HIP_X_AMP * (-cp)   # r_hip_x
        base[11] += self.HIP_X_AMP * ( cp)   # l_hip_x

        # --- TORSO LATERAL LEAN (follows weight shift) ---
        base[32] += self.ABS_X_AMP * (-cp)   # abs_x

        # --- FORWARD LEAN ---
        base[31]  = base[31] + self.FORWARD_LEAN  # abs_y

        # --- ARM COUNTER-SWING ---
        # Right arm swings BACK when right leg is forward
        base[51] += self.ARM_SWING * (-sp)   # r_shoulder_y
        base[41] += self.ARM_SWING * ( sp)   # l_shoulder_y

        return base


# ===========================================================================
#  GESTURE DETECTOR
# ===========================================================================

class GestureDetector:
    """
    Classifies MediaPipe PoseLandmarks into named commands.

    Priority order (highest first):
      CROSS_ARMS > BOW > T_POSE_WALK > HEAD_TILT >
      BOTH_HANDS_UP > RIGHT_WAVE > LEFT_WAVE > BALANCED_STAND

    Uses DEBOUNCE_FRAMES to prevent jitter / accidental triggers.
    """

    DEBOUNCE_FRAMES     = 5       # ~0.17 s at 30 fps
    HEAD_TILT_THRESHOLD = 0.055
    WAVE_X_HISTORY      = 7
    # wave: wrist must be this far above shoulder (normalised)
    WAVE_MIN_HEIGHT     = 0.04
    # T-pose walk: wrists spread this far apart (normalised width)
    TPOSE_SPREAD        = 0.45
    TPOSE_HEIGHT_TOL    = 0.10    # wrist within ±10% of shoulder height
    BOW_THRESHOLD       = 0.12    # nose this far BELOW ear midpoint

    def __init__(self) -> None:
        self._history: collections.deque = collections.deque(
            maxlen=self.DEBOUNCE_FRAMES)
        self._wrist_x: dict[str, collections.deque] = {
            "right": collections.deque(maxlen=self.WAVE_X_HISTORY),
            "left":  collections.deque(maxlen=self.WAVE_X_HISTORY),
        }
        self.confirmed = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._history.append(raw)
        if (len(self._history) == self.DEBOUNCE_FRAMES
                and len(set(self._history)) == 1):
            self.confirmed = raw
        return self.confirmed

    def _classify(self, lm) -> str:
        PL = mp.solutions.pose.PoseLandmark

        nose        = lm[PL.NOSE]
        l_ear       = lm[PL.LEFT_EAR]
        r_ear       = lm[PL.RIGHT_EAR]
        l_shoulder  = lm[PL.LEFT_SHOULDER]
        r_shoulder  = lm[PL.RIGHT_SHOULDER]
        l_elbow     = lm[PL.LEFT_ELBOW]
        r_elbow     = lm[PL.RIGHT_ELBOW]
        l_wrist     = lm[PL.LEFT_WRIST]
        r_wrist     = lm[PL.RIGHT_WRIST]
        l_hip_lm    = lm[PL.LEFT_HIP]
        r_hip_lm    = lm[PL.RIGHT_HIP]

        # ── 1. CROSS ARMS / FREEZE ─────────────────────────────────────────
        l_near_r = (abs(l_wrist.x - r_shoulder.x) < 0.13
                    and abs(l_wrist.y - r_shoulder.y) < 0.16)
        r_near_l = (abs(r_wrist.x - l_shoulder.x) < 0.13
                    and abs(r_wrist.y - l_shoulder.y) < 0.16)
        if l_near_r and r_near_l:
            return "CROSS_ARMS"

        # ── 2. BOW (deep forward lean: nose below ears) ────────────────────
        ear_mid_y = (l_ear.y + r_ear.y) / 2.0
        if nose.y > ear_mid_y + self.BOW_THRESHOLD:
            return "BOW"

        # ── 3. T-POSE → WALK ──────────────────────────────────────────────
        # Both wrists roughly at shoulder height AND far apart
        l_level = abs(l_wrist.y - l_shoulder.y) < self.TPOSE_HEIGHT_TOL
        r_level = abs(r_wrist.y - r_shoulder.y) < self.TPOSE_HEIGHT_TOL
        wide    = abs(l_wrist.x - r_wrist.x) > self.TPOSE_SPREAD
        if l_level and r_level and wide:
            return "WALK"

        # ── 4. HEAD TILT ───────────────────────────────────────────────────
        mid_x  = (l_shoulder.x + r_shoulder.x) / 2.0
        offset = nose.x - mid_x
        if offset >  self.HEAD_TILT_THRESHOLD:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_TILT_THRESHOLD:
            return "HEAD_LEFT"

        # ── 5. BOTH HANDS UP ──────────────────────────────────────────────
        l_up = l_wrist.y < l_shoulder.y - self.WAVE_MIN_HEIGHT
        r_up = r_wrist.y < r_shoulder.y - self.WAVE_MIN_HEIGHT
        if l_up and r_up:
            return "BOTH_HANDS_UP"

        # ── 6. SINGLE WAVES ───────────────────────────────────────────────
        if r_up:
            self._wrist_x["right"].append(r_wrist.x)
            if self._lateral_motion("right"):
                return "RIGHT_WAVE"
        if l_up:
            self._wrist_x["left"].append(l_wrist.x)
            if self._lateral_motion("left"):
                return "LEFT_WAVE"

        return "BALANCED_STAND"

    def _lateral_motion(self, side: str) -> bool:
        h = self._wrist_x[side]
        return (len(h) == self.WAVE_X_HISTORY
                and (max(h) - min(h)) > 0.055)


# ===========================================================================
#  MOTION CONTROLLER
# ===========================================================================

class MotionController:
    """
    Converts gesture names into smooth Dynamixel motor commands.

    Key design:
    - Each gesture runs in its own daemon thread.
    - Waves use fast transitions (0.13 s/step) for natural speed.
    - Walk uses the GaitEngine at 50 Hz, no keyframes.
    - A low-pass filter (alpha) in the drive loop kills jitter.
    - Every gesture returns to BALANCED_STAND on exit.
    """

    DRIVE_HZ = 50

    # seconds per lerp step -- shorter = faster motion
    STEP_TIME: dict[str, float] = {
        "BALANCED_STAND": 0.28,
        "RIGHT_WAVE":     0.13,   # fast natural wave
        "LEFT_WAVE":      0.13,
        "BOTH_HANDS_UP":  0.38,
        "HEAD_LEFT":      0.22,
        "HEAD_RIGHT":     0.22,
        "CROSS_ARMS":     0.30,
        "BOW":            0.45,
        "WALK":           0.04,   # walk uses gait engine, not keyframes
    }

    _SEQUENCES: dict[str, list[str]] = {
        "BALANCED_STAND": ["BALANCED_STAND"],
        # Wave: raise arm first (1x), then oscillate A<->B
        "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],
        "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
        "BOTH_HANDS_UP":  ["BOTH_HANDS_UP"],
        "HEAD_LEFT":      ["HEAD_LEFT"],
        "HEAD_RIGHT":     ["HEAD_RIGHT"],
        "CROSS_ARMS":     ["CROSS_ARMS"],
        "BOW":            ["BOW"],
        "WALK":           [],    # handled by gait engine
    }
    # gestures that loop their keyframe sequences
    _LOOPING = {"RIGHT_WAVE", "LEFT_WAVE", "WALK"}

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl  = dxl
        self._lock = threading.Lock()

        self._current: dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._target:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._active_gesture = "BALANCED_STAND"
        self._gesture_thread: threading.Thread | None = None
        self._stop_flag      = threading.Event()

        self._gait = GaitEngine()

        # low-pass filter coefficient: higher = faster, lower = smoother
        self._alpha = 0.40

        threading.Thread(target=self._drive_loop, daemon=True).start()

        # move to start position on boot
        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(0.8)

    # ── public API ──────────────────────────────────────────────────────────

    def set_gesture(self, gesture: str) -> None:
        with self._lock:
            if gesture == self._active_gesture:
                return
            self._active_gesture = gesture

        self._stop_flag.set()
        if self._gesture_thread and self._gesture_thread.is_alive():
            self._gesture_thread.join(timeout=2.0)
        self._stop_flag.clear()

        self._gesture_thread = threading.Thread(
            target=self._run_gesture, args=(gesture,), daemon=True
        )
        self._gesture_thread.start()

    def set_walk_speed(self, delta: float) -> None:
        self._gait.speed = float(np.clip(self._gait.speed + delta, 0.3, 2.0))
        print(f"[WALK] speed = {self._gait.speed:.2f} cyc/s")

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._current.copy()

    @property
    def walk_speed(self) -> float:
        return self._gait.speed

    def shutdown(self) -> None:
        self._stop_flag.set()
        time.sleep(0.3)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ── gesture thread ───────────────────────────────────────────────────────

    def _run_gesture(self, gesture: str) -> None:
        looping  = gesture in self._LOOPING
        frames   = self._SEQUENCES.get(gesture, ["BALANCED_STAND"])
        step_t   = self.STEP_TIME.get(gesture, 0.28)

        if gesture == "WALK":
            self._walk_loop(step_t)
        elif looping:
            # play first keyframe once (raise), then loop remaining ones
            if len(frames) > 1:
                self._lerp_to(POSES[frames[0]], step_t)
                loop_frames = frames[1:]
            else:
                loop_frames = frames
            while not self._stop_flag.is_set():
                for name in loop_frames:
                    if self._stop_flag.is_set():
                        break
                    self._lerp_to(POSES[name], step_t)
        else:
            for name in frames:
                if self._stop_flag.is_set():
                    break
                self._lerp_to(POSES[name], step_t)

        # always return to balanced stand on exit
        if gesture != "BALANCED_STAND":
            self._lerp_to(POSES["BALANCED_STAND"],
                          self.STEP_TIME["BALANCED_STAND"])

    def _walk_loop(self, dt: float) -> None:
        """Run sinusoidal gait until stop flag is set."""
        self._gait.reset()
        prev = time.time()
        # first move to walk neutral stance
        self._lerp_to(POSES["WALK_NEUTRAL"], 0.35)

        while not self._stop_flag.is_set():
            now     = time.time()
            elapsed = now - prev
            prev    = now
            gait_pose = self._gait.step(elapsed)
            with self._lock:
                self._target = gait_pose
            time.sleep(1.0 / self.DRIVE_HZ)

    # ── LERP helper ──────────────────────────────────────────────────────────

    def _lerp_to(self, target: dict[int, float], step_t: float) -> None:
        steps = max(int(step_t * self.DRIVE_HZ), 1)
        with self._lock:
            start = self._current.copy()

        for i in range(1, steps + 1):
            if self._stop_flag.is_set():
                return
            t = self._smooth_step(i / steps)
            with self._lock:
                self._target = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    # ── 50 Hz drive loop ─────────────────────────────────────────────────────

    def _drive_loop(self) -> None:
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                target  = self._target.copy()
                current = self._current.copy()

            blended = {
                mid: current[mid] + self._alpha * (target[mid] - current[mid])
                for mid in target
            }
            with self._lock:
                self._current = blended

            if self._dxl:
                self._dxl.set_positions(blended)

            time.sleep(dt)

    # ── math ─────────────────────────────────────────────────────────────────

    @staticmethod
    def _smooth_step(t: float) -> float:
        return t * t * (3.0 - 2.0 * t)


# ===========================================================================
#  SIMULATION HUD
# ===========================================================================

class SimVisualiser:
    W, H = 450, 610

    GROUPS = [
        ("LEFT LEG",   [11, 12, 13, 14, 15]),
        ("RIGHT LEG",  [21, 22, 23, 24, 25]),
        ("TORSO",      [31, 32, 33, 34, 35]),
        ("HEAD",       [36, 37]),
        ("LEFT ARM",   [41, 42, 43, 44]),
        ("RIGHT ARM",  [51, 52, 53, 54]),
    ]

    GESTURE_COLORS = {
        "BALANCED_STAND": (40, 160,  60),
        "RIGHT_WAVE":     (60, 200, 255),
        "LEFT_WAVE":      (60, 200, 255),
        "BOTH_HANDS_UP":  (60, 255, 200),
        "HEAD_LEFT":      (200, 160, 60),
        "HEAD_RIGHT":     (200, 160, 60),
        "CROSS_ARMS":     (60,  60, 200),
        "BOW":            (180, 60, 255),
        "WALK":           (255, 160,  0),
    }

    def draw(self, angles: dict[int, float], gesture: str,
             live: bool, walk_speed: float, frame=None) -> np.ndarray:
        canvas = np.zeros((self.H, self.W, 3), dtype=np.uint8)
        canvas[:] = (16, 16, 26)

        # header
        col = self.GESTURE_COLORS.get(gesture, (120, 120, 120))
        cv2.rectangle(canvas, (0, 0), (self.W, 42), col, -1)
        mode_str = "DXL LIVE" if live else "SIMULATION"
        walk_str = f"  walk:{walk_speed:.1f}hz" if gesture == "WALK" else ""
        cv2.putText(canvas, f"{mode_str}  |  {gesture}{walk_str}",
                    (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.56, (10, 10, 10), 2)

        y = 56
        for grp, mids in self.GROUPS:
            cv2.putText(canvas, grp, (8, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                        (100, 170, 255), 1)
            y += 15
            for mid in mids:
                label = MOTOR_MAP[mid][0]
                val   = angles.get(mid, 0.0)
                bw = max(int(np.interp(val, [-180, 180], [0, self.W - 112])), 0)
                cv2.rectangle(canvas, (96, y - 10), (96 + bw, y + 2),
                               (45, 165, 90), -1)
                cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                            (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                            0.34, (210, 210, 210), 1)
                y += 16
            y += 4

        if frame is not None:
            th = cv2.resize(frame, (140, 105))
            canvas[self.H - 110:self.H - 5,
                   self.W - 145:self.W - 5] = th
        return canvas


# ===========================================================================
#  MAIN
# ===========================================================================

def main() -> None:

    dxl  = connect_dxl()
    live = dxl is not None
    if not live:
        print("[INFO] SIMULATION mode -- no hardware commands will be sent")

    # MediaPipe
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.62,
        min_tracking_confidence=0.55,
    )

    detector   = GestureDetector()
    controller = MotionController(dxl=dxl)
    vis        = SimVisualiser()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("[INFO] Running -- Q=quit  W=toggle walk  S=stand  +/-=walk speed")

    prev_t       = time.time()
    walk_toggled = False   # keyboard walk toggle

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            # gesture detection (can be overridden by keyboard toggle)
            pose_gesture = detector.update(results.pose_landmarks)
            gesture = "WALK" if walk_toggled else pose_gesture
            controller.set_gesture(gesture)

            # skeleton overlay
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            # FPS
            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now
            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)

            # gesture label on camera
            g_col = vis.GESTURE_COLORS.get(gesture, (200, 200, 200))
            cv2.putText(annotated, gesture,
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, g_col, 2)
            if walk_toggled:
                cv2.putText(annotated, "[W] WALK ON",
                            (annotated.shape[1] - 180, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 255), 2)

            hud = vis.draw(controller.current_angles, gesture,
                           live, controller.walk_speed, frame)
            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State", hud)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("w"):
                walk_toggled = not walk_toggled
                print(f"[KEY] Walk toggled: {walk_toggled}")
            elif key == ord("s"):
                walk_toggled = False
                controller.set_gesture("BALANCED_STAND")
                print("[KEY] Force BALANCED_STAND")
            elif key == ord("+") or key == ord("="):
                controller.set_walk_speed(+0.1)
            elif key == ord("-"):
                controller.set_walk_speed(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Shutting down -- returning to stand...")
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        walk_toggled = False
        controller.set_gesture("BALANCED_STAND")
        if live:
            time.sleep(1.8)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[DXL] Opened COM4 @ 1000000 baud
[DXL] Active (25): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Torque ON
[INFO] Running -- Q=quit  W=toggle walk  S=stand  +/-=walk speed


In [ ]:
import time
from pypot.dynamixel import DxlIO

# ========= CONFIG =========
PORT = "COM4"
BAUDRATE = 1000000

MOTOR_IDS = [
    11,12,13,14,15,
    21,22,23,24,25,
    31,32,33,34,35,
    36,37,
    41,42,43,44,
    51,52,53,54
]

labels = {
    11:"l_hip_x",12:"l_hip_z",13:"l_hip_y",14:"l_knee_y",15:"l_ankle_y",
    21:"r_hip_x",22:"r_hip_z",23:"r_hip_y",24:"r_knee_y",25:"r_ankle_y",
    31:"abs_y",32:"abs_x",33:"abs_z",34:"bust_y",35:"bust_x",
    36:"head_z",37:"head_y",
    41:"l_shoulder_y",42:"l_shoulder_x",43:"l_arm_z",44:"l_elbow_y",
    51:"r_shoulder_y",52:"r_shoulder_x",53:"r_arm_z",54:"r_elbow_y"
}

# ========= CONNECT =========
dxl = DxlIO(PORT, baudrate=BAUDRATE)
ids = dxl.scan(MOTOR_IDS)

print("Connected Motors:", ids)

# ========= TORQUE OFF =========
dxl.disable_torque(ids)
print("\n🔓 Torque OFF → Move robot freely")

print("\nInstructions:")
print("1. Set pose manually")
print("2. Press ENTER to capture")
print("3. Type pose name")
print("4. Repeat\n")

try:
    while True:
        input("👉 Set pose and press ENTER...")

        angles = dxl.get_present_position(ids)

        pose_name = input("Enter pose name (or 'q' to quit): ").strip()

        if pose_name.lower() == "q":
            break

        print(f"\n=== {pose_name.upper()} ===")
        print("{")

        for i, mid in enumerate(ids):
            print(f"    {mid}: {angles[i]:.2f},   # {labels[mid]}")

        print("}\n")
        print("✅ Pose captured!\n")

except KeyboardInterrupt:
    pass

# ========= RESTORE =========
dxl.enable_torque(ids)
dxl.close()

print("\n✅ Done")

Connected Motors: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]

🔓 Torque OFF → Move robot freely

Instructions:
1. Set pose manually
2. Press ENTER to capture
3. Type pose name
4. Repeat



👉 Set pose and press ENTER... 
Enter pose name (or 'q' to quit):  right hand up



=== RIGHT HAND UP ===
{
    11: -0.31,   # l_hip_x
    12: -1.98,   # l_hip_z
    13: 4.97,   # l_hip_y
    14: 18.33,   # l_knee_y
    15: -16.13,   # l_ankle_y
    21: -11.38,   # r_hip_x
    22: -4.35,   # r_hip_z
    23: 0.40,   # r_hip_y
    24: -10.33,   # r_knee_y
    25: 6.37,   # r_ankle_y
    31: 3.30,   # abs_y
    32: -4.53,   # abs_x
    33: -0.48,   # abs_z
    34: -57.27,   # bust_y
    35: 6.46,   # bust_x
    36: -2.79,   # head_z
    37: -27.42,   # head_y
    41: 49.27,   # l_shoulder_y
    42: 65.45,   # l_shoulder_x
    43: -25.27,   # l_arm_z
    44: 0.66,   # l_elbow_y
    51: 111.60,   # r_shoulder_y
    52: 11.47,   # r_shoulder_x
    53: -88.75,   # r_arm_z
    54: 92.00,   # r_elbow_y
}

✅ Pose captured!



In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        POPPY HUMANOID  --  GESTURE CONTROL  v6.0                            ║
║  Port    : COM4  (DxlIO direct)                                              ║
║  Vision  : MediaPipe Pose                                                    ║
║  Drive   : 50 Hz LERP  +  smooth-step easing                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL :  pip install opencv-python mediapipe pypot numpy                  ║
║  RUN     :  python poppy_gesture_control.py                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  CHANGES v6.0                                                                ║
║  - Wave oscillation: 180 ms peak pause between each A→B swing               ║
║  - Non-looping gestures (SALUTE, HANDS_UP, HANDS_FRONT, etc.) hold their    ║
║    final pose for a configurable duration before returning to stand          ║
║  - Priority gating: a lower-priority gesture cannot interrupt a higher-      ║
║    priority one mid-execute; only strictly-higher priority breaks a hold     ║
║  - _interruptible_sleep: hold checks _stop every 50 ms for clean exit       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  GESTURES (stand in front of webcam)                                         ║
║  -----------------------------------------------------------------------     ║
║  Hands at hips                     →  BALANCED STAND (reset)               ║
║  Right wrist above right shoulder  →  RIGHT WAVE (motor 54 oscillates)     ║
║  Left  wrist above left  shoulder  →  LEFT  WAVE (motor 44 oscillates)     ║
║  Both wrists above shoulders       →  HANDS UP                             ║
║  Both arms pushed forward (chest)  →  HANDS FRONT                          ║
║  Right hand near forehead          →  SALUTE                                ║
║  Tilt head left                    →  ROBOT HEAD LEFT                      ║
║  Tilt head right                   →  ROBOT HEAD RIGHT                     ║
║  Cross wrists at chest             →  FREEZE                                ║
║  T-pose (arms wide & level)        →  WALK  (4-frame keyframe cycle)       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEYBOARD                                                                    ║
║  W  toggle walk on/off  |  S  force stand  |  Q  quit                      ║
║  +  walk faster         |  -  walk slower                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import sys
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE
# ══════════════════════════════════════════════════════════════════════════════

PORT      = "COM4"
BAUDRATE  = 1000000
MOTOR_IDS = [11,12,13,14,15, 21,22,23,24,25,
             31,32,33,34,35,36,37,
             41,42,43,44, 51,52,53,54]

try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot not found -- SIMULATION mode")


# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR MAP  --  limits computed from every user-provided angle + 10° margin
# ══════════════════════════════════════════════════════════════════════════════

MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    # id : (name,              min,   max)
    11: ("l_hip_x",           -23,    22),
    12: ("l_hip_z",           -33,    15),
    13: ("l_hip_y",           -37,    44),
    14: ("l_knee_y",            3,    61),
    15: ("l_ankle_y",         -36,     5),
    21: ("r_hip_x",           -49,     0),
    22: ("r_hip_z",           -22,    17),
    23: ("r_hip_y",           -22,    46),
    24: ("r_knee_y",          -57,    14),
    25: ("r_ankle_y",         -20,    30),
    31: ("abs_y",             -30,    17),
    32: ("abs_x",             -25,    28),
    33: ("abs_z",             -22,    19),
    34: ("bust_y",            -43,    -1),
    35: ("bust_x",            -16,    21),
    36: ("head_z",            -48,    48),
    37: ("head_y",            -37,     7),
    41: ("l_shoulder_y",     -137,    77),
    42: ("l_shoulder_x",      -22,    85),
    43: ("l_arm_z",           -36,   115),
    44: ("l_elbow_y",        -146,    11),
    51: ("r_shoulder_y",      -84,   121),
    52: ("r_shoulder_x",     -111,    91),
    53: ("r_arm_z",          -105,   115),
    54: ("r_elbow_y",        -114,    30),
}


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY
# ══════════════════════════════════════════════════════════════════════════════

def _p(overrides: dict[int, float]) -> dict[int, float]:
    """Clone BALANCED_STAND and apply overrides."""
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {

    # ── BALANCED STAND (default / return state) ────────────────────────────
    "BALANCED_STAND": {
        11:  -1.98,  12:  -3.74,  13:   4.88,  14:  13.32,  15: -16.40,
        21: -22.81,  22:   6.64,  23:   0.48,  24: -10.24,  25:   6.55,
        31:   6.02,  32:  -4.70,  33:  -0.48,  34: -32.13,  35:   4.44,
        36:  -3.08,  37: -26.83,
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ── LEFT WAVE ──────────────────────────────────────────────────────────────
POSES["L_WAVE_RAISE"] = _p({41: -127.0, 42: -11.30, 43: 90.0, 44: -40.0})
POSES["L_WAVE_A"]     = _p({41: -127.0, 42: -11.30, 43: 90.0, 44: -95.52})
POSES["L_WAVE_B"]     = _p({41: -120.0, 42: -11.30, 43: 90.0, 44: -30.0})

# ── RIGHT WAVE ─────────────────────────────────────────────────────────────
POSES["R_WAVE_RAISE"] = _p({51: -72.57, 52: -27.63, 53: 76.79, 54: -40.0})
POSES["R_WAVE_A"]     = _p({51: -72.57, 52: -27.63, 53: 76.79, 54: -103.43})
POSES["R_WAVE_B"]     = _p({51: -72.57, 52: -27.63, 53: 76.79, 54: -35.0})

# ── HANDS UP ───────────────────────────────────────────────────────────────
POSES["HANDS_UP"] = _p({
    41: -112.40,  42:  35.03,  43: -13.58,  44:  0.75,
    51:  110.02,  52: -56.84,  53:   9.80,  54:  0.40,
})

# ── HANDS FRONT ────────────────────────────────────────────────────────────
POSES["HANDS_FRONT"] = _p({
    41: -32.84,  42:  74.59,  43:  11.12,  44:  0.75,
    51:  28.44,  52: -100.53, 53:   3.84,  54:  2.04,
})

# ── SALUTE ─────────────────────────────────────────────────────────────────
POSES["SALUTE"] = _p({
    41: -56.48,  42:  8.31,  43:  59.21,  44: -135.52,
})

# ── HEAD TURNS ─────────────────────────────────────────────────────────────
POSES["HEAD_LEFT"]  = _p({36:  45.0})
POSES["HEAD_RIGHT"] = _p({36: -45.0})

# ── FREEZE / CROSS ARMS ────────────────────────────────────────────────────
POSES["CROSS_ARMS"] = _p({
    41:  15.0,  42: -11.0,  43:  50.0,  44: -100.0,
    51:  15.0,  52: -27.0,  53:  76.0,  54: -100.0,
})

# ── WALK KEYFRAMES ─────────────────────────────────────────────────────────
POSES["WALK_L_UP"] = {
    11:  -1.10,  12:   4.79,  13: -26.24,  14:  50.42,  15: -11.74,
    21: -23.16,  22:   1.98,  23: -11.21,  24:   3.74,  25:  -7.52,
    31: -12.09,  32:   6.73,  33: -11.56,  34: -12.53,  35:  -1.36,
    36:  -8.36,  37:  -3.96,
    41:  50.51,  42:  66.42,  43: 104.75,  44:  -8.04,
    51: -73.54,  52: -83.21,  53: -93.67,  54:   3.21,
}
POSES["WALK_L_DN"] = {
    11:  11.47,  12: -12.35,  13:  -3.91,  14:  28.35,  15:  -5.14,
    21: -10.68,  22:  -6.29,  23:  -0.57,  24: -37.32,  25:  19.47,
    31:  -6.37,  32: -14.64,  33:   8.22,  34: -27.21,  35:  10.68,
    36: -14.22,  37:  -3.96,
    41:  49.45,  42:  68.09,  43:  20.79,  44: -31.08,
    51: -61.05,  52: -93.67,  53: -94.02,  54:   3.12,
}
POSES["WALK_R_UP"] = {
    11:   6.29,  12: -22.02,  13:   6.20,  14:  16.57,  15: -21.32,
    21: -15.08,  22: -11.65,  23:  35.56,  24: -46.99,  25:  -9.36,
    31:  -3.74,  32:  -6.46,  33:   7.43,  34: -11.82,  35:  -4.62,
    36:   4.25,  37:  -8.94,
    41:  66.51,  42:  72.84,  43:  15.08,  44:   0.66,
    51: -72.66,  52: -87.60,  53:  -7.43,  54:  19.82,
}
POSES["WALK_R_DN"] = {
    11: -12.35,  12:  -5.58,  13:  33.10,  14:  13.32,  15: -25.01,
    21: -38.64,  22:   4.79,  23:   1.45,  24: -30.55,  25:   7.87,
    31: -19.56,  32:  17.10,  33: -11.21,  34: -18.95,  35:  -5.14,
    36:  25.07,  37:  -5.13,
    41:  64.48,  42:  70.81,  43:  14.99,  44:  -6.02,
    51: -70.11,  52: -86.11,  53: -29.76,  54:  10.86,
}

WALK_CYCLE = ["WALK_L_UP", "WALK_L_DN", "WALK_R_UP", "WALK_R_DN"]


# ══════════════════════════════════════════════════════════════════════════════
#  GESTURE TIMING  --  hold durations, wave pause, priority levels
# ══════════════════════════════════════════════════════════════════════════════

# Seconds the robot holds the final pose of a non-looping gesture before
# returning to BALANCED_STAND.  During this window a lower-or-equal-priority
# gesture is blocked; only a strictly higher-priority one can interrupt.
GESTURE_HOLD_T: dict[str, float] = {
    "SALUTE":         1.50,
    "HANDS_UP":       1.20,
    "HANDS_FRONT":    1.00,
    "CROSS_ARMS":     1.20,
    "HEAD_LEFT":      0.70,
    "HEAD_RIGHT":     0.70,
    "BALANCED_STAND": 0.00,
}

# Extra pause inserted at the peak of every wave oscillation step (A→B or
# B→A).  Tune this value to control how "bouncy" or "deliberate" the wave
# feels.  180 ms is a natural human wrist-flip pause.
WAVE_PEAK_PAUSE: float = 0.18

# Interrupt priority.  Higher number wins.
# Rules enforced in set_gesture():
#   - Non-hold phase: new_priority >= current_priority  → allowed
#   - Hold phase:     new_priority >  current_priority  → allowed
GESTURE_PRIORITY: dict[str, int] = {
    "CROSS_ARMS":     10,   # freeze -- always wins immediately
    "WALK":            8,
    "SALUTE":          6,
    "HANDS_UP":        5,
    "HANDS_FRONT":     5,
    "HEAD_LEFT":       4,
    "HEAD_RIGHT":      4,
    "RIGHT_WAVE":      3,
    "LEFT_WAVE":       3,
    "BALANCED_STAND":  1,
}


# ══════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK  --  verify no pose value exceeds its motor limit
# ══════════════════════════════════════════════════════════════════════════════

def _check_poses() -> None:
    errors = []
    for pname, pose in POSES.items():
        for mid, val in pose.items():
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            if not (lo <= val <= hi):
                errors.append(
                    f"  {pname}[{mid} {MOTOR_MAP[mid][0]}] = {val}  "
                    f"OUT OF [{lo}, {hi}]"
                )
    if errors:
        print("[POSE CHECK] CLIPPING WARNINGS:")
        for e in errors:
            print(e)
    else:
        print("[POSE CHECK] All poses within limits -- OK")

_check_poses()


# ══════════════════════════════════════════════════════════════════════════════
#  DXL INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

class DxlInterface:
    MOVING_SPEED_RPM = 50

    def __init__(self, port: str, baudrate: int, ids: list[int]) -> None:
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")
        found   = set(self._io.scan(ids))
        missing = set(ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing: {sorted(missing)}")
        self._active = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")
        self._io.set_moving_speed({m: self.MOVING_SPEED_RPM for m in self._active})
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: dict[int, float]) -> None:
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e} -- SIMULATION mode")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR
# ══════════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Gesture priority (highest → lowest):
      CROSS_ARMS > WALK(T-pose) > SALUTE > HEAD_TILT >
      HANDS_FRONT > HANDS_UP > RIGHT_WAVE > LEFT_WAVE > BALANCED_STAND

    Debounce: gesture must hold for N consecutive frames before confirmed.
    """

    DEBOUNCE    = 5
    HEAD_THRESH = 0.055
    WAVE_BUF    = 7
    WAVE_HEIGHT = 0.04
    WAVE_SWING  = 0.055
    TPOSE_SPREAD= 0.44
    TPOSE_TOL   = 0.10

    def __init__(self) -> None:
        self._hist: collections.deque = collections.deque(maxlen=self.DEBOUNCE)
        self._wx = {
            "r": collections.deque(maxlen=self.WAVE_BUF),
            "l": collections.deque(maxlen=self.WAVE_BUF),
        }
        self.confirmed = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._hist.append(raw)
        if len(self._hist) == self.DEBOUNCE and len(set(self._hist)) == 1:
            self.confirmed = raw
        return self.confirmed

    def _classify(self, lm) -> str:
        PL = mp.solutions.pose.PoseLandmark
        nose  = lm[PL.NOSE]
        l_ear = lm[PL.LEFT_EAR]
        r_ear = lm[PL.RIGHT_EAR]
        l_sh  = lm[PL.LEFT_SHOULDER]
        r_sh  = lm[PL.RIGHT_SHOULDER]
        l_wr  = lm[PL.LEFT_WRIST]
        r_wr  = lm[PL.RIGHT_WRIST]

        # 1. CROSS ARMS
        if (abs(l_wr.x - r_sh.x) < 0.13 and abs(l_wr.y - r_sh.y) < 0.16 and
                abs(r_wr.x - l_sh.x) < 0.13 and abs(r_wr.y - l_sh.y) < 0.16):
            return "CROSS_ARMS"

        # 2. T-POSE → WALK
        l_lev = abs(l_wr.y - l_sh.y) < self.TPOSE_TOL
        r_lev = abs(r_wr.y - r_sh.y) < self.TPOSE_TOL
        wide  = abs(l_wr.x - r_wr.x) > self.TPOSE_SPREAD
        if l_lev and r_lev and wide:
            return "WALK"

        # 3. SALUTE (right wrist near forehead, right side)
        ear_mid_y = (l_ear.y + r_ear.y) / 2.0
        r_salute  = (r_wr.y < r_sh.y - 0.05
                     and r_wr.y < ear_mid_y + 0.08
                     and abs(r_wr.x - nose.x) < 0.20)
        if r_salute:
            return "SALUTE"

        # 4. HEAD TILT
        mid_x  = (l_sh.x + r_sh.x) / 2.0
        offset = nose.x - mid_x
        if offset >  self.HEAD_THRESH:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_THRESH:
            return "HEAD_LEFT"

        # 5. HANDS FRONT (both wrists below shoulder, near body centre)
        l_front = (l_sh.y < l_wr.y < l_sh.y + 0.30
                   and abs(l_wr.x - l_sh.x) < 0.22)
        r_front = (r_sh.y < r_wr.y < r_sh.y + 0.30
                   and abs(r_wr.x - r_sh.x) < 0.22)
        if l_front and r_front:
            return "HANDS_FRONT"

        # 6. BOTH WRISTS ABOVE SHOULDERS → HANDS UP
        l_up = l_wr.y < l_sh.y - self.WAVE_HEIGHT
        r_up = r_wr.y < r_sh.y - self.WAVE_HEIGHT
        if l_up and r_up:
            return "HANDS_UP"

        # 7 & 8. SINGLE-HAND WAVES
        if r_up:
            self._wx["r"].append(r_wr.x)
            if self._swing("r"):
                return "RIGHT_WAVE"
        if l_up:
            self._wx["l"].append(l_wr.x)
            if self._swing("l"):
                return "LEFT_WAVE"

        return "BALANCED_STAND"

    def _swing(self, s: str) -> bool:
        h = self._wx[s]
        return len(h) == self.WAVE_BUF and (max(h) - min(h)) > self.WAVE_SWING


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER  (v6.0)
# ══════════════════════════════════════════════════════════════════════════════

class MotionController:
    """
    Thread model
    ────────────
    One gesture thread at a time (_gthread).  set_gesture() stops the old
    thread and starts a new one, subject to priority gating.

    Gesture phases
    ──────────────
    LOOPING  (wave, walk)
        1. Intro frame: lerp to the raise / ready position (played once).
        2. Oscillate loop: lerp A→B, peak pause, B→A, repeat until _stop.

    NON-LOOPING  (salute, hands-up, head turns, etc.)
        1. Execute: lerp through every frame in the sequence.
        2. Hold: freeze at final pose for GESTURE_HOLD_T[gesture] seconds,
           checking _stop every 50 ms (_interruptible_sleep).
        3. Return: lerp back to BALANCED_STAND.

    Priority gating  (set_gesture)
    ───────────────────────────────
    Execute phase: new_priority >= current_priority  → interrupt allowed
    Hold    phase: new_priority >  current_priority  → interrupt allowed
    Otherwise the incoming gesture is silently dropped; the detector will
    keep feeding it and it will be accepted once the hold expires.

    Drive loop
    ──────────
    50 Hz background thread blends _tgt → _cur with a light low-pass
    filter and pushes to DxlIO every tick.
    """

    DRIVE_HZ = 50

    STEP_T: dict[str, float] = {
        "BALANCED_STAND": 0.28,
        "RIGHT_WAVE":     0.10,
        "LEFT_WAVE":      0.10,
        "HANDS_UP":       0.35,
        "HANDS_FRONT":    0.28,
        "SALUTE":         0.35,
        "HEAD_LEFT":      0.20,
        "HEAD_RIGHT":     0.20,
        "CROSS_ARMS":     0.28,
        "WALK":           0.20,
    }

    _SEQ: dict[str, list[str]] = {
        "BALANCED_STAND": ["BALANCED_STAND"],
        "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],
        "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
        "HANDS_UP":       ["HANDS_UP"],
        "HANDS_FRONT":    ["HANDS_FRONT"],
        "SALUTE":         ["SALUTE"],
        "HEAD_LEFT":      ["HEAD_LEFT"],
        "HEAD_RIGHT":     ["HEAD_RIGHT"],
        "CROSS_ARMS":     ["CROSS_ARMS"],
        "WALK":           WALK_CYCLE,
    }
    _LOOPING = {"RIGHT_WAVE", "LEFT_WAVE", "WALK"}

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl  = dxl
        self._lock = threading.Lock()
        self._cur: dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._tgt: dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._gesture  = "BALANCED_STAND"
        self._gthread: threading.Thread | None = None
        self._stop     = threading.Event()
        self._alpha    = 0.40           # low-pass strength (0=no filter, 1=instant)
        self._walk_spd = 1.0            # walk keyframes per second

        # True while _run() is in the hold phase of a non-looping gesture.
        # Used by set_gesture() to enforce the stricter priority rule.
        self._holding  = False

        threading.Thread(target=self._drive_loop, daemon=True).start()

        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(0.8)

    # ── public ───────────────────────────────────────────────────────────────

    def set_gesture(self, g: str) -> None:
        """
        Request a gesture change, subject to priority gating.

        Execute phase: new_priority >= current_priority  → allowed
        Hold    phase: new_priority >  current_priority  → allowed
        """
        with self._lock:
            if g == self._gesture:
                return

            cur_pri = GESTURE_PRIORITY.get(self._gesture, 0)
            new_pri = GESTURE_PRIORITY.get(g, 0)
            holding = self._holding

            # During hold: only strictly higher priority breaks through
            if holding and new_pri <= cur_pri:
                return

            # During execute: equal-or-higher priority can interrupt;
            # a lower-priority gesture cannot kick out a higher-priority one
            if not holding and new_pri < cur_pri:
                return

            self._gesture = g

        # Stop the running gesture thread
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=2.0)
        self._stop.clear()

        # Launch new gesture thread
        self._gthread = threading.Thread(
            target=self._run, args=(g,), daemon=True)
        self._gthread.start()

    def set_walk_speed(self, delta: float) -> None:
        self._walk_spd = float(np.clip(self._walk_spd + delta, 0.3, 3.0))
        print(f"[WALK] speed = {self._walk_spd:.1f} steps/s")

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._cur.copy()

    @property
    def walk_speed(self) -> float:
        return self._walk_spd

    def shutdown(self) -> None:
        self._stop.set()
        time.sleep(0.3)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ── gesture runner ───────────────────────────────────────────────────────

    def _run(self, gesture: str) -> None:
        frames  = self._SEQ.get(gesture, ["BALANCED_STAND"])
        looping = gesture in self._LOOPING
        step_t  = self.STEP_T.get(gesture, 0.28)
        hold_t  = GESTURE_HOLD_T.get(gesture, 0.0)
        is_wave = gesture in ("RIGHT_WAVE", "LEFT_WAVE")

        # ── LOOPING gestures (waves, walk) ───────────────────────────────────
        if looping and len(frames) > 1:
            # Play intro frame once (arm raise / ready position)
            self._lerp(POSES[frames[0]], step_t)

            # Loop remaining frames until _stop is set
            loop_frames = frames[1:]
            while not self._stop.is_set():
                for name in loop_frames:
                    if self._stop.is_set():
                        break
                    self._lerp(POSES[name], step_t)

                    # Wave: pause at each oscillation peak so the wrist
                    # "arrives" naturally before reversing direction.
                    # _interruptible_sleep returns False immediately when
                    # _stop fires, so the loop exits without any extra delay.
                    if is_wave:
                        self._interruptible_sleep(WAVE_PEAK_PAUSE)

        # ── NON-LOOPING gestures (salute, hands-up, head turns, etc.) ────────
        else:
            for name in frames:
                if self._stop.is_set():
                    break
                self._lerp(POSES[name], step_t)

            # Hold the final pose.  The _holding flag tells set_gesture() to
            # apply the stricter priority rule during this window.
            if not self._stop.is_set() and hold_t > 0.0:
                with self._lock:
                    self._holding = True
                try:
                    self._interruptible_sleep(hold_t)
                finally:
                    # Always clear the flag, even if an exception occurs
                    with self._lock:
                        self._holding = False

        # ── Return to stand ───────────────────────────────────────────────────
        # Skipped if _stop is set: the incoming gesture's _run will lerp
        # from whatever _cur is at that moment, giving a smooth hand-off.
        if gesture != "BALANCED_STAND" and not self._stop.is_set():
            self._lerp(POSES["BALANCED_STAND"], self.STEP_T["BALANCED_STAND"])

    # ── interruptible sleep ───────────────────────────────────────────────────

    def _interruptible_sleep(self, duration: float,
                             granularity: float = 0.05) -> bool:
        """
        Sleep for `duration` seconds, waking early if _stop is set.
        Returns True if the full duration elapsed, False if interrupted.
        `granularity` controls how often _stop is polled (default 50 ms).
        """
        deadline = time.monotonic() + duration
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            if self._stop.wait(timeout=min(granularity, max(remaining, 0.0))):
                return False    # interrupted early
        return True             # completed full duration

    # ── LERP ─────────────────────────────────────────────────────────────────

    def _lerp(self, target: dict[int, float], step_t: float) -> None:
        """
        Smooth cubic-eased interpolation from current position to `target`.
        Runs at DRIVE_HZ ticks; bails immediately if _stop is set.
        """
        steps = max(int(step_t * self.DRIVE_HZ), 1)
        with self._lock:
            start = self._cur.copy()
        for i in range(1, steps + 1):
            if self._stop.is_set():
                return
            t = _smooth(i / steps)
            with self._lock:
                self._tgt = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    # ── 50 Hz drive loop ─────────────────────────────────────────────────────

    def _drive_loop(self) -> None:
        """
        Background thread: blend _tgt → _cur with a low-pass filter and
        push the result to DxlIO every tick.  Runs for the lifetime of the
        process.
        """
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                tgt = self._tgt.copy()
                cur = self._cur.copy()
            blended = {
                mid: cur[mid] + self._alpha * (tgt[mid] - cur[mid])
                for mid in tgt
            }
            with self._lock:
                self._cur = blended
            if self._dxl:
                self._dxl.set_positions(blended)
            time.sleep(dt)


def _smooth(t: float) -> float:
    """Cubic ease-in-out: 3t² − 2t³  (slow start, fast middle, slow end)."""
    return t * t * (3.0 - 2.0 * t)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_COLORS = {
    "BALANCED_STAND": (40,  160,  60),
    "RIGHT_WAVE":     (60,  210, 255),
    "LEFT_WAVE":      (60,  210, 255),
    "HANDS_UP":       (60,  255, 180),
    "HANDS_FRONT":    (60,  255, 120),
    "SALUTE":         (255, 200,  60),
    "HEAD_LEFT":      (200, 140,  60),
    "HEAD_RIGHT":     (200, 140,  60),
    "CROSS_ARMS":     (80,   80, 220),
    "WALK":           (255, 160,   0),
}

_GROUPS = [
    ("LEFT  LEG",  [11, 12, 13, 14, 15]),
    ("RIGHT LEG",  [21, 22, 23, 24, 25]),
    ("TORSO",      [31, 32, 33, 34, 35]),
    ("HEAD",       [36, 37]),
    ("LEFT  ARM",  [41, 42, 43, 44]),
    ("RIGHT ARM",  [51, 52, 53, 54]),
]


def draw_hud(angles: dict[int, float], gesture: str,
             live: bool, walk_spd: float,
             holding: bool = False, frame=None) -> np.ndarray:
    W, H = 460, 635
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:] = (14, 14, 24)

    col = GESTURE_COLORS.get(gesture, (120, 120, 120))
    cv2.rectangle(canvas, (0, 0), (W, 44), col, -1)

    mode    = "DXL LIVE" if live else "SIMULATION"
    spd_str = f"  walk {walk_spd:.1f}/s" if gesture == "WALK" else ""
    hold_str = "  [HOLD]" if holding else ""
    cv2.putText(canvas, f"{mode}   {gesture}{spd_str}{hold_str}",
                (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (10, 10, 10), 2)

    y = 58
    for grp, mids in _GROUPS:
        cv2.putText(canvas, grp, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (110, 170, 255), 1)
        y += 15
        for mid in mids:
            label  = MOTOR_MAP[mid][0]
            val    = angles.get(mid, 0.0)
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            bw     = max(int(np.interp(val, [lo, hi], [0, W - 116])), 0)
            cv2.rectangle(canvas, (98, y - 10), (98 + bw, y + 2),
                           (45, 165, 90), -1)
            cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                        (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.34, (210, 210, 210), 1)
            y += 16
        y += 4

    if frame is not None:
        th = cv2.resize(frame, (140, 105))
        canvas[H - 110:H - 5, W - 145:W - 5] = th
    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    dxl  = connect_dxl()
    live = dxl is not None

    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.62,
        min_tracking_confidence=0.55,
    )

    detector   = GestureDetector()
    controller = MotionController(dxl=dxl)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("[INFO] Running -- Q quit | W walk | S stand | +/- walk speed")
    walk_on = False
    prev_t  = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            pose_g  = detector.update(results.pose_landmarks)
            gesture = "WALK" if walk_on else pose_g
            controller.set_gesture(gesture)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now
            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)
            g_col = GESTURE_COLORS.get(gesture, (200, 200, 200))
            cv2.putText(annotated, gesture,
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, g_col, 2)

            # Show hold indicator on camera feed
            with controller._lock:
                is_holding = controller._holding
            if is_holding:
                cv2.putText(annotated, "[HOLD]",
                            (8, annotated.shape[0] - 38),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (80, 80, 220), 2)

            if walk_on:
                cv2.putText(annotated, "[W] WALK ON",
                            (annotated.shape[1] - 185, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (0, 200, 255), 2)

            hud = draw_hud(
                controller.current_angles, gesture, live,
                controller.walk_speed, holding=is_holding, frame=frame,
            )
            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State", hud)

            key = cv2.waitKey(1) & 0xFF
            if   key == ord("q"):
                break
            elif key == ord("w"):
                walk_on = not walk_on
                print(f"[KEY] Walk = {walk_on}")
            elif key == ord("s"):
                walk_on = False
                controller.set_gesture("BALANCED_STAND")
            elif key in (ord("+"), ord("=")):
                controller.set_walk_speed(+0.1)
            elif key == ord("-"):
                controller.set_walk_speed(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Returning to stand...")
        walk_on = False
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        controller.set_gesture("BALANCED_STAND")
        if live:
            time.sleep(1.8)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[POSE CHECK] All poses within limits -- OK
[DXL] Opened COM4 @ 1000000 baud
[DXL] Active (25): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Torque ON
[INFO] Running -- Q quit | W walk | S stand | +/- walk speed


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        POPPY HUMANOID  --  GESTURE CONTROL  v7.0                            ║
║  Port    : COM4  (DxlIO direct)                                              ║
║  Vision  : MediaPipe Pose                                                    ║
║  Drive   : 50 Hz LERP  +  smooth-step easing                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL :  pip install opencv-python mediapipe pypot numpy                  ║
║  RUN     :  python poppy_gesture_control.py                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  CHANGES v7.0 (COMPLETE REWRITE)                                             ║
║  - FIXED: Same gesture can restart after it completes (wave restarts fine)   ║
║  - FIXED: Non-looping gestures (salute, hands-up, etc.) ALWAYS complete      ║
║    their full motion sequence before returning to stand — not interruptible  ║
║    mid-execution. Only CROSS_ARMS (freeze) can interrupt mid-execution.      ║
║  - FIXED: Wave oscillation has proper configurable peak pause                ║
║  - FIXED: Transition speeds are slower and configurable via keys             ║
║  - FIXED: Priority logic rewritten — cleaner, no deadlocks                   ║
║  - FIXED: Gesture re-trigger works (same gesture detected again → restart)   ║
║  - NEW:   [ and ] keys → decrease/increase wave swing delay                  ║
║  - NEW:   , and . keys → decrease/increase global transition speed           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  GESTURES (stand in front of webcam)                                         ║
║  ──────────────────────────────────────────────────────────────────────────  ║
║  Hands at hips / neutral          →  BALANCED STAND (reset)                 ║
║  Right wrist above right shoulder →  RIGHT WAVE (oscillates smoothly)       ║
║  Left  wrist above left  shoulder →  LEFT  WAVE (oscillates smoothly)       ║
║  Both wrists above shoulders      →  HANDS UP (completes fully then rest)   ║
║  Both arms pushed forward (chest) →  HANDS FRONT (completes fully)          ║
║  Right hand near forehead         →  SALUTE (full salute then rest)         ║
║  Tilt head left                   →  ROBOT HEAD LEFT                        ║
║  Tilt head right                  →  ROBOT HEAD RIGHT                       ║
║  Cross wrists at chest            →  FREEZE (highest priority, always wins) ║
║  T-pose (arms wide & level)       →  WALK  (4-frame keyframe cycle)         ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEYBOARD                                                                    ║
║  W  toggle walk on/off             S  force stand        Q  quit            ║
║  +  walk faster                    -  walk slower                           ║
║  [  wave delay slower (more pause) ]  wave delay faster (less pause)        ║
║  ,  transitions slower             .  transitions faster                    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import sys
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE
# ══════════════════════════════════════════════════════════════════════════════

PORT      = "COM4"
BAUDRATE  = 1000000
MOTOR_IDS = [11,12,13,14,15, 21,22,23,24,25,
             31,32,33,34,35,36,37,
             41,42,43,44, 51,52,53,54]

try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot not found -- SIMULATION mode")


# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR MAP
# ══════════════════════════════════════════════════════════════════════════════

MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",           -23,    22),
    12: ("l_hip_z",           -33,    15),
    13: ("l_hip_y",           -37,    44),
    14: ("l_knee_y",            3,    61),
    15: ("l_ankle_y",         -36,     5),
    21: ("r_hip_x",           -49,     0),
    22: ("r_hip_z",           -22,    17),
    23: ("r_hip_y",           -22,    46),
    24: ("r_knee_y",          -57,    14),
    25: ("r_ankle_y",         -20,    30),
    31: ("abs_y",             -30,    17),
    32: ("abs_x",             -25,    28),
    33: ("abs_z",             -22,    19),
    34: ("bust_y",            -43,    -1),
    35: ("bust_x",            -16,    21),
    36: ("head_z",            -48,    48),
    37: ("head_y",            -37,     7),
    41: ("l_shoulder_y",     -137,    77),
    42: ("l_shoulder_x",      -22,    85),
    43: ("l_arm_z",           -36,   115),
    44: ("l_elbow_y",        -146,    11),
    51: ("r_shoulder_y",      -84,   121),
    52: ("r_shoulder_x",     -111,    91),
    53: ("r_arm_z",          -105,   115),
    54: ("r_elbow_y",        -114,    30),
}


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY
# ══════════════════════════════════════════════════════════════════════════════

def _p(overrides: dict[int, float]) -> dict[int, float]:
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {
    "BALANCED_STAND": {
        11:  -1.98,  12:  -3.74,  13:   4.88,  14:  13.32,  15: -16.40,
        21: -22.81,  22:   6.64,  23:   0.48,  24: -10.24,  25:   6.55,
        31:   6.02,  32:  -4.70,  33:  -0.48,  34: -32.13,  35:   4.44,
        36:  -3.08,  37: -26.83,
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ── LEFT WAVE ──────────────────────────────────────────────────────────────
POSES["L_WAVE_RAISE"] = _p({41: -127.0, 42: -11.30, 43: 90.0, 44: -40.0})
POSES["L_WAVE_A"]     = _p({41: -127.0, 42: -11.30, 43: 90.0, 44: -95.52})
POSES["L_WAVE_B"]     = _p({41: -120.0, 42: -11.30, 43: 90.0, 44: -30.0})

# ── RIGHT WAVE ─────────────────────────────────────────────────────────────
POSES["R_WAVE_RAISE"] = _p({51: 72.57, 52: 27.63, 53: 76.79, 54: -40.0})
POSES["R_WAVE_A"]     = _p({51: 72.57, 52: 27.63, 53: 76.79, 54: -103.43})
POSES["R_WAVE_B"]     = _p({51: 72.57, 52: 27.63, 53: 76.79, 54: -35.0})

# ── HANDS UP ───────────────────────────────────────────────────────────────
POSES["HANDS_UP"] = _p({
    41: -112.40,  42:  35.03,  43: -13.58,  44:  0.75,
    51:  110.02,  52: -56.84,  53:   9.80,  54:  0.40,
})

# ── HANDS FRONT ────────────────────────────────────────────────────────────
POSES["HANDS_FRONT"] = _p({
    41: -32.84,  42:  74.59,  43:  11.12,  44:  0.75,
    51:  28.44,  52: -100.53, 53:   3.84,  54:  2.04,
})

# ── SALUTE ─────────────────────────────────────────────────────────────────
POSES["SALUTE"] = _p({
    41: -56.48,  42:  8.31,  43:  59.21,  44: -135.52,
})

# ── HEAD TURNS ─────────────────────────────────────────────────────────────
POSES["HEAD_LEFT"]  = _p({36:  45.0})
POSES["HEAD_RIGHT"] = _p({36: -45.0})

# ── FREEZE / CROSS ARMS ────────────────────────────────────────────────────
POSES["CROSS_ARMS"] = _p({
    41:  15.0,  42: -11.0,  43:  50.0,  44: -100.0,
    51:  15.0,  52: -27.0,  53:  76.0,  54: -100.0,
})

# ── WALK KEYFRAMES ─────────────────────────────────────────────────────────
POSES["WALK_L_UP"] = {
    11:  -1.10,  12:   4.79,  13: -26.24,  14:  50.42,  15: -11.74,
    21: -23.16,  22:   1.98,  23: -11.21,  24:   3.74,  25:  -7.52,
    31: -12.09,  32:   6.73,  33: -11.56,  34: -12.53,  35:  -1.36,
    36:  -8.36,  37:  -3.96,
    41:  50.51,  42:  66.42,  43: 104.75,  44:  -8.04,
    51: -73.54,  52: -83.21,  53: -93.67,  54:   3.21,
}
POSES["WALK_L_DN"] = {
    11:  11.47,  12: -12.35,  13:  -3.91,  14:  28.35,  15:  -5.14,
    21: -10.68,  22:  -6.29,  23:  -0.57,  24: -37.32,  25:  19.47,
    31:  -6.37,  32: -14.64,  33:   8.22,  34: -27.21,  35:  10.68,
    36: -14.22,  37:  -3.96,
    41:  49.45,  42:  68.09,  43:  20.79,  44: -31.08,
    51: -61.05,  52: -93.67,  53: -94.02,  54:   3.12,
}
POSES["WALK_R_UP"] = {
    11:   6.29,  12: -22.02,  13:   6.20,  14:  16.57,  15: -21.32,
    21: -15.08,  22: -11.65,  23:  35.56,  24: -46.99,  25:  -9.36,
    31:  -3.74,  32:  -6.46,  33:   7.43,  34: -11.82,  35:  -4.62,
    36:   4.25,  37:  -8.94,
    41:  66.51,  42:  72.84,  43:  15.08,  44:   0.66,
    51: -72.66,  52: -87.60,  53:  -7.43,  54:  19.82,
}
POSES["WALK_R_DN"] = {
    11: -12.35,  12:  -5.58,  13:  33.10,  14:  13.32,  15: -25.01,
    21: -38.64,  22:   4.79,  23:   1.45,  24: -30.55,  25:   7.87,
    31: -19.56,  32:  17.10,  33: -11.21,  34: -18.95,  35:  -5.14,
    36:  25.07,  37:  -5.13,
    41:  64.48,  42:  70.81,  43:  14.99,  44:  -6.02,
    51: -70.11,  52: -86.11,  53: -29.76,  54:  10.86,
}

WALK_CYCLE = ["WALK_L_UP", "WALK_L_DN", "WALK_R_UP", "WALK_R_DN"]


# ══════════════════════════════════════════════════════════════════════════════
#  TIMING CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# Seconds spent lerping to each pose (default, tunable via ',' and '.')
BASE_STEP_T: dict[str, float] = {
    "BALANCED_STAND": 0.55,   # slow return to stand
    "RIGHT_WAVE":     0.18,   # individual wave swing step
    "LEFT_WAVE":      0.18,
    "HANDS_UP":       0.60,
    "HANDS_FRONT":    0.55,
    "SALUTE":         0.55,
    "HEAD_LEFT":      0.35,
    "HEAD_RIGHT":     0.35,
    "CROSS_ARMS":     0.45,
    "WALK":           0.22,
}

# How long the robot holds a completed non-looping pose before returning
GESTURE_HOLD_T: dict[str, float] = {
    "SALUTE":         2.00,   # full salute hold
    "HANDS_UP":       1.50,
    "HANDS_FRONT":    1.20,
    "CROSS_ARMS":     1.80,
    "HEAD_LEFT":      0.90,
    "HEAD_RIGHT":     0.90,
    "BALANCED_STAND": 0.00,
}

# Pause at each wave peak (A peak, B peak).  Tunable with [ and ] keys.
# This makes the wave feel natural, not robotic flicking.
WAVE_PEAK_PAUSE: float = 0.35   # seconds — increase for slower, deliberate wave

# Speed multiplier for all transitions (tunable with , and . keys)
SPEED_SCALE: float = 1.0        # >1 = slower, <1 = faster


# ══════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def _check_poses() -> None:
    errors = []
    for pname, pose in POSES.items():
        for mid, val in pose.items():
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            if not (lo <= val <= hi):
                errors.append(
                    f"  {pname}[{mid} {MOTOR_MAP[mid][0]}] = {val}  "
                    f"OUT OF [{lo}, {hi}]"
                )
    if errors:
        print("[POSE CHECK] CLIPPING WARNINGS:")
        for e in errors:
            print(e)
    else:
        print("[POSE CHECK] All poses within limits -- OK")

_check_poses()


# ══════════════════════════════════════════════════════════════════════════════
#  DXL INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

class DxlInterface:
    MOVING_SPEED_RPM = 50

    def __init__(self, port: str, baudrate: int, ids: list[int]) -> None:
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")
        found   = set(self._io.scan(ids))
        missing = set(ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing: {sorted(missing)}")
        self._active = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")
        self._io.set_moving_speed({m: self.MOVING_SPEED_RPM for m in self._active})
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: dict[int, float]) -> None:
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e} -- SIMULATION mode")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR
# ══════════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Classifies the camera frame into a gesture label.

    Debounce: gesture must hold for DEBOUNCE consecutive frames before
    becoming "confirmed".  This prevents single-frame glitches.

    Priority (highest → lowest):
      CROSS_ARMS > WALK(T-pose) > SALUTE > HANDS_UP > HANDS_FRONT >
      HEAD_TILT > RIGHT_WAVE > LEFT_WAVE > BALANCED_STAND
    """

    DEBOUNCE    = 6      # frames gesture must be stable before confirmed
    HEAD_THRESH = 0.055
    WAVE_BUF    = 8
    WAVE_HEIGHT = 0.04   # wrist must be this far above shoulder (normalised)
    WAVE_SWING  = 0.050  # minimum lateral swing to count as "waving"
    TPOSE_SPREAD= 0.44
    TPOSE_TOL   = 0.10

    def __init__(self) -> None:
        self._hist: collections.deque = collections.deque(maxlen=self.DEBOUNCE)
        self._wx = {
            "r": collections.deque(maxlen=self.WAVE_BUF),
            "l": collections.deque(maxlen=self.WAVE_BUF),
        }
        self.confirmed = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._hist.append(raw)
        # Only confirm if all DEBOUNCE frames agree
        if len(self._hist) == self.DEBOUNCE and len(set(self._hist)) == 1:
            self.confirmed = raw
        return self.confirmed

    def _classify(self, lm) -> str:
        PL = mp.solutions.pose.PoseLandmark
        nose  = lm[PL.NOSE]
        l_ear = lm[PL.LEFT_EAR]
        r_ear = lm[PL.RIGHT_EAR]
        l_sh  = lm[PL.LEFT_SHOULDER]
        r_sh  = lm[PL.RIGHT_SHOULDER]
        l_wr  = lm[PL.LEFT_WRIST]
        r_wr  = lm[PL.RIGHT_WRIST]

        # 1. CROSS ARMS (freeze — always highest priority)
        if (abs(l_wr.x - r_sh.x) < 0.13 and abs(l_wr.y - r_sh.y) < 0.16 and
                abs(r_wr.x - l_sh.x) < 0.13 and abs(r_wr.y - l_sh.y) < 0.16):
            return "CROSS_ARMS"

        # 2. T-POSE → WALK
        l_lev = abs(l_wr.y - l_sh.y) < self.TPOSE_TOL
        r_lev = abs(r_wr.y - r_sh.y) < self.TPOSE_TOL
        wide  = abs(l_wr.x - r_wr.x) > self.TPOSE_SPREAD
        if l_lev and r_lev and wide:
            return "WALK"

        # 3. SALUTE (right wrist near forehead)
        ear_mid_y = (l_ear.y + r_ear.y) / 2.0
        r_salute  = (r_wr.y < r_sh.y - 0.05
                     and r_wr.y < ear_mid_y + 0.08
                     and abs(r_wr.x - nose.x) < 0.20)
        if r_salute:
            return "SALUTE"

        # 4. BOTH WRISTS ABOVE SHOULDERS → HANDS UP
        l_up = l_wr.y < l_sh.y - self.WAVE_HEIGHT
        r_up = r_wr.y < r_sh.y - self.WAVE_HEIGHT
        if l_up and r_up:
            return "HANDS_UP"

        # 5. HEAD TILT
        mid_x  = (l_sh.x + r_sh.x) / 2.0
        offset = nose.x - mid_x
        if offset >  self.HEAD_THRESH:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_THRESH:
            return "HEAD_LEFT"

        # 6. HANDS FRONT (both wrists in front of chest)
        l_front = (l_sh.y < l_wr.y < l_sh.y + 0.30
                   and abs(l_wr.x - l_sh.x) < 0.22)
        r_front = (r_sh.y < r_wr.y < r_sh.y + 0.30
                   and abs(r_wr.x - r_sh.x) < 0.22)
        if l_front and r_front:
            return "HANDS_FRONT"

        # 7 & 8. SINGLE-HAND WAVES (only one hand up)
        if r_up and not l_up:
            self._wx["r"].append(r_wr.x)
            if self._swing("r"):
                return "RIGHT_WAVE"
        if l_up and not r_up:
            self._wx["l"].append(l_wr.x)
            if self._swing("l"):
                return "LEFT_WAVE"

        return "BALANCED_STAND"

    def _swing(self, s: str) -> bool:
        h = self._wx[s]
        return len(h) == self.WAVE_BUF and (max(h) - min(h)) > self.WAVE_SWING


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER  (v7.0 — complete rewrite)
# ══════════════════════════════════════════════════════════════════════════════
#
#  KEY DESIGN DECISIONS:
#
#  1.  GESTURE THREAD MODEL
#      One gesture thread (_gthread) runs at a time.
#      set_gesture() checks whether the new gesture is allowed to interrupt,
#      then stops the old thread and starts a new one.
#
#  2.  NON-LOOPING GESTURES ARE PROTECTED
#      Once a non-looping gesture (salute, hands-up, etc.) starts executing,
#      it CANNOT be interrupted by any same-or-lower priority gesture.
#      Only CROSS_ARMS (priority 10) can break in at any point.
#      After the full sequence + hold, the gesture thread returns to STAND and
#      marks itself DONE — then set_gesture() is free again.
#
#  3.  SAME GESTURE CAN RESTART
#      When the same gesture is requested again after it finishes, it simply
#      restarts.  The "same gesture = no-op" check is done only while the
#      gesture is still running.
#
#  4.  WAVE OSCILLATION
#      Raise arm once → then loop A→B→pause→B→A→pause until gesture changes.
#      WAVE_PEAK_PAUSE is tunable with [ ] keys at runtime.
#
#  5.  WALK
#      Loops all 4 keyframes until stopped.  Walk speed tunable with +/-.
#
# ══════════════════════════════════════════════════════════════════════════════

# Interrupt priority.  Higher number = higher priority.
GESTURE_PRIORITY: dict[str, int] = {
    "CROSS_ARMS":     10,   # freeze — always interrupts immediately
    "WALK":            8,
    "SALUTE":          6,
    "HANDS_UP":        5,
    "HANDS_FRONT":     5,
    "HEAD_LEFT":       4,
    "HEAD_RIGHT":      4,
    "RIGHT_WAVE":      3,
    "LEFT_WAVE":       3,
    "BALANCED_STAND":  1,
}

# Gestures that loop indefinitely
_LOOPING_GESTURES = {"RIGHT_WAVE", "LEFT_WAVE", "WALK"}

# Gesture sequences — list of pose names to execute in order
_SEQ: dict[str, list[str]] = {
    "BALANCED_STAND": ["BALANCED_STAND"],
    "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],  # [0]=raise, [1:]=loop
    "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
    "HANDS_UP":       ["HANDS_UP"],
    "HANDS_FRONT":    ["HANDS_FRONT"],
    "SALUTE":         ["SALUTE"],
    "HEAD_LEFT":      ["HEAD_LEFT"],
    "HEAD_RIGHT":     ["HEAD_RIGHT"],
    "CROSS_ARMS":     ["CROSS_ARMS"],
    "WALK":           WALK_CYCLE,
}


class MotionController:
    DRIVE_HZ = 50

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl   = dxl
        self._lock  = threading.Lock()
        self._cur:  dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._tgt:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._gesture    = "BALANCED_STAND"
        self._gthread:   threading.Thread | None = None
        self._stop       = threading.Event()
        self._alpha      = 0.40      # low-pass blend strength
        self._walk_spd   = 1.0       # walk keyframes/second

        # State flags (protected by _lock)
        self._executing  = False     # True while inside gesture move frames
        self._holding    = False     # True during hold phase
        self._done       = False     # True after gesture fully completes + returns to stand

        threading.Thread(target=self._drive_loop, daemon=True).start()

        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.0)

    # ────────────────────────────────────────────────────────────────────────
    #  PUBLIC API
    # ────────────────────────────────────────────────────────────────────────

    def set_gesture(self, g: str) -> None:
        """
        Request a gesture change.

        Interrupt rules:
        ─────────────────
        • CROSS_ARMS (priority 10): always interrupts immediately.
        • While EXECUTING a non-looping gesture:
          → only CROSS_ARMS can interrupt.
        • While in HOLD phase:
          → only strictly higher priority can interrupt.
        • While EXECUTING a looping gesture (wave/walk):
          → any same-or-higher priority can interrupt.
        • After the gesture is DONE:
          → any gesture is accepted, including the same one (restart).
        """
        with self._lock:
            cur_pri  = GESTURE_PRIORITY.get(self._gesture, 0)
            new_pri  = GESTURE_PRIORITY.get(g, 0)
            executing = self._executing
            holding   = self._holding
            done      = self._done
            looping   = self._gesture in _LOOPING_GESTURES

        # Always accept if gesture is finished
        if done:
            pass

        # CROSS_ARMS always wins immediately
        elif g == "CROSS_ARMS":
            pass

        # Same gesture while still running: ignore (it will restart when done)
        elif g == self._gesture and not done:
            return

        # Non-looping gesture is mid-execution: block everything except CROSS_ARMS
        elif executing and not looping and not holding:
            return

        # Hold phase: only strictly higher priority breaks through
        elif holding and new_pri <= cur_pri:
            return

        # Looping gesture: allow same-or-higher priority
        elif looping and not done and new_pri < cur_pri:
            return

        # Lower priority cannot kick out higher priority in any phase
        elif not done and not looping and new_pri < cur_pri:
            return

        # ── All checks passed: stop old thread and start new one ─────────────
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=3.0)
        self._stop.clear()

        with self._lock:
            self._gesture   = g
            self._executing = False
            self._holding   = False
            self._done      = False

        self._gthread = threading.Thread(
            target=self._run, args=(g,), daemon=True)
        self._gthread.start()

    def set_walk_speed(self, delta: float) -> None:
        self._walk_spd = float(np.clip(self._walk_spd + delta, 0.3, 3.0))
        print(f"[WALK] speed = {self._walk_spd:.1f} steps/s")

    def adjust_wave_pause(self, delta: float) -> None:
        global WAVE_PEAK_PAUSE
        WAVE_PEAK_PAUSE = float(np.clip(WAVE_PEAK_PAUSE + delta, 0.05, 1.5))
        print(f"[WAVE] peak pause = {WAVE_PEAK_PAUSE:.2f}s")

    def adjust_speed_scale(self, delta: float) -> None:
        global SPEED_SCALE
        SPEED_SCALE = float(np.clip(SPEED_SCALE + delta, 0.3, 3.0))
        print(f"[SPEED] transition scale = {SPEED_SCALE:.2f}x  "
              f"(>1 = slower, <1 = faster)")

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock:
            return self._cur.copy()

    @property
    def walk_speed(self) -> float:
        return self._walk_spd

    @property
    def is_holding(self) -> bool:
        with self._lock:
            return self._holding

    @property
    def is_executing(self) -> bool:
        with self._lock:
            return self._executing

    def shutdown(self) -> None:
        self._stop.set()
        time.sleep(0.4)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ────────────────────────────────────────────────────────────────────────
    #  GESTURE RUNNER
    # ────────────────────────────────────────────────────────────────────────

    def _run(self, gesture: str) -> None:
        frames  = _SEQ.get(gesture, ["BALANCED_STAND"])
        looping = gesture in _LOOPING_GESTURES
        step_t  = BASE_STEP_T.get(gesture, 0.40)
        hold_t  = GESTURE_HOLD_T.get(gesture, 0.0)
        is_wave = gesture in ("RIGHT_WAVE", "LEFT_WAVE")

        # ── Mark executing ───────────────────────────────────────────────────
        with self._lock:
            self._executing = True
            self._holding   = False
            self._done      = False

        try:
            # ── WALK: loop all 4 frames ──────────────────────────────────────
            if gesture == "WALK":
                while not self._stop.is_set():
                    walk_step_t = 1.0 / self._walk_spd
                    for name in frames:
                        if self._stop.is_set():
                            break
                        self._lerp(POSES[name], walk_step_t)
                return  # Walk returns immediately (no hold, no return-to-stand)

            # ── WAVE: raise once, then oscillate ────────────────────────────
            elif is_wave:
                # Raise arm to ready position
                self._lerp(POSES[frames[0]], step_t * SPEED_SCALE)
                if self._stop.is_set():
                    return

                loop_frames = frames[1:]   # A and B
                while not self._stop.is_set():
                    for name in loop_frames:
                        if self._stop.is_set():
                            break
                        self._lerp(POSES[name], step_t * SPEED_SCALE)
                        # Pause at each peak — gives the wave a natural feel
                        if not self._stop.is_set():
                            self._interruptible_sleep(WAVE_PEAK_PAUSE)
                return  # Wave: no hold, no return-to-stand from here

            # ── NON-LOOPING gestures ─────────────────────────────────────────
            else:
                # Execute all frames in sequence
                # These are PROTECTED: only CROSS_ARMS can interrupt
                for name in frames:
                    if self._stop.is_set():
                        return
                    actual_step = step_t * SPEED_SCALE
                    self._lerp(POSES[name], actual_step)

                if self._stop.is_set():
                    return

                # ── HOLD phase ───────────────────────────────────────────────
                if hold_t > 0.0:
                    with self._lock:
                        self._holding   = True
                        self._executing = False
                    try:
                        self._interruptible_sleep(hold_t)
                    finally:
                        with self._lock:
                            self._holding = False
                else:
                    with self._lock:
                        self._executing = False

                if self._stop.is_set():
                    return

                # ── Return to STAND ──────────────────────────────────────────
                # Always complete return-to-stand so robot is in clean state
                stand_t = BASE_STEP_T["BALANCED_STAND"] * SPEED_SCALE
                self._lerp(POSES["BALANCED_STAND"], stand_t)

        finally:
            # Mark gesture as fully done so same gesture can restart
            with self._lock:
                self._executing = False
                self._holding   = False
                if gesture == self._gesture and not self._stop.is_set():
                    self._done = True

    # ────────────────────────────────────────────────────────────────────────
    #  INTERRUPTIBLE SLEEP
    # ────────────────────────────────────────────────────────────────────────

    def _interruptible_sleep(self, duration: float,
                              granularity: float = 0.05) -> bool:
        """
        Sleep up to `duration` seconds, checking _stop every `granularity` s.
        Returns True if full duration elapsed, False if interrupted.
        """
        deadline = time.monotonic() + duration
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            if self._stop.wait(timeout=min(granularity, max(remaining, 0.0))):
                return False
        return True

    # ────────────────────────────────────────────────────────────────────────
    #  LERP  (cubic ease-in-out)
    # ────────────────────────────────────────────────────────────────────────

    def _lerp(self, target: dict[int, float], step_t: float) -> None:
        """
        Cubic-eased interpolation from current _cur to `target`.
        Runs at DRIVE_HZ ticks; exits immediately if _stop is set.
        """
        steps = max(int(step_t * self.DRIVE_HZ), 2)
        with self._lock:
            start = self._cur.copy()
        for i in range(1, steps + 1):
            if self._stop.is_set():
                return
            t = _smooth(i / steps)
            with self._lock:
                self._tgt = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    # ────────────────────────────────────────────────────────────────────────
    #  50 Hz DRIVE LOOP
    # ────────────────────────────────────────────────────────────────────────

    def _drive_loop(self) -> None:
        """
        Background thread: blend _tgt → _cur with a low-pass filter and
        push to DxlIO every tick.  Runs for the lifetime of the process.
        """
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                tgt = self._tgt.copy()
                cur = self._cur.copy()
            blended = {
                mid: cur[mid] + self._alpha * (tgt[mid] - cur[mid])
                for mid in tgt
            }
            with self._lock:
                self._cur = blended
            if self._dxl:
                self._dxl.set_positions(blended)
            time.sleep(dt)


def _smooth(t: float) -> float:
    """Cubic ease-in-out: 3t² − 2t³."""
    return t * t * (3.0 - 2.0 * t)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_COLORS = {
    "BALANCED_STAND": (40,  160,  60),
    "RIGHT_WAVE":     (60,  210, 255),
    "LEFT_WAVE":      (60,  210, 255),
    "HANDS_UP":       (60,  255, 180),
    "HANDS_FRONT":    (60,  255, 120),
    "SALUTE":         (255, 200,  60),
    "HEAD_LEFT":      (200, 140,  60),
    "HEAD_RIGHT":     (200, 140,  60),
    "CROSS_ARMS":     (80,   80, 220),
    "WALK":           (255, 160,   0),
}

_GROUPS = [
    ("LEFT  LEG",  [11, 12, 13, 14, 15]),
    ("RIGHT LEG",  [21, 22, 23, 24, 25]),
    ("TORSO",      [31, 32, 33, 34, 35]),
    ("HEAD",       [36, 37]),
    ("LEFT  ARM",  [41, 42, 43, 44]),
    ("RIGHT ARM",  [51, 52, 53, 54]),
]


def draw_hud(angles: dict[int, float], gesture: str,
             live: bool, walk_spd: float,
             executing: bool, holding: bool,
             wave_pause: float, speed_scale: float,
             frame=None) -> np.ndarray:
    W, H = 480, 660
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:] = (14, 14, 24)

    col = GESTURE_COLORS.get(gesture, (120, 120, 120))
    cv2.rectangle(canvas, (0, 0), (W, 44), col, -1)

    mode    = "DXL LIVE" if live else "SIMULATION"
    spd_str = f"  walk {walk_spd:.1f}/s" if gesture == "WALK" else ""
    state_str = "  [HOLD]" if holding else ("  [EXEC]" if executing else "  [DONE]")
    cv2.putText(canvas, f"{mode}   {gesture}{spd_str}{state_str}",
                (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (10, 10, 10), 2)

    y = 58
    for grp, mids in _GROUPS:
        cv2.putText(canvas, grp, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (110, 170, 255), 1)
        y += 15
        for mid in mids:
            label  = MOTOR_MAP[mid][0]
            val    = angles.get(mid, 0.0)
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            bw     = max(int(np.interp(val, [lo, hi], [0, W - 120])), 0)
            cv2.rectangle(canvas, (100, y - 10), (100 + bw, y + 2),
                          (45, 165, 90), -1)
            cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                        (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.34, (210, 210, 210), 1)
            y += 16
        y += 4

    # Runtime settings panel
    cv2.rectangle(canvas, (0, H - 62), (W, H), (24, 24, 40), -1)
    cv2.putText(canvas, f"Wave pause: {wave_pause:.2f}s  [  ]",
                (8, H - 44), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (160, 220, 255), 1)
    cv2.putText(canvas, f"Speed scale: {speed_scale:.2f}x  ,  .",
                (8, H - 26), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (160, 255, 200), 1)
    cv2.putText(canvas, f"Walk: {walk_spd:.1f}/s  +  -   |  W walk  S stand  Q quit",
                (8, H - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.32, (200, 200, 200), 1)

    if frame is not None:
        th = cv2.resize(frame, (140, 105))
        canvas[H - 170:H - 65, W - 145:W - 5] = th

    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    dxl  = connect_dxl()
    live = dxl is not None

    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.62,
        min_tracking_confidence=0.55,
    )

    detector   = GestureDetector()
    controller = MotionController(dxl=dxl)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("[INFO] Running")
    print("  Q  quit       W  toggle walk     S  force stand")
    print("  +  walk fast  -  walk slow")
    print("  [  wave slower (more pause)   ]  wave faster (less pause)")
    print("  ,  transitions slower         .  transitions faster")

    walk_on = False
    prev_t  = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            pose_g  = detector.update(results.pose_landmarks)
            gesture = "WALK" if walk_on else pose_g
            controller.set_gesture(gesture)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now

            # Overlay info on camera feed
            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)
            g_col = GESTURE_COLORS.get(gesture, (200, 200, 200))
            cv2.putText(annotated, gesture,
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, g_col, 2)

            is_exec    = controller.is_executing
            is_holding = controller.is_holding

            if is_holding:
                cv2.putText(annotated, "[HOLD]",
                            (8, annotated.shape[0] - 42),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (80, 80, 220), 2)
            elif is_exec:
                cv2.putText(annotated, "[EXEC]",
                            (8, annotated.shape[0] - 42),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (0, 200, 100), 2)

            if walk_on:
                cv2.putText(annotated, "[W] WALK ON",
                            (annotated.shape[1] - 185, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                            (0, 200, 255), 2)

            hud = draw_hud(
                controller.current_angles, gesture, live,
                controller.walk_speed,
                executing=is_exec, holding=is_holding,
                wave_pause=WAVE_PEAK_PAUSE,
                speed_scale=SPEED_SCALE,
                frame=frame,
            )
            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State",   hud)

            key = cv2.waitKey(1) & 0xFF

            if   key == ord("q"):
                break
            elif key == ord("w"):
                walk_on = not walk_on
                if not walk_on:
                    controller.set_gesture("BALANCED_STAND")
                print(f"[KEY] Walk = {walk_on}")
            elif key == ord("s"):
                walk_on = False
                controller._stop.set()   # force-interrupt current gesture
                if controller._gthread and controller._gthread.is_alive():
                    controller._gthread.join(timeout=1.0)
                controller._stop.clear()
                with controller._lock:
                    controller._gesture   = "BALANCED_STAND"
                    controller._executing = False
                    controller._holding   = False
                    controller._done      = True
                controller.set_gesture("BALANCED_STAND")
                print("[KEY] Forced stand")
            elif key in (ord("+"), ord("=")):
                controller.set_walk_speed(+0.2)
            elif key == ord("-"):
                controller.set_walk_speed(-0.2)
            elif key == ord("["):
                controller.adjust_wave_pause(+0.05)
            elif key == ord("]"):
                controller.adjust_wave_pause(-0.05)
            elif key == ord(","):
                controller.adjust_speed_scale(+0.1)
            elif key == ord("."):
                controller.adjust_speed_scale(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Returning to stand...")
        walk_on = False
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        # Force stand regardless of state
        controller._stop.set()
        time.sleep(0.3)
        controller._stop.clear()
        with controller._lock:
            controller._gesture   = "BALANCED_STAND"
            controller._executing = False
            controller._holding   = False
            controller._done      = True
        if live:
            dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.5)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[POSE CHECK] All poses within limits -- OK
[DXL] Opened COM4 @ 1000000 baud
[DXL] WARNING -- missing: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Active (0): []
[DXL] Torque ON
[INFO] Running
  Q  quit       W  toggle walk     S  force stand
  +  walk fast  -  walk slow
  [  wave slower (more pause)   ]  wave faster (less pause)
  ,  transitions slower         .  transitions faster


In [ ]:
 """
╔══════════════════════════════════════════════════════════════════════════════╗
║        POPPY HUMANOID  --  GESTURE CONTROL  v8.0                            ║
║  Port    : COM4  (DxlIO direct)                                              ║
║  Vision  : MediaPipe Pose                                                    ║
║  Drive   : 50 Hz LERP  +  smooth-step easing                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL :  pip install opencv-python mediapipe pypot numpy                  ║
║  RUN     :  python poppy_gesture_control.py                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  8 GESTURES  (stand ~1.5 m from webcam, full body visible)                  ║
║  ──────────────────────────────────────────────────────────────────────────  ║
║                                                                              ║
║  BALANCED STAND   Hands relaxed at sides / hips                             ║
║                                                                              ║
║  RIGHT WAVE       ONLY right wrist raised high (left stays down)            ║
║                   + wrist far from face + lateral swing detected             ║
║                                                                              ║
║  LEFT WAVE        ONLY left wrist raised high (right stays down)            ║
║                   + wrist far from face + lateral swing detected             ║
║                                                                              ║
║  HANDS UP         BOTH wrists clearly above BOTH shoulders                  ║
║                                                                              ║
║  HANDS FRONT      Both wrists chest-height, arms pushed forward             ║
║                                                                              ║
║  HEAD LEFT/RIGHT  Tilt/turn head — nose drifts past shoulder midpoint       ║
║                   (only fires when arms are NOT raised)                     ║
║                                                                              ║
║  FREEZE           Cross wrists at chest — both wrists near opposite shoulder║
║                                                                              ║
║  WALK             T-pose: both arms level with shoulders, spread wide       ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  HOW MIXING IS PREVENTED                                                    ║
║  RIGHT WAVE needs wrist far from face (x-dist > 0.20). Waving near your    ║
║  face no longer triggers it. Also requires ONLY one hand up.               ║
║  HANDS UP needs BOTH hands up — one hand prevents it.                      ║
║  HEAD TILT only fires when no wrists are raised above shoulder.             ║
║  HANDS FRONT only fires when no wrists are raised above shoulder.           ║
║  T-POSE needs both wrists LEVEL (not above) AND wide — vs HANDS UP which   ║
║  needs wrists ABOVE shoulders.                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEYBOARD                                                                    ║
║  Q  quit          W  toggle walk       S  force stand                       ║
║  +  walk faster   -  walk slower                                            ║
║  [  wave pause +  ]  wave pause -   (default 0.35 s)                       ║
║  ,  transitions slower   .  transitions faster                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import sys
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE
# ══════════════════════════════════════════════════════════════════════════════

PORT      = "COM4"
BAUDRATE  = 1000000
MOTOR_IDS = [11,12,13,14,15, 21,22,23,24,25,
             31,32,33,34,35,36,37,
             41,42,43,44, 51,52,53,54]

try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot not found -- SIMULATION mode")


# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR MAP
# ══════════════════════════════════════════════════════════════════════════════

MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",           -23,    22),
    12: ("l_hip_z",           -33,    15),
    13: ("l_hip_y",           -37,    44),
    14: ("l_knee_y",            3,    61),
    15: ("l_ankle_y",         -36,     5),
    21: ("r_hip_x",           -49,     0),
    22: ("r_hip_z",           -22,    17),
    23: ("r_hip_y",           -22,    46),
    24: ("r_knee_y",          -57,    14),
    25: ("r_ankle_y",         -20,    30),
    31: ("abs_y",             -30,    17),
    32: ("abs_x",             -25,    28),
    33: ("abs_z",             -22,    19),
    34: ("bust_y",            -43,    -1),
    35: ("bust_x",            -16,    21),
    36: ("head_z",            -48,    48),
    37: ("head_y",            -37,     7),
    41: ("l_shoulder_y",     -137,    77),
    42: ("l_shoulder_x",      -22,    85),
    43: ("l_arm_z",           -36,   115),
    44: ("l_elbow_y",        -146,    11),
    51: ("r_shoulder_y",      -84,   121),
    52: ("r_shoulder_x",     -111,    91),
    53: ("r_arm_z",          -105,   115),
    54: ("r_elbow_y",        -114,    30),
}


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY
# ══════════════════════════════════════════════════════════════════════════════

def _p(overrides: dict[int, float]) -> dict[int, float]:
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {
    "BALANCED_STAND": {
        11:  -1.98,  12:  -3.74,  13:   4.88,  14:  13.32,  15: -16.40,
        21: -22.81,  22:   6.64,  23:   0.48,  24: -10.24,  25:   6.55,
        31:   6.02,  32:  -4.70,  33:  -0.48,  34: -32.13,  35:   4.44,
        36:  -3.08,  37: -26.83,
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ── LEFT WAVE ──────────────────────────────────────────────────────────────
POSES["L_WAVE_RAISE"] = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -40.0})
POSES["L_WAVE_A"]     = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -95.52})
POSES["L_WAVE_B"]     = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -30.0})

# ── RIGHT WAVE ─────────────────────────────────────────────────────────────
POSES["R_WAVE_RAISE"] = _p({51: 111, 52: 11.47, 53: -88.75, 54: 45})
POSES["R_WAVE_A"]     = _p({51: 111, 52: 11.47, 53: -88.75, 54: 100})
POSES["R_WAVE_B"]     = _p({51: 111, 52: 11.47, 53: -88.75, 54: 10})

# ── HANDS UP ───────────────────────────────────────────────────────────────
POSES["HANDS_UP"] = _p({
    41: -112.40,  42:  35.03,  43: -13.58,  44:  0.75,
    51:  110.02,  52: -56.84,  53:   9.80,  54:  0.40,
})

# ── HANDS FRONT ────────────────────────────────────────────────────────────
POSES["HANDS_FRONT"] = _p({
    41: -32.84,  42:  74.59,  43:  11.12,  44:  0.75,
    51:  28.44,  52: -100.53, 53:   3.84,  54:  2.04,
})

# ── HEAD TURNS ─────────────────────────────────────────────────────────────
POSES["HEAD_LEFT"]  = _p({36:  45.0})
POSES["HEAD_RIGHT"] = _p({36: -45.0})

# ── FREEZE / CROSS ARMS ────────────────────────────────────────────────────
POSES["CROSS_ARMS"] = _p({
    41:  15.0,  42: -11.0,  43:  50.0,  44: -100.0,
    51:  15.0,  52: -27.0,  53:  76.0,  54: -100.0,
})

# ── WALK KEYFRAMES ─────────────────────────────────────────────────────────
POSES["WALK_L_UP"] = {
    11:  -1.10,  12:   4.79,  13: -26.24,  14:  50.42,  15: -11.74,
    21: -23.16,  22:   1.98,  23: -11.21,  24:   3.74,  25:  -7.52,
    31: -12.09,  32:   6.73,  33: -11.56,  34: -12.53,  35:  -1.36,
    36:  -8.36,  37:  -3.96,
    41:  50.51,  42:  66.42,  43: 104.75,  44:  -8.04,
    51: -73.54,  52: -83.21,  53: -93.67,  54:   3.21,
}
POSES["WALK_L_DN"] = {
    11:  11.47,  12: -12.35,  13:  -3.91,  14:  28.35,  15:  -5.14,
    21: -10.68,  22:  -6.29,  23:  -0.57,  24: -37.32,  25:  19.47,
    31:  -6.37,  32: -14.64,  33:   8.22,  34: -27.21,  35:  10.68,
    36: -14.22,  37:  -3.96,
    41:  49.45,  42:  68.09,  43:  20.79,  44: -31.08,
    51: -61.05,  52: -93.67,  53: -94.02,  54:   3.12,
}
POSES["WALK_R_UP"] = {
    11:   6.29,  12: -22.02,  13:   6.20,  14:  16.57,  15: -21.32,
    21: -15.08,  22: -11.65,  23:  35.56,  24: -46.99,  25:  -9.36,
    31:  -3.74,  32:  -6.46,  33:   7.43,  34: -11.82,  35:  -4.62,
    36:   4.25,  37:  -8.94,
    41:  66.51,  42:  72.84,  43:  15.08,  44:   0.66,
    51: -72.66,  52: -87.60,  53:  -7.43,  54:  19.82,
}
POSES["WALK_R_DN"] = {
    11: -12.35,  12:  -5.58,  13:  33.10,  14:  13.32,  15: -25.01,
    21: -38.64,  22:   4.79,  23:   1.45,  24: -30.55,  25:   7.87,
    31: -19.56,  32:  17.10,  33: -11.21,  34: -18.95,  35:  -5.14,
    36:  25.07,  37:  -5.13,
    41:  64.48,  42:  70.81,  43:  14.99,  44:  -6.02,
    51: -70.11,  52: -86.11,  53: -29.76,  54:  10.86,
}

WALK_CYCLE = ["WALK_L_UP", "WALK_L_DN", "WALK_R_UP", "WALK_R_DN"]


# ══════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def _check_poses() -> None:
    errors = []
    for pname, pose in POSES.items():
        for mid, val in pose.items():
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            if not (lo <= val <= hi):
                errors.append(
                    f"  {pname}[{mid} {MOTOR_MAP[mid][0]}] = {val}  "
                    f"OUT OF [{lo}, {hi}]"
                )
    if errors:
        print("[POSE CHECK] CLIPPING WARNINGS:")
        for e in errors: print(e)
    else:
        print("[POSE CHECK] All poses within limits -- OK")

_check_poses()


# ══════════════════════════════════════════════════════════════════════════════
#  TIMING  (all tunable at runtime)
# ══════════════════════════════════════════════════════════════════════════════

BASE_STEP_T: dict[str, float] = {
    "BALANCED_STAND": 0.60,
    "RIGHT_WAVE":     0.20,   # per oscillation step
    "LEFT_WAVE":      0.20,
    "HANDS_UP":       0.65,
    "HANDS_FRONT":    0.55,
    "HEAD_LEFT":      0.38,
    "HEAD_RIGHT":     0.38,
    "CROSS_ARMS":     0.45,
    "WALK":           0.22,
}

GESTURE_HOLD_T: dict[str, float] = {
    "HANDS_UP":       1.50,
    "HANDS_FRONT":    1.20,
    "CROSS_ARMS":     1.80,
    "HEAD_LEFT":      0.90,
    "HEAD_RIGHT":     0.90,
    "BALANCED_STAND": 0.00,
}

WAVE_PEAK_PAUSE: float = 0.35   # pause at each wave peak  → [ ] keys
SPEED_SCALE:     float = 1.0    # multiplies all step times → , . keys


# ══════════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR  v8.0
# ══════════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Clean, non-overlapping detection zones for all 8 gestures.

    Priority order (checked top-to-bottom, first match wins):
      1. CROSS_ARMS  — wrists crossed at chest
      2. WALK        — T-pose (both arms level AND wide)
      3. HANDS_UP    — BOTH wrists above BOTH shoulders
      4. RIGHT_WAVE  — ONLY right wrist up, away from face, swinging
      5. LEFT_WAVE   — ONLY left  wrist up, away from face, swinging
      6. HEAD_LEFT / HEAD_RIGHT — nose offset (only when arms down)
      7. HANDS_FRONT — both wrists chest-height (only when arms down)
      8. BALANCED_STAND — everything else
    """

    DEBOUNCE       = 7      # frames for stable confirmation
    WAVE_HEIGHT    = 0.06   # wrist above shoulder threshold (normalised)
    WAVE_BUF       = 9      # x-history frames for swing detection
    WAVE_SWING     = 0.055  # min wrist x-range to count as swing
    WAVE_FACE_DIST = 0.20   # wrist must be this far from nose.x
    TPOSE_SPREAD   = 0.44   # min wrist-to-wrist x-spread for T-pose
    TPOSE_TOL      = 0.09   # T-pose: wrist y vs shoulder y tolerance
    HEAD_THRESH    = 0.060  # nose x-offset from shoulder midpoint
    CROSS_X_TOL    = 0.14
    CROSS_Y_TOL    = 0.17

    def __init__(self) -> None:
        self._hist: collections.deque = collections.deque(maxlen=self.DEBOUNCE)
        self._wx = {
            "r": collections.deque(maxlen=self.WAVE_BUF),
            "l": collections.deque(maxlen=self.WAVE_BUF),
        }
        self.confirmed  = "BALANCED_STAND"
        self._raw_label = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._raw_label = raw
        self._hist.append(raw)
        if len(self._hist) == self.DEBOUNCE and len(set(self._hist)) == 1:
            self.confirmed = raw
        return self.confirmed

    @property
    def raw(self) -> str:
        return self._raw_label

    def _classify(self, lm) -> str:
        PL    = mp.solutions.pose.PoseLandmark
        nose  = lm[PL.NOSE]
        l_sh  = lm[PL.LEFT_SHOULDER]
        r_sh  = lm[PL.RIGHT_SHOULDER]
        l_hip = lm[PL.LEFT_HIP]
        r_hip = lm[PL.RIGHT_HIP]
        l_wr  = lm[PL.LEFT_WRIST]
        r_wr  = lm[PL.RIGHT_WRIST]

        sh_mid_x  = (l_sh.x + r_sh.x) / 2.0
        hip_mid_y = (l_hip.y + r_hip.y) / 2.0

        # Wrist-above-shoulder flags
        r_up = r_wr.y < r_sh.y - self.WAVE_HEIGHT
        l_up = l_wr.y < l_sh.y - self.WAVE_HEIGHT

        # ── 1. CROSS ARMS ────────────────────────────────────────────────────
        l_to_rsh = (abs(l_wr.x - r_sh.x) < self.CROSS_X_TOL and
                    abs(l_wr.y - r_sh.y) < self.CROSS_Y_TOL)
        r_to_lsh = (abs(r_wr.x - l_sh.x) < self.CROSS_X_TOL and
                    abs(r_wr.y - l_sh.y) < self.CROSS_Y_TOL)
        if l_to_rsh and r_to_lsh:
            return "CROSS_ARMS"

        # ── 2. WALK — T-pose ─────────────────────────────────────────────────
        # Both wrists level (NOT above) with shoulders AND wide spread.
        # Mutually exclusive with HANDS_UP because HANDS_UP requires wrists
        # to be *above* shoulder, while T-pose requires them to be *level*.
        l_level = abs(l_wr.y - l_sh.y) < self.TPOSE_TOL
        r_level = abs(r_wr.y - r_sh.y) < self.TPOSE_TOL
        wide    = abs(l_wr.x - r_wr.x) > self.TPOSE_SPREAD
        if l_level and r_level and wide:
            return "WALK"

        # ── 3. HANDS UP — both wrists above shoulders ────────────────────────
        if r_up and l_up:
            return "HANDS_UP"

        # ── 4. RIGHT WAVE — only right wrist up, away from face, swinging ───
        if r_up and not l_up:
            self._wx["r"].append(r_wr.x)
            face_clear = abs(r_wr.x - nose.x) > self.WAVE_FACE_DIST
            if face_clear and self._swing("r"):
                return "RIGHT_WAVE"
            # Right hand is up but not waving yet — fall through to STAND
            # (avoids misfire while user is just raising their hand)
            return "BALANCED_STAND"

        # ── 5. LEFT WAVE — only left wrist up, away from face, swinging ─────
        if l_up and not r_up:
            self._wx["l"].append(l_wr.x)
            face_clear = abs(l_wr.x - nose.x) > self.WAVE_FACE_DIST
            if face_clear and self._swing("l"):
                return "LEFT_WAVE"
            return "BALANCED_STAND"

        # ── 6 & 7: Arms-down-only gestures ───────────────────────────────────
        # HEAD TILT and HANDS FRONT are only checked when NEITHER wrist
        # is raised. This prevents head movements during wave from misfiring.

        # ── 6. HEAD TILT ─────────────────────────────────────────────────────
        offset = nose.x - sh_mid_x
        if offset >  self.HEAD_THRESH:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_THRESH:
            return "HEAD_LEFT"

        # ── 7. HANDS FRONT ───────────────────────────────────────────────────
        l_front = (l_sh.y < l_wr.y < hip_mid_y and
                   abs(l_wr.x - l_sh.x) < 0.22)
        r_front = (r_sh.y < r_wr.y < hip_mid_y and
                   abs(r_wr.x - r_sh.x) < 0.22)
        if l_front and r_front:
            return "HANDS_FRONT"

        return "BALANCED_STAND"

    def _swing(self, s: str) -> bool:
        h = self._wx[s]
        return len(h) == self.WAVE_BUF and (max(h) - min(h)) > self.WAVE_SWING


# ══════════════════════════════════════════════════════════════════════════════
#  DXL INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

class DxlInterface:
    MOVING_SPEED_RPM = 50

    def __init__(self, port: str, baudrate: int, ids: list[int]) -> None:
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")
        found   = set(self._io.scan(ids))
        missing = set(ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing: {sorted(missing)}")
        self._active = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")
        self._io.set_moving_speed({m: self.MOVING_SPEED_RPM for m in self._active})
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: dict[int, float]) -> None:
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e} -- SIMULATION mode")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER  v8.0
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_PRIORITY: dict[str, int] = {
    "CROSS_ARMS":     10,
    "WALK":            8,
    "HANDS_UP":        5,
    "HANDS_FRONT":     5,
    "HEAD_LEFT":       4,
    "HEAD_RIGHT":      4,
    "RIGHT_WAVE":      3,
    "LEFT_WAVE":       3,
    "BALANCED_STAND":  1,
}

_LOOPING = {"RIGHT_WAVE", "LEFT_WAVE", "WALK"}

_SEQ: dict[str, list[str]] = {
    "BALANCED_STAND": ["BALANCED_STAND"],
    "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],
    "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
    "HANDS_UP":       ["HANDS_UP"],
    "HANDS_FRONT":    ["HANDS_FRONT"],
    "HEAD_LEFT":      ["HEAD_LEFT"],
    "HEAD_RIGHT":     ["HEAD_RIGHT"],
    "CROSS_ARMS":     ["CROSS_ARMS"],
    "WALK":           WALK_CYCLE,
}


class MotionController:
    DRIVE_HZ = 50

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl   = dxl
        self._lock  = threading.Lock()
        self._cur:  dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._tgt:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._gesture  = "BALANCED_STAND"
        self._gthread: threading.Thread | None = None
        self._stop     = threading.Event()
        self._alpha    = 0.40
        self._walk_spd = 1.0

        # State flags
        self._executing = False
        self._holding   = False
        self._done      = True    # starts done so first gesture fires

        threading.Thread(target=self._drive_loop, daemon=True).start()

        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.0)

    # ── public ───────────────────────────────────────────────────────────────

    def set_gesture(self, g: str) -> None:
        """
        Request a gesture change, subject to interrupt rules:
          - CROSS_ARMS:         always wins immediately
          - Gesture DONE:       anything accepted (enables restart)
          - Same gesture running: ignored (will restart once done)
          - Non-looping EXECUTING: blocked (must complete fully)
          - Non-looping HOLDING:  only strictly higher priority breaks through
          - LOOPING running:    same-or-higher priority can interrupt
          - Lower priority:     always blocked
        """
        with self._lock:
            cur_pri   = GESTURE_PRIORITY.get(self._gesture, 0)
            new_pri   = GESTURE_PRIORITY.get(g, 0)
            executing = self._executing
            holding   = self._holding
            done      = self._done
            looping   = self._gesture in _LOOPING

        if g == "CROSS_ARMS":           pass
        elif done:                      pass
        elif g == self._gesture:        return   # same gesture still running
        elif executing and not looping: return   # non-looping mid-exec: blocked
        elif holding and new_pri <= cur_pri: return
        elif looping and new_pri < cur_pri:  return
        elif not done and not looping and new_pri < cur_pri: return

        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=3.0)
        self._stop.clear()

        with self._lock:
            self._gesture   = g
            self._executing = False
            self._holding   = False
            self._done      = False

        self._gthread = threading.Thread(
            target=self._run, args=(g,), daemon=True)
        self._gthread.start()

    def set_walk_speed(self, delta: float) -> None:
        self._walk_spd = float(np.clip(self._walk_spd + delta, 0.3, 3.0))
        print(f"[WALK] speed = {self._walk_spd:.1f} steps/s")

    def adjust_wave_pause(self, delta: float) -> None:
        global WAVE_PEAK_PAUSE
        WAVE_PEAK_PAUSE = float(np.clip(WAVE_PEAK_PAUSE + delta, 0.05, 1.5))
        print(f"[WAVE] peak pause = {WAVE_PEAK_PAUSE:.2f}s")

    def adjust_speed_scale(self, delta: float) -> None:
        global SPEED_SCALE
        SPEED_SCALE = float(np.clip(SPEED_SCALE + delta, 0.3, 3.0))
        print(f"[SPEED] scale = {SPEED_SCALE:.2f}x  (1.0 = default)")

    def force_stand(self) -> None:
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=2.0)
        self._stop.clear()
        with self._lock:
            self._gesture   = "BALANCED_STAND"
            self._executing = False
            self._holding   = False
            self._done      = True
            self._tgt       = POSES["BALANCED_STAND"].copy()

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock: return self._cur.copy()

    @property
    def walk_speed(self) -> float:
        return self._walk_spd

    @property
    def is_holding(self) -> bool:
        with self._lock: return self._holding

    @property
    def is_executing(self) -> bool:
        with self._lock: return self._executing

    def shutdown(self) -> None:
        self._stop.set()
        time.sleep(0.4)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ── gesture runner ───────────────────────────────────────────────────────

    def _run(self, gesture: str) -> None:
        frames  = _SEQ.get(gesture, ["BALANCED_STAND"])
        looping = gesture in _LOOPING
        step_t  = BASE_STEP_T.get(gesture, 0.40)
        hold_t  = GESTURE_HOLD_T.get(gesture, 0.0)
        is_wave = gesture in ("RIGHT_WAVE", "LEFT_WAVE")

        with self._lock:
            self._executing = True
            self._holding   = False
            self._done      = False

        try:
            # ── WALK ─────────────────────────────────────────────────────────
            if gesture == "WALK":
                while not self._stop.is_set():
                    wt = 1.0 / self._walk_spd
                    for name in frames:
                        if self._stop.is_set(): break
                        self._lerp(POSES[name], wt)
                return

            # ── WAVE ──────────────────────────────────────────────────────────
            elif is_wave:
                # Raise arm once, then oscillate A ↔ B with peak pauses
                self._lerp(POSES[frames[0]], step_t * SPEED_SCALE)
                if self._stop.is_set(): return
                loop_frames = frames[1:]
                while not self._stop.is_set():
                    for name in loop_frames:
                        if self._stop.is_set(): break
                        self._lerp(POSES[name], step_t * SPEED_SCALE)
                        if not self._stop.is_set():
                            self._interruptible_sleep(WAVE_PEAK_PAUSE)
                return

            # ── NON-LOOPING ───────────────────────────────────────────────────
            else:
                # Execute all frames — PROTECTED (only CROSS_ARMS interrupts)
                for name in frames:
                    if self._stop.is_set(): return
                    self._lerp(POSES[name], step_t * SPEED_SCALE)

                if self._stop.is_set(): return

                # Hold phase
                if hold_t > 0.0:
                    with self._lock:
                        self._holding   = True
                        self._executing = False
                    try:
                        self._interruptible_sleep(hold_t)
                    finally:
                        with self._lock:
                            self._holding = False
                else:
                    with self._lock:
                        self._executing = False

                if self._stop.is_set(): return

                # Return to balanced stand
                self._lerp(POSES["BALANCED_STAND"],
                            BASE_STEP_T["BALANCED_STAND"] * SPEED_SCALE)

        finally:
            with self._lock:
                self._executing = False
                self._holding   = False
                if gesture == self._gesture and not self._stop.is_set():
                    self._done = True

    # ── helpers ───────────────────────────────────────────────────────────────

    def _interruptible_sleep(self, duration: float,
                              granularity: float = 0.05) -> bool:
        deadline = time.monotonic() + duration
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            if self._stop.wait(timeout=min(granularity, max(remaining, 0.0))):
                return False
        return True

    def _lerp(self, target: dict[int, float], step_t: float) -> None:
        steps = max(int(step_t * self.DRIVE_HZ), 2)
        with self._lock:
            start = self._cur.copy()
        for i in range(1, steps + 1):
            if self._stop.is_set(): return
            t = _smooth(i / steps)
            with self._lock:
                self._tgt = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    def _drive_loop(self) -> None:
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                tgt = self._tgt.copy()
                cur = self._cur.copy()
            blended = {
                mid: cur[mid] + self._alpha * (tgt[mid] - cur[mid])
                for mid in tgt
            }
            with self._lock:
                self._cur = blended
            if self._dxl:
                self._dxl.set_positions(blended)
            time.sleep(dt)


def _smooth(t: float) -> float:
    return t * t * (3.0 - 2.0 * t)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_COLORS = {
    "BALANCED_STAND": (40,  160,  60),
    "RIGHT_WAVE":     (60,  210, 255),
    "LEFT_WAVE":      (60,  210, 255),
    "HANDS_UP":       (60,  255, 180),
    "HANDS_FRONT":    (60,  255, 120),
    "HEAD_LEFT":      (200, 140,  60),
    "HEAD_RIGHT":     (200, 140,  60),
    "CROSS_ARMS":     (80,   80, 220),
    "WALK":           (255, 160,   0),
}

_GROUPS = [
    ("LEFT  LEG",  [11, 12, 13, 14, 15]),
    ("RIGHT LEG",  [21, 22, 23, 24, 25]),
    ("TORSO",      [31, 32, 33, 34, 35]),
    ("HEAD",       [36, 37]),
    ("LEFT  ARM",  [41, 42, 43, 44]),
    ("RIGHT ARM",  [51, 52, 53, 54]),
]

_GUIDE = [
    ("BALANCED STAND", "hands at sides"),
    ("RIGHT WAVE",     "only R wrist up + swing away from face"),
    ("LEFT  WAVE",     "only L wrist up + swing away from face"),
    ("HANDS UP",       "BOTH wrists above both shoulders"),
    ("HANDS FRONT",    "both wrists chest-height, arms forward"),
    ("HEAD LEFT/R",    "turn head — nose drifts sideways"),
    ("FREEZE",         "cross wrists at chest"),
    ("WALK",           "T-pose: arms level + wide"),
]


def draw_hud(angles, gesture, raw_gesture, live, walk_spd,
             executing, holding, wave_pause, speed_scale, frame=None):
    W, H = 500, 710
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:] = (14, 14, 24)

    # Title bar
    col = GESTURE_COLORS.get(gesture, (100, 100, 100))
    cv2.rectangle(canvas, (0, 0), (W, 46), col, -1)
    mode    = "DXL LIVE" if live else "SIMULATION"
    spd_str = f"  {walk_spd:.1f}/s" if gesture == "WALK" else ""
    state   = "  [HOLD]" if holding else ("  [EXEC]" if executing else "  [DONE]")
    cv2.putText(canvas, f"{mode}  {gesture}{spd_str}{state}",
                (8, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (10, 10, 10), 2)

    # Raw detection (unconfirmed)
    cv2.putText(canvas, f"raw detect: {raw_gesture}",
                (8, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (150, 150, 150), 1)

    # Motor bars
    y = 76
    for grp, mids in _GROUPS:
        cv2.putText(canvas, grp, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.37, (110, 170, 255), 1)
        y += 15
        for mid in mids:
            label  = MOTOR_MAP[mid][0]
            val    = angles.get(mid, 0.0)
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            bw     = max(int(np.interp(val, [lo, hi], [0, W - 124])), 0)
            cv2.rectangle(canvas, (104, y - 10), (104 + bw, y + 2),
                          (45, 165, 90), -1)
            cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                        (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.34, (210, 210, 210), 1)
            y += 16
        y += 4

    # Gesture guide
    y += 4
    cv2.putText(canvas, "GESTURE GUIDE", (8, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.36, (80, 180, 255), 1)
    y += 14
    for name, desc in _GUIDE:
        key = name.strip().replace(" ", "_").upper()
        # Map guide name → gesture key for colour lookup
        if "RIGHT" in name and "WAVE" in name: key = "RIGHT_WAVE"
        elif "LEFT" in name and "WAVE" in name: key = "LEFT_WAVE"
        elif "HEAD" in name: key = "HEAD_LEFT"
        elif "FREEZE" in name: key = "CROSS_ARMS"
        elif "BALANCED" in name: key = "BALANCED_STAND"
        elif "HANDS UP" in name: key = "HANDS_UP"
        elif "HANDS FRONT" in name: key = "HANDS_FRONT"
        elif "WALK" in name: key = "WALK"

        active  = (gesture == key or
                   (key == "HEAD_LEFT" and gesture in ("HEAD_LEFT","HEAD_RIGHT")))
        txt_col = (255, 240, 80) if active else (150, 150, 150)
        cv2.putText(canvas, f"  {name:<18} {desc}",
                    (4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.30, txt_col, 1)
        y += 13

    # Settings strip
    sy = H - 40
    cv2.rectangle(canvas, (0, sy), (W, H), (22, 22, 38), -1)
    cv2.putText(canvas,
                f"Wave pause {wave_pause:.2f}s  [/]   "
                f"Speed {speed_scale:.2f}x  ,/.   "
                f"Walk {walk_spd:.1f}/s  +/-",
                (6, sy + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.31, (180, 220, 255), 1)
    cv2.putText(canvas, "W walk    S force stand    Q quit",
                (6, sy + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.31, (200, 200, 200), 1)

    # Camera thumbnail
    if frame is not None:
        th = cv2.resize(frame, (120, 90))
        canvas[sy - 95:sy - 5, W - 125:W - 5] = th

    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    dxl  = connect_dxl()
    live = dxl is not None

    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.65,
        min_tracking_confidence=0.60,
    )

    detector   = GestureDetector()
    controller = MotionController(dxl=dxl)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("\n[INFO] Running — stand ~1.5 m from camera so full body is visible")
    print("  Q quit   W toggle walk   S force stand")
    print("  +/- walk speed   [/] wave pause   ,/. transition speed\n")

    walk_on = False
    prev_t  = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            pose_g  = detector.update(results.pose_landmarks)
            gesture = "WALK" if walk_on else pose_g
            controller.set_gesture(gesture)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now

            # Camera feed overlays
            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)

            g_col = GESTURE_COLORS.get(gesture, (200, 200, 200))
            cv2.putText(annotated, f"CONFIRMED: {gesture}",
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, g_col, 2)
            cv2.putText(annotated, f"raw: {detector.raw}",
                        (8, annotated.shape[0] - 36),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (150, 150, 150), 1)

            is_exec    = controller.is_executing
            is_holding = controller.is_holding
            state_lbl  = "[HOLD]" if is_holding else ("[EXEC]" if is_exec else "[DONE]")
            state_col  = ((80,80,220) if is_holding else
                          ((0,200,100) if is_exec else (130,130,130)))
            cv2.putText(annotated, state_lbl,
                        (8, annotated.shape[0] - 62),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.48, state_col, 2)

            if walk_on:
                cv2.putText(annotated, "[W] WALK",
                            (annotated.shape[1] - 140, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,200,255), 2)

            hud = draw_hud(
                controller.current_angles, gesture, detector.raw,
                live, controller.walk_speed,
                is_exec, is_holding,
                WAVE_PEAK_PAUSE, SPEED_SCALE, frame,
            )

            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State",   hud)

            key = cv2.waitKey(1) & 0xFF
            if   key == ord("q"): break
            elif key == ord("w"):
                walk_on = not walk_on
                if not walk_on:
                    controller.force_stand()
                print(f"[KEY] Walk = {walk_on}")
            elif key == ord("s"):
                walk_on = False
                controller.force_stand()
                print("[KEY] Forced stand")
            elif key in (ord("+"), ord("=")):
                controller.set_walk_speed(+0.2)
            elif key == ord("-"):
                controller.set_walk_speed(-0.2)
            elif key == ord("["):
                controller.adjust_wave_pause(+0.05)
            elif key == ord("]"):
                controller.adjust_wave_pause(-0.05)
            elif key == ord(","):
                controller.adjust_speed_scale(+0.1)
            elif key == ord("."):
                controller.adjust_speed_scale(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Returning to stand...")
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        controller.force_stand()
        if live:
            dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.5)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[POSE CHECK] CLIPPING WARNINGS:
  L_WAVE_RAISE[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_RAISE[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  L_WAVE_A[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_A[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  L_WAVE_B[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_B[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  R_WAVE_RAISE[54 r_elbow_y] = 45  OUT OF [-114, 30]
  R_WAVE_A[54 r_elbow_y] = 100  OUT OF [-114, 30]
[DXL] Opened COM4 @ 1000000 baud
[DXL] WARNING -- missing: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Active (0): []
[DXL] Torque ON

[INFO] Running — stand ~1.5 m from camera so full body is visible
  Q quit   W toggle walk   S force stand
  +/- walk speed   [/] wave pause   ,/. transition speed



In [ ]:
import time
from pypot.dynamixel import DxlIO

# ========= CONFIG =========
PORT = "COM4"
BAUDRATE = 1000000

MOTOR_IDS = [
    11,12,13,14,15,
    21,22,23,24,25,
    31,32,33,34,35,
    36,37,
    41,42,43,44,
    51,52,53,54
]

labels = {
    11:"l_hip_x",12:"l_hip_z",13:"l_hip_y",14:"l_knee_y",15:"l_ankle_y",
    21:"r_hip_x",22:"r_hip_z",23:"r_hip_y",24:"r_knee_y",25:"r_ankle_y",
    31:"abs_y",32:"abs_x",33:"abs_z",34:"bust_y",35:"bust_x",
    36:"head_z",37:"head_y",
    41:"l_shoulder_y",42:"l_shoulder_x",43:"l_arm_z",44:"l_elbow_y",
    51:"r_shoulder_y",52:"r_shoulder_x",53:"r_arm_z",54:"r_elbow_y"
}

# ========= CONNECT =========
dxl = DxlIO(PORT, baudrate=BAUDRATE)
ids = dxl.scan(MOTOR_IDS)

print("Connected Motors:", ids)

# ========= INIT MIN/MAX =========
min_pos = {mid: float('inf') for mid in ids}
max_pos = {mid: float('-inf') for mid in ids}

# ========= TORQUE OFF =========
dxl.disable_torque(ids)
print("\n🔓 Torque OFF → Move robot freely")

print("\nInstructions:")
print("1. Set pose manually")
print("2. Press ENTER to capture")
print("3. Type pose name")
print("4. Repeat\n")

try:
    while True:
        input("👉 Set pose and press ENTER...")

        angles = dxl.get_present_position(ids)

        pose_name = input("Enter pose name (or 'q' to quit): ").strip()

        if pose_name.lower() == "q":
            break

        print(f"\n=== {pose_name.upper()} ===")
        print("{")

        for i, mid in enumerate(ids):
            val = angles[i]

            # ✅ update min/max
            if val < min_pos[mid]:
                min_pos[mid] = val
            if val > max_pos[mid]:
                max_pos[mid] = val

            print(f"    {mid}: {val:.2f},   # {labels[mid]}")

        print("}\n")
        print("✅ Pose captured!\n")

except KeyboardInterrupt:
    pass

# ========= PRINT MIN/MAX =========
print("\n========== MOTOR LIMITS ==========")
for mid in ids:
    print(f"{mid}: ({min_pos[mid]:.2f}, {max_pos[mid]:.2f})   # {labels[mid]}")

# ========= RESTORE =========
dxl.enable_torque(ids)
dxl.close()

print("\n✅ Done")

Connected Motors: []

🔓 Torque OFF → Move robot freely

Instructions:
1. Set pose manually
2. Press ENTER to capture
3. Type pose name
4. Repeat



In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        POPPY HUMANOID  --  GESTURE CONTROL  v9.0                            ║
║  Port    : COM4  (DxlIO direct)                                              ║
║  Vision  : MediaPipe Pose + Hands                                            ║
║  Drive   : 50 Hz LERP  +  smooth-step easing                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL :  pip install opencv-python mediapipe pypot numpy                  ║
║  RUN     :  python poppy_gesture_control.py                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  9 GESTURES  (stand ~1.5 m from webcam, full body visible)                  ║
║  ──────────────────────────────────────────────────────────────────────────  ║
║                                                                              ║
║  BALANCED STAND   BOTH thumbs up (MediaPipe Hands)                          ║
║                                                                              ║
║  SALUTE           Right wrist raised to forehead level (above nose),        ║
║                   only right wrist up, left stays low                       ║
║                                                                              ║
║  RIGHT WAVE       ONLY right wrist raised high (left stays down)            ║
║                   + wrist far from face + lateral swing detected             ║
║                                                                              ║
║  LEFT WAVE        ONLY left wrist raised high (right stays down)            ║
║                   + wrist far from face + lateral swing detected             ║
║                                                                              ║
║  HANDS UP         BOTH wrists clearly above BOTH shoulders                  ║
║                                                                              ║
║  HANDS FRONT      Both wrists chest-height, arms pushed forward             ║
║                                                                              ║
║  HEAD LEFT/RIGHT  Tilt/turn head — nose drifts past shoulder midpoint       ║
║                   (only fires when arms are NOT raised)                     ║
║                                                                              ║
║  FREEZE           Cross wrists at chest — both wrists near opposite shoulder║
║                                                                              ║
║  WALK             T-pose: both arms level with shoulders, spread wide       ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  GESTURE PRIORITY & MIXING PREVENTION                                        ║
║  SALUTE: right wrist above nose level, only right hand up, left stays low   ║
║  SALUTE has higher priority than WAVE — checked before wave logic            ║
║  THUMBS UP (both): detected via MediaPipe Hands — triggers BALANCED_STAND   ║
║  CROSS_ARMS: always wins immediately                                         ║
║  WALK (T-pose): wrists level (not above) AND wide                           ║
║  HANDS_UP: BOTH wrists above shoulders                                       ║
║  HEAD TILT: only fires when no wrists raised                                 ║
║  HANDS FRONT: only fires when no wrists raised                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEYBOARD                                                                    ║
║  Q  quit          W  toggle walk       S  force stand                       ║
║  +  walk faster   -  walk slower                                            ║
║  [  wave pause +  ]  wave pause -   (default 0.35 s)                       ║
║  ,  transitions slower   .  transitions faster                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import sys
import time
import threading
import collections
import numpy as np
import cv2
import mediapipe as mp

# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE
# ══════════════════════════════════════════════════════════════════════════════

PORT      = "COM4"
BAUDRATE  = 1000000
MOTOR_IDS = [11,12,13,14,15, 21,22,23,24,25,
             31,32,33,34,35,36,37,
             41,42,43,44, 51,52,53,54]

try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot not found -- SIMULATION mode")


# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR MAP
# ══════════════════════════════════════════════════════════════════════════════

MOTOR_MAP: dict[int, tuple[str, float, float]] = {
    11: ("l_hip_x",           -23,    22),
    12: ("l_hip_z",           -33,    15),
    13: ("l_hip_y",           -37,    44),
    14: ("l_knee_y",            3,    61),
    15: ("l_ankle_y",         -36,     5),
    21: ("r_hip_x",           -49,     0),
    22: ("r_hip_z",           -22,    17),
    23: ("r_hip_y",           -22,    46),
    24: ("r_knee_y",          -57,    14),
    25: ("r_ankle_y",         -20,    30),
    31: ("abs_y",             -30,    17),
    32: ("abs_x",             -25,    28),
    33: ("abs_z",             -22,    19),
    34: ("bust_y",            -43,    -1),
    35: ("bust_x",            -16,    21),
    36: ("head_z",            -48,    48),
    37: ("head_y",            -37,     7),
    41: ("l_shoulder_y",     -137,    77),
    42: ("l_shoulder_x",      -22,    85),
    43: ("l_arm_z",           -36,   115),
    44: ("l_elbow_y",        -146,    11),
    51: ("r_shoulder_y",      -84,   121),
    52: ("r_shoulder_x",     -111,    91),
    53: ("r_arm_z",          -105,   115),
    54: ("r_elbow_y",        -114,    30),
}


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY
# ══════════════════════════════════════════════════════════════════════════════

def _p(overrides: dict[int, float]) -> dict[int, float]:
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: dict[str, dict[int, float]] = {
    "BALANCED_STAND": {
        11:  -1.98,  12:  -3.74,  13:   4.88,  14:  13.32,  15: -16.40,
        21: -22.81,  22:   6.64,  23:   0.48,  24: -10.24,  25:   6.55,
        31:   6.02,  32:  -4.70,  33:  -0.48,  34: -32.13,  35:   4.44,
        36:  -3.08,  37: -26.83,
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ── LEFT WAVE ──────────────────────────────────────────────────────────────
POSES["L_WAVE_RAISE"] = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -40.0})
POSES["L_WAVE_A"]     = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -95.52})
POSES["L_WAVE_B"]     = _p({41: 111.0, 42: 11.47, 43: -88.75, 44: -30.0})

# ── RIGHT WAVE ─────────────────────────────────────────────────────────────
POSES["R_WAVE_RAISE"] = _p({51: 111, 52: 11.47, 53: -88.75, 54: 45})
POSES["R_WAVE_A"]     = _p({51: 111, 52: 11.47, 53: -88.75, 54: 100})
POSES["R_WAVE_B"]     = _p({51: 111, 52: 11.47, 53: -88.75, 54: 10})

# ── HANDS UP ───────────────────────────────────────────────────────────────
POSES["HANDS_UP"] = _p({
    41: -112.40,  42:  35.03,  43: -13.58,  44:  0.75,
    51:  110.02,  52: -56.84,  53:   9.80,  54:  0.40,
})

# ── HANDS FRONT ────────────────────────────────────────────────────────────
POSES["HANDS_FRONT"] = _p({
    41: -32.84,  42:  74.59,  43:  11.12,  44:  0.75,
    51:  28.44,  52: -100.53, 53:   3.84,  54:  2.04,
})

# ── HEAD TURNS ─────────────────────────────────────────────────────────────
POSES["HEAD_LEFT"]  = _p({36:  45.0})
POSES["HEAD_RIGHT"] = _p({36: -45.0})

# ── FREEZE / CROSS ARMS ────────────────────────────────────────────────────
POSES["CROSS_ARMS"] = _p({
    41:  15.0,  42: -11.0,  43:  50.0,  44: -100.0,
    51:  15.0,  52: -27.0,  53:  76.0,  54: -100.0,
})

# ── SALUTE ─────────────────────────────────────────────────────────────────
# Left arm stays in balanced stand; only RIGHT arm raises to forehead salute.
# We override only the left arm values explicitly here.
POSES["SALUTE"] = _p({
    # Left arm — salute position as specified
    41: -56.48,   # l_shoulder_y
    42:   8.31,   # l_shoulder_x
    43:  59.21,   # l_arm_z
    44: -135.52,  # l_elbow_y
    # Right arm stays at balanced stand (inherited from _p base)
})

# ── WALK KEYFRAMES ─────────────────────────────────────────────────────────
POSES["WALK_L_UP"] = {
    11:  -1.10,  12:   4.79,  13: -26.24,  14:  50.42,  15: -11.74,
    21: -23.16,  22:   1.98,  23: -11.21,  24:   3.74,  25:  -7.52,
    31: -12.09,  32:   6.73,  33: -11.56,  34: -12.53,  35:  -1.36,
    36:  -8.36,  37:  -3.96,
    41:  50.51,  42:  66.42,  43: 104.75,  44:  -8.04,
    51: -73.54,  52: -83.21,  53: -93.67,  54:   3.21,
}
POSES["WALK_L_DN"] = {
    11:  11.47,  12: -12.35,  13:  -3.91,  14:  28.35,  15:  -5.14,
    21: -10.68,  22:  -6.29,  23:  -0.57,  24: -37.32,  25:  19.47,
    31:  -6.37,  32: -14.64,  33:   8.22,  34: -27.21,  35:  10.68,
    36: -14.22,  37:  -3.96,
    41:  49.45,  42:  68.09,  43:  20.79,  44: -31.08,
    51: -61.05,  52: -93.67,  53: -94.02,  54:   3.12,
}
POSES["WALK_R_UP"] = {
    11:   6.29,  12: -22.02,  13:   6.20,  14:  16.57,  15: -21.32,
    21: -15.08,  22: -11.65,  23:  35.56,  24: -46.99,  25:  -9.36,
    31:  -3.74,  32:  -6.46,  33:   7.43,  34: -11.82,  35:  -4.62,
    36:   4.25,  37:  -8.94,
    41:  66.51,  42:  72.84,  43:  15.08,  44:   0.66,
    51: -72.66,  52: -87.60,  53:  -7.43,  54:  19.82,
}
POSES["WALK_R_DN"] = {
    11: -12.35,  12:  -5.58,  13:  33.10,  14:  13.32,  15: -25.01,
    21: -38.64,  22:   4.79,  23:   1.45,  24: -30.55,  25:   7.87,
    31: -19.56,  32:  17.10,  33: -11.21,  34: -18.95,  35:  -5.14,
    36:  25.07,  37:  -5.13,
    41:  64.48,  42:  70.81,  43:  14.99,  44:  -6.02,
    51: -70.11,  52: -86.11,  53: -29.76,  54:  10.86,
}

WALK_CYCLE = ["WALK_L_UP", "WALK_L_DN", "WALK_R_UP", "WALK_R_DN"]


# ══════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def _check_poses() -> None:
    errors = []
    for pname, pose in POSES.items():
        for mid, val in pose.items():
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            if not (lo <= val <= hi):
                errors.append(
                    f"  {pname}[{mid} {MOTOR_MAP[mid][0]}] = {val}  "
                    f"OUT OF [{lo}, {hi}]"
                )
    if errors:
        print("[POSE CHECK] CLIPPING WARNINGS:")
        for e in errors: print(e)
    else:
        print("[POSE CHECK] All poses within limits -- OK")

_check_poses()


# ══════════════════════════════════════════════════════════════════════════════
#  TIMING  (all tunable at runtime)
# ══════════════════════════════════════════════════════════════════════════════

BASE_STEP_T: dict[str, float] = {
    "BALANCED_STAND": 0.60,
    "RIGHT_WAVE":     0.20,   # per oscillation step
    "LEFT_WAVE":      0.20,
    "HANDS_UP":       0.65,
    "HANDS_FRONT":    0.55,
    "HEAD_LEFT":      0.38,
    "HEAD_RIGHT":     0.38,
    "CROSS_ARMS":     0.45,
    "WALK":           0.22,
    "SALUTE":         0.55,   # smooth raise to forehead
}

GESTURE_HOLD_T: dict[str, float] = {
    "HANDS_UP":       1.50,
    "HANDS_FRONT":    1.20,
    "CROSS_ARMS":     1.80,
    "HEAD_LEFT":      0.90,
    "HEAD_RIGHT":     0.90,
    "BALANCED_STAND": 0.00,
    "SALUTE":         1.20,   # hold the salute for 1.2 s before lowering
}

# Time to pause between two distinct motion phases (e.g. raise → hold → lower)
INTER_MOVE_PAUSE: float = 0.40   # seconds between motion segments

WAVE_PEAK_PAUSE: float = 0.35    # pause at each wave peak  → [ ] keys
SPEED_SCALE:     float = 1.0     # multiplies all step times → , . keys


# ══════════════════════════════════════════════════════════════════════════════
#  THUMBS-UP DETECTOR  (MediaPipe Hands)
# ══════════════════════════════════════════════════════════════════════════════

class ThumbsUpDetector:
    """
    Uses MediaPipe Hands to detect whether BOTH visible hands show a thumbs-up.
    A thumbs-up is: thumb tip clearly above the index MCP knuckle,
    and all other fingers curled (fingertips below their own PIP joints).
    Requires both hands in view and both showing thumbs-up for BALANCED_STAND.
    """

    CONFIRM_FRAMES = 8   # frames both hands must stay thumbs-up

    def __init__(self) -> None:
        self._hands = mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.65,
            min_tracking_confidence=0.60,
        )
        self._buf: collections.deque = collections.deque(maxlen=self.CONFIRM_FRAMES)
        self.confirmed = False

    def update(self, rgb_frame) -> bool:
        result = self._hands.process(rgb_frame)
        raw = self._both_thumbs_up(result)
        self._buf.append(raw)
        if len(self._buf) == self.CONFIRM_FRAMES and all(self._buf):
            self.confirmed = True
        elif not raw:
            self.confirmed = False
        return self.confirmed

    def _both_thumbs_up(self, result) -> bool:
        if not result.multi_hand_landmarks:
            return False
        if len(result.multi_hand_landmarks) < 2:
            return False   # need both hands
        return all(self._is_thumbs_up(hl) for hl in result.multi_hand_landmarks)

    @staticmethod
    def _is_thumbs_up(hand_lm) -> bool:
        lm = hand_lm.landmark
        H  = mp.solutions.hands.HandLandmark

        # Thumb tip clearly above index MCP (lower y = higher on screen)
        thumb_up = lm[H.THUMB_TIP].y < lm[H.INDEX_FINGER_MCP].y - 0.04

        # Other four fingers curled: tip y > pip y  (tip is lower = curled)
        fingers_curled = all(
            lm[tip].y > lm[pip].y
            for tip, pip in [
                (H.INDEX_FINGER_TIP,  H.INDEX_FINGER_PIP),
                (H.MIDDLE_FINGER_TIP, H.MIDDLE_FINGER_PIP),
                (H.RING_FINGER_TIP,   H.RING_FINGER_PIP),
                (H.PINKY_TIP,         H.PINKY_PIP),
            ]
        )
        return thumb_up and fingers_curled

    def close(self) -> None:
        self._hands.close()


# ══════════════════════════════════════════════════════════════════════════════
#  GESTURE DETECTOR  v9.0
# ══════════════════════════════════════════════════════════════════════════════

class GestureDetector:
    """
    Clean, non-overlapping detection zones for all 9 gestures.

    Priority order (checked top-to-bottom, first match wins):
      0. BALANCED_STAND — both thumbs up (handled externally via ThumbsUpDetector)
      1. CROSS_ARMS  — wrists crossed at chest
      2. WALK        — T-pose (both arms level AND wide)
      3. HANDS_UP    — BOTH wrists above BOTH shoulders
      4. SALUTE      — right wrist above nose level, only right hand up
      5. RIGHT_WAVE  — ONLY right wrist up, away from face, swinging
      6. LEFT_WAVE   — ONLY left  wrist up, away from face, swinging
      7. HEAD_LEFT / HEAD_RIGHT — nose offset (only when arms down)
      8. HANDS_FRONT — both wrists chest-height (only when arms down)
      9. BALANCED_STAND — everything else
    """

    DEBOUNCE        = 7      # frames for stable confirmation
    WAVE_HEIGHT     = 0.06   # wrist above shoulder threshold (normalised)
    WAVE_BUF        = 9      # x-history frames for swing detection
    WAVE_SWING      = 0.055  # min wrist x-range to count as swing
    WAVE_FACE_DIST  = 0.20   # wrist must be this far from nose.x
    TPOSE_SPREAD    = 0.44   # min wrist-to-wrist x-spread for T-pose
    TPOSE_TOL       = 0.09   # T-pose: wrist y vs shoulder y tolerance
    HEAD_THRESH     = 0.060  # nose x-offset from shoulder midpoint
    CROSS_X_TOL     = 0.14
    CROSS_Y_TOL     = 0.17

    # Salute: right wrist must be at or above nose y, and NOT far out to side
    SALUTE_NOSE_TOL = 0.04   # wrist.y must be < nose.y + this (i.e. at/above nose)
    SALUTE_X_TOL    = 0.30   # wrist must be within 0.30 of nose.x (near face)

    def __init__(self) -> None:
        self._hist: collections.deque = collections.deque(maxlen=self.DEBOUNCE)
        self._wx = {
            "r": collections.deque(maxlen=self.WAVE_BUF),
            "l": collections.deque(maxlen=self.WAVE_BUF),
        }
        self.confirmed  = "BALANCED_STAND"
        self._raw_label = "BALANCED_STAND"

    def update(self, landmarks) -> str:
        if landmarks is None:
            return self.confirmed
        raw = self._classify(landmarks.landmark)
        self._raw_label = raw
        self._hist.append(raw)
        if len(self._hist) == self.DEBOUNCE and len(set(self._hist)) == 1:
            self.confirmed = raw
        return self.confirmed

    @property
    def raw(self) -> str:
        return self._raw_label

    def _classify(self, lm) -> str:
        PL    = mp.solutions.pose.PoseLandmark
        nose  = lm[PL.NOSE]
        l_sh  = lm[PL.LEFT_SHOULDER]
        r_sh  = lm[PL.RIGHT_SHOULDER]
        l_hip = lm[PL.LEFT_HIP]
        r_hip = lm[PL.RIGHT_HIP]
        l_wr  = lm[PL.LEFT_WRIST]
        r_wr  = lm[PL.RIGHT_WRIST]

        sh_mid_x  = (l_sh.x + r_sh.x) / 2.0
        hip_mid_y = (l_hip.y + r_hip.y) / 2.0

        # Wrist-above-shoulder flags
        r_up = r_wr.y < r_sh.y - self.WAVE_HEIGHT
        l_up = l_wr.y < l_sh.y - self.WAVE_HEIGHT

        # ── 1. CROSS ARMS ────────────────────────────────────────────────────
        l_to_rsh = (abs(l_wr.x - r_sh.x) < self.CROSS_X_TOL and
                    abs(l_wr.y - r_sh.y) < self.CROSS_Y_TOL)
        r_to_lsh = (abs(r_wr.x - l_sh.x) < self.CROSS_X_TOL and
                    abs(r_wr.y - l_sh.y) < self.CROSS_Y_TOL)
        if l_to_rsh and r_to_lsh:
            return "CROSS_ARMS"

        # ── 2. WALK — T-pose ─────────────────────────────────────────────────
        l_level = abs(l_wr.y - l_sh.y) < self.TPOSE_TOL
        r_level = abs(r_wr.y - r_sh.y) < self.TPOSE_TOL
        wide    = abs(l_wr.x - r_wr.x) > self.TPOSE_SPREAD
        if l_level and r_level and wide:
            return "WALK"

        # ── 3. HANDS UP — both wrists above shoulders ────────────────────────
        if r_up and l_up:
            return "HANDS_UP"

        # ── 4. SALUTE — right wrist near forehead, only right hand up ────────
        # Right wrist must be at or above nose level AND close to face (not waving)
        if r_up and not l_up:
            r_at_forehead = r_wr.y < nose.y + self.SALUTE_NOSE_TOL
            r_near_face   = abs(r_wr.x - nose.x) < self.SALUTE_X_TOL
            if r_at_forehead and r_near_face:
                return "SALUTE"

            # Not a salute — check if it's a wave (far from face + swinging)
            self._wx["r"].append(r_wr.x)
            face_clear = abs(r_wr.x - nose.x) > self.WAVE_FACE_DIST
            if face_clear and self._swing("r"):
                return "RIGHT_WAVE"
            return "BALANCED_STAND"

        # ── 5. LEFT WAVE — only left wrist up, away from face, swinging ─────
        if l_up and not r_up:
            self._wx["l"].append(l_wr.x)
            face_clear = abs(l_wr.x - nose.x) > self.WAVE_FACE_DIST
            if face_clear and self._swing("l"):
                return "LEFT_WAVE"
            return "BALANCED_STAND"

        # ── 6 & 7: Arms-down-only gestures ───────────────────────────────────

        # ── 6. HEAD TILT ─────────────────────────────────────────────────────
        offset = nose.x - sh_mid_x
        if offset >  self.HEAD_THRESH:
            return "HEAD_RIGHT"
        if offset < -self.HEAD_THRESH:
            return "HEAD_LEFT"

        # ── 7. HANDS FRONT ───────────────────────────────────────────────────
        l_front = (l_sh.y < l_wr.y < hip_mid_y and
                   abs(l_wr.x - l_sh.x) < 0.22)
        r_front = (r_sh.y < r_wr.y < hip_mid_y and
                   abs(r_wr.x - r_sh.x) < 0.22)
        if l_front and r_front:
            return "HANDS_FRONT"

        return "BALANCED_STAND"

    def _swing(self, s: str) -> bool:
        h = self._wx[s]
        return len(h) == self.WAVE_BUF and (max(h) - min(h)) > self.WAVE_SWING


# ══════════════════════════════════════════════════════════════════════════════
#  DXL INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

class DxlInterface:
    MOVING_SPEED_RPM = 50

    def __init__(self, port: str, baudrate: int, ids: list[int]) -> None:
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")
        found   = set(self._io.scan(ids))
        missing = set(ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing: {sorted(missing)}")
        self._active = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")
        self._io.set_moving_speed({m: self.MOVING_SPEED_RPM for m in self._active})
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: dict[int, float]) -> None:
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> "DxlInterface | None":
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e} -- SIMULATION mode")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER  v9.0
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_PRIORITY: dict[str, int] = {
    "CROSS_ARMS":     10,
    "BALANCED_STAND": 9,   # thumbs-up stand can always interrupt (high priority)
    "WALK":            8,
    "SALUTE":          6,
    "HANDS_UP":        5,
    "HANDS_FRONT":     5,
    "HEAD_LEFT":       4,
    "HEAD_RIGHT":      4,
    "RIGHT_WAVE":      3,
    "LEFT_WAVE":       3,
}

_LOOPING = {"RIGHT_WAVE", "LEFT_WAVE", "WALK"}

_SEQ: dict[str, list[str]] = {
    "BALANCED_STAND": ["BALANCED_STAND"],
    "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],
    "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
    "HANDS_UP":       ["HANDS_UP"],
    "HANDS_FRONT":    ["HANDS_FRONT"],
    "HEAD_LEFT":      ["HEAD_LEFT"],
    "HEAD_RIGHT":     ["HEAD_RIGHT"],
    "CROSS_ARMS":     ["CROSS_ARMS"],
    "WALK":           WALK_CYCLE,
    "SALUTE":         ["SALUTE"],
}


class MotionController:
    DRIVE_HZ = 50

    def __init__(self, dxl: "DxlInterface | None") -> None:
        self._dxl   = dxl
        self._lock  = threading.Lock()
        self._cur:  dict[int, float] = POSES["BALANCED_STAND"].copy()
        self._tgt:  dict[int, float] = POSES["BALANCED_STAND"].copy()

        self._gesture  = "BALANCED_STAND"
        self._gthread: threading.Thread | None = None
        self._stop     = threading.Event()
        self._alpha    = 0.40
        self._walk_spd = 1.0

        # State flags
        self._executing = False
        self._holding   = False
        self._done      = True    # starts done so first gesture fires

        threading.Thread(target=self._drive_loop, daemon=True).start()

        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.0)

    # ── public ───────────────────────────────────────────────────────────────

    def set_gesture(self, g: str) -> None:
        """
        Request a gesture change, subject to interrupt rules:
          - CROSS_ARMS / BALANCED_STAND (thumbs): always wins immediately
          - Gesture DONE:                anything accepted
          - Same gesture running:        ignored
          - Non-looping EXECUTING:       blocked (must complete fully)
          - Non-looping HOLDING:         only strictly higher priority breaks
          - LOOPING running:             same-or-higher priority can interrupt
          - Lower priority:              always blocked
        """
        with self._lock:
            cur_pri   = GESTURE_PRIORITY.get(self._gesture, 0)
            new_pri   = GESTURE_PRIORITY.get(g, 0)
            executing = self._executing
            holding   = self._holding
            done      = self._done
            looping   = self._gesture in _LOOPING

        # Always-interrupt gestures
        if g in ("CROSS_ARMS", "BALANCED_STAND"):  pass
        elif done:                                   pass
        elif g == self._gesture:                     return
        elif executing and not looping:              return
        elif holding and new_pri <= cur_pri:         return
        elif looping and new_pri < cur_pri:          return
        elif not done and not looping and new_pri < cur_pri: return

        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=3.0)
        self._stop.clear()

        with self._lock:
            self._gesture   = g
            self._executing = False
            self._holding   = False
            self._done      = False

        self._gthread = threading.Thread(
            target=self._run, args=(g,), daemon=True)
        self._gthread.start()

    def set_walk_speed(self, delta: float) -> None:
        self._walk_spd = float(np.clip(self._walk_spd + delta, 0.3, 3.0))
        print(f"[WALK] speed = {self._walk_spd:.1f} steps/s")

    def adjust_wave_pause(self, delta: float) -> None:
        global WAVE_PEAK_PAUSE
        WAVE_PEAK_PAUSE = float(np.clip(WAVE_PEAK_PAUSE + delta, 0.05, 1.5))
        print(f"[WAVE] peak pause = {WAVE_PEAK_PAUSE:.2f}s")

    def adjust_speed_scale(self, delta: float) -> None:
        global SPEED_SCALE
        SPEED_SCALE = float(np.clip(SPEED_SCALE + delta, 0.3, 3.0))
        print(f"[SPEED] scale = {SPEED_SCALE:.2f}x  (1.0 = default)")

    def force_stand(self) -> None:
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=2.0)
        self._stop.clear()
        with self._lock:
            self._gesture   = "BALANCED_STAND"
            self._executing = False
            self._holding   = False
            self._done      = True
            self._tgt       = POSES["BALANCED_STAND"].copy()

    @property
    def current_angles(self) -> dict[int, float]:
        with self._lock: return self._cur.copy()

    @property
    def walk_speed(self) -> float:
        return self._walk_spd

    @property
    def is_holding(self) -> bool:
        with self._lock: return self._holding

    @property
    def is_executing(self) -> bool:
        with self._lock: return self._executing

    def shutdown(self) -> None:
        self._stop.set()
        time.sleep(0.4)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ── gesture runner ───────────────────────────────────────────────────────

    def _run(self, gesture: str) -> None:
        frames  = _SEQ.get(gesture, ["BALANCED_STAND"])
        looping = gesture in _LOOPING
        step_t  = BASE_STEP_T.get(gesture, 0.40)
        hold_t  = GESTURE_HOLD_T.get(gesture, 0.0)
        is_wave = gesture in ("RIGHT_WAVE", "LEFT_WAVE")

        with self._lock:
            self._executing = True
            self._holding   = False
            self._done      = False

        try:
            # ── WALK ─────────────────────────────────────────────────────────
            if gesture == "WALK":
                while not self._stop.is_set():
                    wt = 1.0 / self._walk_spd
                    for name in frames:
                        if self._stop.is_set(): break
                        self._lerp(POSES[name], wt)
                return

            # ── WAVE ──────────────────────────────────────────────────────────
            elif is_wave:
                # Raise arm once, then oscillate A ↔ B with peak pauses
                self._lerp(POSES[frames[0]], step_t * SPEED_SCALE)
                if self._stop.is_set(): return
                # Small pause after raise, before oscillation begins
                if not self._interruptible_sleep(INTER_MOVE_PAUSE): return
                loop_frames = frames[1:]
                while not self._stop.is_set():
                    for name in loop_frames:
                        if self._stop.is_set(): break
                        self._lerp(POSES[name], step_t * SPEED_SCALE)
                        if not self._stop.is_set():
                            self._interruptible_sleep(WAVE_PEAK_PAUSE)
                return

            # ── NON-LOOPING ───────────────────────────────────────────────────
            else:
                # Execute all frames with inter-move pause between each
                for idx, name in enumerate(frames):
                    if self._stop.is_set(): return
                    self._lerp(POSES[name], step_t * SPEED_SCALE)
                    # Pause between frames (but not after the last one)
                    if idx < len(frames) - 1:
                        if not self._interruptible_sleep(INTER_MOVE_PAUSE): return

                if self._stop.is_set(): return

                # Hold phase
                if hold_t > 0.0:
                    with self._lock:
                        self._holding   = True
                        self._executing = False
                    try:
                        self._interruptible_sleep(hold_t)
                    finally:
                        with self._lock:
                            self._holding = False
                else:
                    with self._lock:
                        self._executing = False

                if self._stop.is_set(): return

                # Inter-move pause before returning to stand
                if not self._interruptible_sleep(INTER_MOVE_PAUSE): return

                # Return to balanced stand
                self._lerp(POSES["BALANCED_STAND"],
                            BASE_STEP_T["BALANCED_STAND"] * SPEED_SCALE)

        finally:
            with self._lock:
                self._executing = False
                self._holding   = False
                if gesture == self._gesture and not self._stop.is_set():
                    self._done = True

    # ── helpers ───────────────────────────────────────────────────────────────

    def _interruptible_sleep(self, duration: float,
                              granularity: float = 0.05) -> bool:
        """Sleep for duration seconds; returns False if interrupted."""
        deadline = time.monotonic() + duration
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            if self._stop.wait(timeout=min(granularity, max(remaining, 0.0))):
                return False
        return True

    def _lerp(self, target: dict[int, float], step_t: float) -> None:
        steps = max(int(step_t * self.DRIVE_HZ), 2)
        with self._lock:
            start = self._cur.copy()
        for i in range(1, steps + 1):
            if self._stop.is_set(): return
            t = _smooth(i / steps)
            with self._lock:
                self._tgt = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    def _drive_loop(self) -> None:
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                tgt = self._tgt.copy()
                cur = self._cur.copy()
            blended = {
                mid: cur[mid] + self._alpha * (tgt[mid] - cur[mid])
                for mid in tgt
            }
            with self._lock:
                self._cur = blended
            if self._dxl:
                self._dxl.set_positions(blended)
            time.sleep(dt)


def _smooth(t: float) -> float:
    return t * t * (3.0 - 2.0 * t)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_COLORS = {
    "BALANCED_STAND": (40,  160,  60),
    "RIGHT_WAVE":     (60,  210, 255),
    "LEFT_WAVE":      (60,  210, 255),
    "HANDS_UP":       (60,  255, 180),
    "HANDS_FRONT":    (60,  255, 120),
    "HEAD_LEFT":      (200, 140,  60),
    "HEAD_RIGHT":     (200, 140,  60),
    "CROSS_ARMS":     (80,   80, 220),
    "WALK":           (255, 160,   0),
    "SALUTE":         (220, 180,  60),  # gold tone for salute
}

_GROUPS = [
    ("LEFT  LEG",  [11, 12, 13, 14, 15]),
    ("RIGHT LEG",  [21, 22, 23, 24, 25]),
    ("TORSO",      [31, 32, 33, 34, 35]),
    ("HEAD",       [36, 37]),
    ("LEFT  ARM",  [41, 42, 43, 44]),
    ("RIGHT ARM",  [51, 52, 53, 54]),
]

_GUIDE = [
    ("BALANCED STAND", "both thumbs up (MediaPipe Hands)"),
    ("SALUTE",         "R wrist to forehead, left arm down"),
    ("RIGHT WAVE",     "only R wrist up + swing away from face"),
    ("LEFT  WAVE",     "only L wrist up + swing away from face"),
    ("HANDS UP",       "BOTH wrists above both shoulders"),
    ("HANDS FRONT",    "both wrists chest-height, arms forward"),
    ("HEAD LEFT/R",    "turn head — nose drifts sideways"),
    ("FREEZE",         "cross wrists at chest"),
    ("WALK",           "T-pose: arms level + wide"),
]


def draw_hud(angles, gesture, raw_gesture, live, walk_spd,
             executing, holding, wave_pause, speed_scale,
             thumbs_up_raw=False, frame=None):
    W, H = 500, 740
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:] = (14, 14, 24)

    # Title bar
    col = GESTURE_COLORS.get(gesture, (100, 100, 100))
    cv2.rectangle(canvas, (0, 0), (W, 46), col, -1)
    mode    = "DXL LIVE" if live else "SIMULATION"
    spd_str = f"  {walk_spd:.1f}/s" if gesture == "WALK" else ""
    state   = "  [HOLD]" if holding else ("  [EXEC]" if executing else "  [DONE]")
    cv2.putText(canvas, f"{mode}  {gesture}{spd_str}{state}",
                (8, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (10, 10, 10), 2)

    # Raw detection (unconfirmed)
    thumbs_str = "  THUMBS-UP!" if thumbs_up_raw else ""
    cv2.putText(canvas, f"raw detect: {raw_gesture}{thumbs_str}",
                (8, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (150, 150, 150), 1)

    # Motor bars
    y = 76
    for grp, mids in _GROUPS:
        cv2.putText(canvas, grp, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.37, (110, 170, 255), 1)
        y += 15
        for mid in mids:
            label  = MOTOR_MAP[mid][0]
            val    = angles.get(mid, 0.0)
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            bw     = max(int(np.interp(val, [lo, hi], [0, W - 124])), 0)
            cv2.rectangle(canvas, (104, y - 10), (104 + bw, y + 2),
                          (45, 165, 90), -1)
            cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                        (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.34, (210, 210, 210), 1)
            y += 16
        y += 4

    # Gesture guide
    y += 4
    cv2.putText(canvas, "GESTURE GUIDE", (8, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.36, (80, 180, 255), 1)
    y += 14
    for name, desc in _GUIDE:
        key = name.strip().upper()
        if "RIGHT" in name and "WAVE" in name: key = "RIGHT_WAVE"
        elif "LEFT" in name and "WAVE" in name: key = "LEFT_WAVE"
        elif "HEAD" in name: key = "HEAD_LEFT"
        elif "FREEZE" in name: key = "CROSS_ARMS"
        elif "BALANCED" in name: key = "BALANCED_STAND"
        elif "HANDS UP" in name: key = "HANDS_UP"
        elif "HANDS FRONT" in name: key = "HANDS_FRONT"
        elif "WALK" in name: key = "WALK"
        elif "SALUTE" in name: key = "SALUTE"

        active  = (gesture == key or
                   (key == "HEAD_LEFT" and gesture in ("HEAD_LEFT","HEAD_RIGHT")))
        txt_col = (255, 240, 80) if active else (150, 150, 150)
        cv2.putText(canvas, f"  {name:<18} {desc}",
                    (4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.30, txt_col, 1)
        y += 13

    # Settings strip
    sy = H - 40
    cv2.rectangle(canvas, (0, sy), (W, H), (22, 22, 38), -1)
    cv2.putText(canvas,
                f"Wave pause {wave_pause:.2f}s  [/]   "
                f"Speed {speed_scale:.2f}x  ,/.   "
                f"Walk {walk_spd:.1f}/s  +/-",
                (6, sy + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.31, (180, 220, 255), 1)
    cv2.putText(canvas, "W walk    S force stand    Q quit",
                (6, sy + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.31, (200, 200, 200), 1)

    # Camera thumbnail
    if frame is not None:
        th = cv2.resize(frame, (120, 90))
        canvas[sy - 95:sy - 5, W - 125:W - 5] = th

    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    dxl  = connect_dxl()
    live = dxl is not None

    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.65,
        min_tracking_confidence=0.60,
    )

    detector      = GestureDetector()
    thumbs_det    = ThumbsUpDetector()
    controller    = MotionController(dxl=dxl)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("\n[INFO] Running — stand ~1.5 m from camera so full body is visible")
    print("  BOTH THUMBS UP → Balanced Stand")
    print("  RIGHT WRIST TO FOREHEAD → Salute")
    print("  Q quit   W toggle walk   S force stand")
    print("  +/- walk speed   [/] wave pause   ,/. transition speed\n")

    walk_on = False
    prev_t  = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results   = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            # Thumbs-up detection (needs writeable rgb — process separately)
            rgb_hands = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            thumbs_up = thumbs_det.update(rgb_hands)

            pose_g = detector.update(results.pose_landmarks)

            # Gesture priority: thumbs-up overrides pose gestures → BALANCED_STAND
            if thumbs_up:
                gesture = "BALANCED_STAND"
            elif walk_on:
                gesture = "WALK"
            else:
                gesture = pose_g

            controller.set_gesture(gesture)

            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 220, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(200, 200, 0), thickness=2),
                )

            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now

            # Camera feed overlays
            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)

            g_col = GESTURE_COLORS.get(gesture, (200, 200, 200))
            cv2.putText(annotated, f"CONFIRMED: {gesture}",
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, g_col, 2)
            cv2.putText(annotated, f"raw: {detector.raw}",
                        (8, annotated.shape[0] - 36),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (150, 150, 150), 1)

            is_exec    = controller.is_executing
            is_holding = controller.is_holding
            state_lbl  = "[HOLD]" if is_holding else ("[EXEC]" if is_exec else "[DONE]")
            state_col  = ((80,80,220) if is_holding else
                          ((0,200,100) if is_exec else (130,130,130)))
            cv2.putText(annotated, state_lbl,
                        (8, annotated.shape[0] - 62),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.48, state_col, 2)

            if thumbs_up:
                cv2.putText(annotated, "THUMBS UP!",
                            (annotated.shape[1] - 160, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 255, 80), 2)

            if walk_on:
                cv2.putText(annotated, "[W] WALK",
                            (annotated.shape[1] - 140, 56),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,200,255), 2)

            hud = draw_hud(
                controller.current_angles, gesture, detector.raw,
                live, controller.walk_speed,
                is_exec, is_holding,
                WAVE_PEAK_PAUSE, SPEED_SCALE,
                thumbs_up_raw=thumbs_up,
                frame=frame,
            )

            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State",   hud)

            key = cv2.waitKey(1) & 0xFF
            if   key == ord("q"): break
            elif key == ord("w"):
                walk_on = not walk_on
                if not walk_on:
                    controller.force_stand()
                print(f"[KEY] Walk = {walk_on}")
            elif key == ord("s"):
                walk_on = False
                controller.force_stand()
                print("[KEY] Forced stand")
            elif key in (ord("+"), ord("=")):
                controller.set_walk_speed(+0.2)
            elif key == ord("-"):
                controller.set_walk_speed(-0.2)
            elif key == ord("["):
                controller.adjust_wave_pause(+0.05)
            elif key == ord("]"):
                controller.adjust_wave_pause(-0.05)
            elif key == ord(","):
                controller.adjust_speed_scale(+0.1)
            elif key == ord("."):
                controller.adjust_speed_scale(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Returning to stand...")
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        thumbs_det.close()
        controller.force_stand()
        if live:
            dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.5)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[POSE CHECK] CLIPPING WARNINGS:
  L_WAVE_RAISE[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_RAISE[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  L_WAVE_A[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_A[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  L_WAVE_B[41 l_shoulder_y] = 111.0  OUT OF [-137, 77]
  L_WAVE_B[43 l_arm_z] = -88.75  OUT OF [-36, 115]
  R_WAVE_RAISE[54 r_elbow_y] = 45  OUT OF [-114, 30]
  R_WAVE_A[54 r_elbow_y] = 100  OUT OF [-114, 30]
[DXL] Opened COM4 @ 1000000 baud
[DXL] WARNING -- missing: [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Active (0): []
[DXL] Torque ON

[INFO] Running — stand ~1.5 m from camera so full body is visible
  BOTH THUMBS UP → Balanced Stand
  RIGHT WRIST TO FOREHEAD → Salute
  Q quit   W toggle walk   S force stand
  +/- walk speed   [/] wave pause   ,/. transition speed



In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        POPPY HUMANOID  --  FINGER + HEAD CONTROL  v10.0  (FINAL)           ║
║  Port    : COM4  (DxlIO direct)                                              ║
║  Vision  : MediaPipe Hands (fingers) + MediaPipe Pose (head tracking)       ║
║  Drive   : 50 Hz LERP  +  smooth-step easing                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INSTALL :  pip install opencv-python mediapipe pypot numpy                  ║
║  RUN     :  python poppy_gesture_control_v10.py                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  FINGER COMMANDS  (show one hand clearly, ~0.5-1 m from camera)             ║
║  ─────────────────────────────────────────────────────────────────────────  ║
║  0  Fist          →  Balanced stand  (always available)                     ║
║  1  Index up      →  Right wave                                              ║
║  2  Peace sign    →  Left wave                                               ║
║  3  Three up      →  Hands up                                                ║
║  4  Four up       →  Hands front                                             ║
║  5  Open palm     →  Walk                                                    ║
║  8  Eight fingers →  Salute                                                  ║
║                                                                              ║
║  HEAD TRACKING  (pose landmarks — continuous, no finger needed)             ║
║  Nose drifts left  →  Head left                                             ║
║  Nose drifts right →  Head right                                            ║
║  Only active when NO finger command is running                              ║
║                                                                              ║
║  FLOW                                                                        ║
║  Every gesture completes → returns to BALANCED_STAND → waits for next      ║
║  Fist (0) at any time forces immediate return to stand                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEYBOARD                                                                    ║
║  Q  quit          S  force stand                                            ║
║  +  walk faster   -  walk slower                                            ║
║  [  wave pause+   ]  wave pause-                                            ║
║  ,  transitions slower   .  transitions faster                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import sys
import time
import threading
import collections
from typing import Optional, Tuple, Dict, List  # ← FIX: use typing module for Python 3.9
import numpy as np
import cv2
import mediapipe as mp

# ══════════════════════════════════════════════════════════════════════════════
#  HARDWARE
# ══════════════════════════════════════════════════════════════════════════════

PORT      = "COM4"
BAUDRATE  = 1000000
MOTOR_IDS = [11,12,13,14,15, 21,22,23,24,25,
             31,32,33,34,35,36,37,
             41,42,43,44, 51,52,53,54]

try:
    from pypot.dynamixel import DxlIO
    DXLIO_AVAILABLE = True
except ImportError:
    DXLIO_AVAILABLE = False
    print("[WARN] pypot not found -- SIMULATION mode")


# ══════════════════════════════════════════════════════════════════════════════
#  MOTOR MAP
# ══════════════════════════════════════════════════════════════════════════════

MOTOR_MAP: Dict[int, Tuple[str, float, float]] = {   # ← FIX
    11: ("l_hip_x",           -23,    22),
    12: ("l_hip_z",           -33,    15),
    13: ("l_hip_y",           -37,    44),
    14: ("l_knee_y",            3,    61),
    15: ("l_ankle_y",         -36,     5),
    21: ("r_hip_x",           -49,     0),
    22: ("r_hip_z",           -22,    17),
    23: ("r_hip_y",           -22,    46),
    24: ("r_knee_y",          -57,    14),
    25: ("r_ankle_y",         -20,    30),
    31: ("abs_y",             -30,    17),
    32: ("abs_x",             -25,    28),
    33: ("abs_z",             -22,    19),
    34: ("bust_y",            -43,    -1),
    35: ("bust_x",            -16,    21),
    36: ("head_z",            -48,    48),
    37: ("head_y",            -37,     7),
    41: ("l_shoulder_y",     -137,    77),
    42: ("l_shoulder_x",      -22,    85),
    43: ("l_arm_z",           -36,   115),
    44: ("l_elbow_y",        -146,    11),
    51: ("r_shoulder_y", -155.0,  120.0),
    52: ("r_shoulder_x",     -111,    91),
    53: ("r_arm_z",          -105,   115),
    54: ("r_elbow_y",        -1,    148),
}


# ══════════════════════════════════════════════════════════════════════════════
#  POSE LIBRARY
# ══════════════════════════════════════════════════════════════════════════════

def _p(overrides: Dict[int, float]) -> Dict[int, float]:   # ← FIX
    base = POSES["BALANCED_STAND"].copy()
    base.update(overrides)
    return base


POSES: Dict[str, Dict[int, float]] = {   # ← FIX
    "BALANCED_STAND": {
        11:  -1.98,  12:  -3.74,  13:   4.88,  14:  13.32,  15: -16.40,
        21: -22.81,  22:   6.64,  23:   0.48,  24: -10.24,  25:   6.55,
        31:   6.02,  32:  -4.70,  33:  -0.48,  34: -32.13,  35:   4.44,
        36:  -3.08,  37: -26.83,
        41:  60.70,  42:  66.33,  43: -25.45,  44:   0.48,
        51: 101.32,  52:  80.92,  53: 104.13,  54:   7.34,
    },
}

# ── LEFT WAVE ──────────────────────────────────────────────────────────────
POSES["L_WAVE_RAISE"] = _p({41: -111.0, 42: -11.47, 43: 88.75, 44: -40.0})
POSES["L_WAVE_A"]     = _p({41: -111.0, 42: -11.47, 43: 88.75, 44: -85.52})
POSES["L_WAVE_B"]     = _p({41: -111.0, 42: -11.47, 43: 88.75, 44: 5.0})

# ── RIGHT WAVE ─────────────────────────────────────────────────────────────
POSES["R_WAVE_RAISE"] = _p({51: -72.57, 52: -27.63, 53: 76.75, 54:103 })
POSES["R_WAVE_A"]     = _p({51: -72.57, 52: -27.63, 53: 76.75, 54: 148})
POSES["R_WAVE_B"]     = _p({51: -72.57, 52: -27.63, 53: 76.75, 54: 58})

# ── HANDS UP ───────────────────────────────────────────────────────────────
POSES["HANDS_UP"] = _p({
    41: -112.40,  42:  35.03,  43: -13.58,  44:  0.75,
    51:  110.02,  52: -56.84,  53:   9.80,  54:  0.40,
})

# ── HANDS FRONT ────────────────────────────────────────────────────────────
POSES["HANDS_FRONT"] = _p({
    41: -32.84,  42:  74.59,  43:  11.12,  44:  0.75,
    51:  28.44,  52: -100.53, 53:   3.84,  54:  2.04,
})
# ── HEAD TURNS ─────────────────────────────────────────────────────────────

POSES["HEAD_LEFT"]  = _p({36:  45.0})
POSES["HEAD_RIGHT"] = _p({36: -45.0})

# ── SALUTE ─────────────────────────────────────────────────────────────────
POSES["SALUTE"] = _p({
    41: -56.48,   # l_shoulder_y
    42:   8.31,   # l_shoulder_x
    43:  59.21,   # l_arm_z
    44: -135.52,  # l_elbow_y
})

# ── WALK KEYFRAMES ─────────────────────────────────────────────────────────
POSES["WALK_L_UP"] = {
    11:  -1.10,  12:   4.79,  13: -26.24,  14:  50.42,  15: -11.74,
    21: -23.16,  22:   1.98,  23: -11.21,  24:   3.74,  25:  -7.52,
    31: -12.09,  32:   6.73,  33: -11.56,  34: -12.53,  35:  -1.36,
    36:  -8.36,  37:  -3.96,
    41:  50.51,  42:  66.42,  43: 104.75,  44:  -8.04,
    51: -73.54,  52: -83.21,  53: -93.67,  54:   3.21,
}
POSES["WALK_L_DN"] = {
    11:  11.47,  12: -12.35,  13:  -3.91,  14:  28.35,  15:  -5.14,
    21: -10.68,  22:  -6.29,  23:  -0.57,  24: -37.32,  25:  19.47,
    31:  -6.37,  32: -14.64,  33:   8.22,  34: -27.21,  35:  10.68,
    36: -14.22,  37:  -3.96,
    41:  49.45,  42:  68.09,  43:  20.79,  44: -31.08,
    51: -61.05,  52: -93.67,  53: -94.02,  54:   3.12,
}
POSES["WALK_R_UP"] = {
    11:   6.29,  12: -22.02,  13:   6.20,  14:  16.57,  15: -21.32,
    21: -15.08,  22: -11.65,  23:  35.56,  24: -46.99,  25:  -9.36,
    31:  -3.74,  32:  -6.46,  33:   7.43,  34: -11.82,  35:  -4.62,
    36:   4.25,  37:  -8.94,
    41:  66.51,  42:  72.84,  43:  15.08,  44:   0.66,
    51: -72.66,  52: -87.60,  53:  -7.43,  54:  19.82,
}
POSES["WALK_R_DN"] = {
    11: -12.35,  12:  -5.58,  13:  33.10,  14:  13.32,  15: -25.01,
    21: -38.64,  22:   4.79,  23:   1.45,  24: -30.55,  25:   7.87,
    31: -19.56,  32:  17.10,  33: -11.21,  34: -18.95,  35:  -5.14,
    36:  25.07,  37:  -5.13,
    41:  64.48,  42:  70.81,  43:  14.99,  44:  -6.02,
    51: -70.11,  52: -86.11,  53: -29.76,  54:  10.86,
}

WALK_CYCLE = ["WALK_L_UP", "WALK_L_DN", "WALK_R_UP", "WALK_R_DN"]


# ══════════════════════════════════════════════════════════════════════════════
#  SANITY CHECK
# ══════════════════════════════════════════════════════════════════════════════

def _check_poses() -> None:
    errors = []
    for pname, pose in POSES.items():
        for mid, val in pose.items():
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            if not (lo <= val <= hi):
                errors.append(
                    f"  {pname}[{mid} {MOTOR_MAP[mid][0]}] = {val}  "
                    f"OUT OF [{lo}, {hi}]"
                )
    if errors:
        print("[POSE CHECK] CLIPPING WARNINGS:")
        for e in errors: print(e)
    else:
        print("[POSE CHECK] All poses within limits -- OK")

_check_poses()


# ══════════════════════════════════════════════════════════════════════════════
#  TIMING
# ══════════════════════════════════════════════════════════════════════════════

BASE_STEP_T: Dict[str, float] = {   # ← FIX
    "BALANCED_STAND": 0.60,
    "RIGHT_WAVE":     0.20,
    "LEFT_WAVE":      0.20,
    "HANDS_UP":       0.65,
    "HANDS_FRONT":    0.55,
    "HEAD_LEFT":      0.38,
    "HEAD_RIGHT":     0.38,
    "WALK":           0.22,
    "SALUTE":         0.55,
}

GESTURE_HOLD_T: Dict[str, float] = {   # ← FIX
    "HANDS_UP":       1.50,
    "HANDS_FRONT":    1.20,
    "HEAD_LEFT":      0.90,
    "HEAD_RIGHT":     0.90,
    "BALANCED_STAND": 0.00,
    "SALUTE":         1.20,
}

INTER_MOVE_PAUSE: float = 0.40   # pause between motion segments
WAVE_PEAK_PAUSE:  float = 0.35   # pause at each wave peak
SPEED_SCALE:      float = 1.0    # global speed multiplier


# ══════════════════════════════════════════════════════════════════════════════
#  FINGER COUNT DETECTOR  (MediaPipe Hands)
# ══════════════════════════════════════════════════════════════════════════════

# Finger-count → gesture mapping
FINGER_GESTURE_MAP: Dict[int, str] = {   # ← FIX
    0: "BALANCED_STAND",   # fist
    1: "RIGHT_WAVE",       # index only
    2: "LEFT_WAVE",        # peace sign
    3: "HANDS_UP",         # three fingers
    4: "HANDS_FRONT",      # four fingers
    5: "WALK",             # open palm
    8: "SALUTE",           # eight fingers (two hands: 5+3 or 4+4)
}

# Gestures that loop until stopped by fist — walk and waves
LOOPING_GESTURES = {"WALK", "RIGHT_WAVE", "LEFT_WAVE"}


class FingerDetector:
    """
    Counts extended fingers using MediaPipe Hands.
    Supports 1 or 2 hands. Two-hand total enables counts 6-10.

    Detection rule per finger:
      Thumb : tip.x vs ip.x (direction flipped for left hand)
      Others: tip.y < pip.y  (tip above pip = extended)

    Debounce: COUNT_CONFIRM_FRAMES stable frames before gesture fires.
    Fist (0) fires immediately to allow instant abort.
    """

    COUNT_CONFIRM_FRAMES = 7
    # How many wave oscillations before auto-returning to stand
    WAVE_MAX_CYCLES = 3

    def __init__(self) -> None:
        self._hands = mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.70,
            min_tracking_confidence=0.65,
        )
        self._buf: collections.deque = collections.deque(
            maxlen=self.COUNT_CONFIRM_FRAMES)
        self.confirmed_count: Optional[int] = None       # ← FIX
        self.confirmed_gesture: Optional[str] = None     # ← FIX
        self._raw_count: Optional[int] = None            # ← FIX

    def update(self, rgb_frame) -> Tuple[Optional[int], Optional[str]]:  # ← FIX
        """Returns (raw_count, confirmed_gesture_or_None)."""
        result = self._hands.process(rgb_frame)
        raw = self._count_fingers(result)
        self._raw_count = raw

        if raw is None:
            self._buf.clear()
            return None, None

        # Fist fires immediately — instant abort
        if raw == 0:
            self.confirmed_count   = 0
            self.confirmed_gesture = "BALANCED_STAND"
            self._buf.clear()
            return raw, "BALANCED_STAND"

        self._buf.append(raw)
        if (len(self._buf) == self.COUNT_CONFIRM_FRAMES
                and len(set(self._buf)) == 1):
            gesture = FINGER_GESTURE_MAP.get(raw)
            if gesture:
                self.confirmed_count   = raw
                self.confirmed_gesture = gesture
                return raw, gesture

        return raw, None

    @property
    def raw_count(self) -> Optional[int]:   # ← FIX
        return self._raw_count

    def _count_fingers(self, result) -> Optional[int]:   # ← FIX
        if not result.multi_hand_landmarks:
            return None
        total = 0
        for hl, hinfo in zip(
            result.multi_hand_landmarks,
            result.multi_handedness
        ):
            total += self._count_one_hand(hl, hinfo)
        return total

    @staticmethod
    def _count_one_hand(hand_lm, hand_info) -> int:
        lm     = hand_lm.landmark
        H      = mp.solutions.hands.HandLandmark
        is_right = hand_info.classification[0].label == "Right"
        count  = 0

        # Thumb — compare x positions (mirrored for left hand)
        if is_right:
            if lm[H.THUMB_TIP].x < lm[H.THUMB_IP].x:
                count += 1
        else:
            if lm[H.THUMB_TIP].x > lm[H.THUMB_IP].x:
                count += 1

        # Four fingers — tip above pip means extended
        for tip, pip in [
            (H.INDEX_FINGER_TIP,  H.INDEX_FINGER_PIP),
            (H.MIDDLE_FINGER_TIP, H.MIDDLE_FINGER_PIP),
            (H.RING_FINGER_TIP,   H.RING_FINGER_PIP),
            (H.PINKY_TIP,         H.PINKY_PIP),
        ]:
            if lm[tip].y < lm[pip].y:
                count += 1

        return count

    def close(self) -> None:
        self._hands.close()


# ══════════════════════════════════════════════════════════════════════════════
#  HEAD TRACKER  (MediaPipe Pose)
# ══════════════════════════════════════════════════════════════════════════════

class HeadTracker:
    """
    Continuously tracks nose position vs shoulder midpoint.
    Only used when no finger gesture is active.
    Returns "HEAD_LEFT", "HEAD_RIGHT", or None (centred).
    """

    HEAD_THRESH    = 0.060   # nose x-offset to trigger head turn
    CONFIRM_FRAMES = 6

    def __init__(self) -> None:
        self._buf: collections.deque = collections.deque(
            maxlen=self.CONFIRM_FRAMES)
        self.confirmed: Optional[str] = None   # ← FIX

    def update(self, landmarks) -> Optional[str]:   # ← FIX
        if landmarks is None:
            self._buf.clear()
            return None

        PL = mp.solutions.pose.PoseLandmark
        lm = landmarks.landmark
        nose  = lm[PL.NOSE]
        l_sh  = lm[PL.LEFT_SHOULDER]
        r_sh  = lm[PL.RIGHT_SHOULDER]
        sh_mid_x = (l_sh.x + r_sh.x) / 2.0

        offset = nose.x - sh_mid_x
        if offset > self.HEAD_THRESH:
            raw = "HEAD_RIGHT"
        elif offset < -self.HEAD_THRESH:
            raw = "HEAD_LEFT"
        else:
            raw = None

        self._buf.append(raw)
        if (len(self._buf) == self.CONFIRM_FRAMES
                and len(set(self._buf)) == 1):
            self.confirmed = raw
        elif raw is None:
            self.confirmed = None

        return self.confirmed


# ══════════════════════════════════════════════════════════════════════════════
#  DXL INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

class DxlInterface:
    MOVING_SPEED_RPM = 50

    def __init__(self, port: str, baudrate: int, ids: List[int]) -> None:   # ← FIX
        self._io = DxlIO(port, baudrate=baudrate)
        print(f"[DXL] Opened {port} @ {baudrate} baud")
        found   = set(self._io.scan(ids))
        missing = set(ids) - found
        if missing:
            print(f"[DXL] WARNING -- missing: {sorted(missing)}")
        self._active = found
        print(f"[DXL] Active ({len(found)}): {sorted(found)}")
        self._io.set_moving_speed({m: self.MOVING_SPEED_RPM for m in self._active})
        self._io.enable_torque(list(self._active))
        print("[DXL] Torque ON")

    def set_positions(self, angles: Dict[int, float]) -> None:   # ← FIX
        payload = {
            mid: float(np.clip(val, MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]))
            for mid, val in angles.items()
            if mid in self._active
        }
        if payload:
            try:
                self._io.set_goal_position(payload)
            except Exception as e:
                print(f"[DXL] err: {e}")

    def safe_shutdown(self) -> None:
        try:
            self._io.disable_torque(list(self._active))
            print("[DXL] Torque OFF")
        except Exception:
            pass
        try:
            self._io.close()
            print("[DXL] Port closed")
        except Exception:
            pass


def connect_dxl() -> Optional["DxlInterface"]:   # ← FIX
    if not DXLIO_AVAILABLE:
        return None
    try:
        return DxlInterface(PORT, BAUDRATE, MOTOR_IDS)
    except Exception as e:
        print(f"[WARN] DxlIO failed: {e} -- SIMULATION mode")
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  MOTION CONTROLLER  v10.0
# ══════════════════════════════════════════════════════════════════════════════

_SEQ: Dict[str, List[str]] = {   # ← FIX
    "BALANCED_STAND": ["BALANCED_STAND"],
    "RIGHT_WAVE":     ["R_WAVE_RAISE", "R_WAVE_A", "R_WAVE_B"],
    "LEFT_WAVE":      ["L_WAVE_RAISE", "L_WAVE_A", "L_WAVE_B"],
    "HANDS_UP":       ["HANDS_UP"],
    "HANDS_FRONT":    ["HANDS_FRONT"],
    "HEAD_LEFT":      ["HEAD_LEFT"],
    "HEAD_RIGHT":     ["HEAD_RIGHT"],
    "WALK":           WALK_CYCLE,
    "SALUTE":         ["SALUTE"],
}


class MotionController:
    """
    Executes gestures one at a time.
    After every non-looping gesture completes, automatically returns to
    BALANCED_STAND and sets ready=True to signal it awaits the next command.

    Looping gestures (WALK, waves) run until a FIST (0) is detected,
    which stops them and returns to stand.

    Head gestures (HEAD_LEFT/RIGHT) are lightweight — they move the head
    motor, hold briefly, then return. Head tracking fires these continuously
    while the head is turned.
    """

    DRIVE_HZ = 50

    def __init__(self, dxl: Optional["DxlInterface"]) -> None:   # ← FIX
        self._dxl   = dxl
        self._lock  = threading.Lock()
        self._cur:  Dict[int, float] = POSES["BALANCED_STAND"].copy()   # ← FIX
        self._tgt:  Dict[int, float] = POSES["BALANCED_STAND"].copy()   # ← FIX

        self._gesture  = "BALANCED_STAND"
        self._gthread: Optional[threading.Thread] = None   # ← FIX
        self._stop     = threading.Event()
        self._alpha    = 0.40
        self._walk_spd = 1.0

        # Public state flags
        self._executing  = False
        self._ready      = True    # True = waiting for next command
        self._phase      = "idle"  # for HUD

        threading.Thread(target=self._drive_loop, daemon=True).start()

        if self._dxl:
            self._dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.0)

    # ── public ───────────────────────────────────────────────────────────────

    @property
    def ready(self) -> bool:
        """True when robot is at stand and waiting for the next command."""
        with self._lock:
            return self._ready

    @property
    def current_gesture(self) -> str:
        with self._lock:
            return self._gesture

    @property
    def phase(self) -> str:
        with self._lock:
            return self._phase

    @property
    def is_executing(self) -> bool:
        with self._lock:
            return self._executing

    @property
    def current_angles(self) -> Dict[int, float]:   # ← FIX
        with self._lock:
            return self._cur.copy()

    @property
    def walk_speed(self) -> float:
        return self._walk_spd

    def request_gesture(self, g: str) -> bool:
        """
        Request a new gesture. Only accepted when:
          - robot is ready (at stand, waiting), OR
          - it is a BALANCED_STAND request (fist abort), OR
          - it is a head gesture (always interrupts head-only moves)
        Returns True if accepted.
        """
        with self._lock:
            ready      = self._ready
            cur        = self._gesture
            executing  = self._executing

        # Fist always stops everything
        if g == "BALANCED_STAND":
            self._abort_and_stand()
            return True

        # Head gestures: only replace other head gestures or when ready
        if g in ("HEAD_LEFT", "HEAD_RIGHT"):
            if ready or cur in ("HEAD_LEFT", "HEAD_RIGHT", "BALANCED_STAND"):
                self._launch(g)
                return True
            return False

        # All other gestures: only when ready
        if not ready:
            return False

        self._launch(g)
        return True

    def set_walk_speed(self, delta: float) -> None:
        self._walk_spd = float(np.clip(self._walk_spd + delta, 0.3, 3.0))
        print(f"[WALK] speed = {self._walk_spd:.1f} steps/s")

    def adjust_wave_pause(self, delta: float) -> None:
        global WAVE_PEAK_PAUSE
        WAVE_PEAK_PAUSE = float(
            np.clip(WAVE_PEAK_PAUSE + delta, 0.05, 1.5))
        print(f"[WAVE] peak pause = {WAVE_PEAK_PAUSE:.2f}s")

    def adjust_speed_scale(self, delta: float) -> None:
        global SPEED_SCALE
        SPEED_SCALE = float(np.clip(SPEED_SCALE + delta, 0.3, 3.0))
        print(f"[SPEED] scale = {SPEED_SCALE:.2f}x")

    def force_stand(self) -> None:
        self._abort_and_stand()

    def shutdown(self) -> None:
        self._stop.set()
        time.sleep(0.4)
        if self._dxl:
            self._dxl.safe_shutdown()

    # ── internal ─────────────────────────────────────────────────────────────

    def _abort_and_stand(self) -> None:
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=2.0)
        self._stop.clear()
        with self._lock:
            self._gesture   = "BALANCED_STAND"
            self._executing = False
            self._ready     = True
            self._phase     = "idle"
            self._tgt       = POSES["BALANCED_STAND"].copy()

    def _launch(self, g: str) -> None:
        self._stop.set()
        if self._gthread and self._gthread.is_alive():
            self._gthread.join(timeout=2.0)
        self._stop.clear()
        with self._lock:
            self._gesture   = g
            self._executing = True
            self._ready     = False
            self._phase     = "executing"
        self._gthread = threading.Thread(
            target=self._run, args=(g,), daemon=True)
        self._gthread.start()

    def _run(self, gesture: str) -> None:
        frames    = _SEQ.get(gesture, ["BALANCED_STAND"])
        looping   = gesture in LOOPING_GESTURES
        step_t    = BASE_STEP_T.get(gesture, 0.40)
        hold_t    = GESTURE_HOLD_T.get(gesture, 0.0)
        is_wave   = gesture in ("RIGHT_WAVE", "LEFT_WAVE")
        is_head   = gesture in ("HEAD_LEFT", "HEAD_RIGHT")

        try:
            # ── WALK ─────────────────────────────────────────────────────────
            if gesture == "WALK":
                with self._lock:
                    self._phase = "walking"
                while not self._stop.is_set():
                    wt = 1.0 / self._walk_spd
                    for name in frames:
                        if self._stop.is_set():
                            break
                        self._lerp(POSES[name], wt)

            # ── WAVE ──────────────────────────────────────────────────────────
            elif is_wave:
                with self._lock:
                    self._phase = "raising"
                self._lerp(POSES[frames[0]], step_t * SPEED_SCALE)
                if self._stop.is_set():
                    return
                if not self._interruptible_sleep(INTER_MOVE_PAUSE):
                    return
                loop_frames = frames[1:]
                with self._lock:
                    self._phase = "waving"
                while not self._stop.is_set():
                    for name in loop_frames:
                        if self._stop.is_set():
                            break
                        self._lerp(POSES[name], step_t * SPEED_SCALE)
                        if not self._stop.is_set():
                            self._interruptible_sleep(WAVE_PEAK_PAUSE)

            # ── HEAD ──────────────────────────────────────────────────────────
            elif is_head:
                with self._lock:
                    self._phase = "head turn"
                self._lerp(POSES[frames[0]], step_t * SPEED_SCALE)
                if self._stop.is_set():
                    return
                if hold_t > 0.0:
                    with self._lock:
                        self._phase = "holding"
                    if not self._interruptible_sleep(hold_t):
                        return
                # Return to stand head position
                with self._lock:
                    self._phase = "returning"
                self._lerp(POSES["BALANCED_STAND"],
                            BASE_STEP_T["BALANCED_STAND"] * SPEED_SCALE)

            # ── OTHER NON-LOOPING ─────────────────────────────────────────────
            else:
                with self._lock:
                    self._phase = "executing"
                for idx, name in enumerate(frames):
                    if self._stop.is_set():
                        return
                    self._lerp(POSES[name], step_t * SPEED_SCALE)
                    if idx < len(frames) - 1:
                        if not self._interruptible_sleep(INTER_MOVE_PAUSE):
                            return

                if self._stop.is_set():
                    return

                if hold_t > 0.0:
                    with self._lock:
                        self._phase = "holding"
                    if not self._interruptible_sleep(hold_t):
                        return

                if self._stop.is_set():
                    return

                # Pause before returning
                if not self._interruptible_sleep(INTER_MOVE_PAUSE):
                    return

                # Return to balanced stand
                with self._lock:
                    self._phase = "returning"
                self._lerp(POSES["BALANCED_STAND"],
                            BASE_STEP_T["BALANCED_STAND"] * SPEED_SCALE)

        finally:
            # Mark robot ready only if we weren't externally aborted
            with self._lock:
                self._executing = False
                if not self._stop.is_set():
                    self._gesture = "BALANCED_STAND"
                    self._ready   = True
                    self._phase   = "idle"
                    self._tgt     = POSES["BALANCED_STAND"].copy()

    # ── motion helpers ────────────────────────────────────────────────────────

    def _interruptible_sleep(self, duration: float,
                              granularity: float = 0.05) -> bool:
        deadline = time.monotonic() + duration
        while time.monotonic() < deadline:
            remaining = deadline - time.monotonic()
            if self._stop.wait(
                    timeout=min(granularity, max(remaining, 0.0))):
                return False
        return True

    def _lerp(self, target: Dict[int, float], step_t: float) -> None:   # ← FIX
        steps = max(int(step_t * self.DRIVE_HZ), 2)
        with self._lock:
            start = self._cur.copy()
        for i in range(1, steps + 1):
            if self._stop.is_set():
                return
            t = _smooth(i / steps)
            with self._lock:
                self._tgt = {
                    mid: start[mid] + t * (target[mid] - start[mid])
                    for mid in target
                }
            time.sleep(1.0 / self.DRIVE_HZ)

    def _drive_loop(self) -> None:
        dt = 1.0 / self.DRIVE_HZ
        while True:
            with self._lock:
                tgt = self._tgt.copy()
                cur = self._cur.copy()
            blended = {
                mid: cur[mid] + self._alpha * (tgt[mid] - cur[mid])
                for mid in tgt
            }
            with self._lock:
                self._cur = blended
            if self._dxl:
                self._dxl.set_positions(blended)
            time.sleep(dt)


def _smooth(t: float) -> float:
    return t * t * (3.0 - 2.0 * t)


# ══════════════════════════════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════════════════════════════

GESTURE_COLORS = {
    "BALANCED_STAND": (40,  160,  60),
    "RIGHT_WAVE":     (60,  210, 255),
    "LEFT_WAVE":      (60,  210, 255),
    "HANDS_UP":       (60,  255, 180),
    "HANDS_FRONT":    (60,  255, 120),
    "HEAD_LEFT":      (200, 140,  60),
    "HEAD_RIGHT":     (200, 140,  60),
    "WALK":           (255, 160,   0),
    "SALUTE":         (220, 180,  60),
}

_GROUPS = [
    ("LEFT  LEG",  [11, 12, 13, 14, 15]),
    ("RIGHT LEG",  [21, 22, 23, 24, 25]),
    ("TORSO",      [31, 32, 33, 34, 35]),
    ("HEAD",       [36, 37]),
    ("LEFT  ARM",  [41, 42, 43, 44]),
    ("RIGHT ARM",  [51, 52, 53, 54]),
]

_FINGER_GUIDE = [
    ("0", "Fist",        "Balanced stand  [abort]"),
    ("1", "Index",       "Right wave"),
    ("2", "Peace",       "Left wave"),
    ("3", "Three",       "Hands up"),
    ("4", "Four",        "Hands front"),
    ("5", "Open palm",   "Walk (fist to stop)"),
    ("8", "Eight",       "Salute"),
    ("-", "Head turn",   "Head L / R  (pose tracking)"),
]


def draw_hud(angles, gesture, phase, live, walk_spd,
             executing, ready, raw_count, head_dir,
             wave_pause, speed_scale, frame=None):
    W, H = 520, 730
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:] = (14, 14, 24)

    # Title bar
    col = GESTURE_COLORS.get(gesture, (100, 100, 100))
    cv2.rectangle(canvas, (0, 0), (W, 46), col, -1)
    mode     = "DXL LIVE" if live else "SIMULATION"
    spd_str  = f"  {walk_spd:.1f}/s" if gesture == "WALK" else ""
    state_lbl = "READY" if ready else phase.upper()
    cv2.putText(canvas, f"{mode}  {gesture}{spd_str}  [{state_lbl}]",
                (8, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (10, 10, 10), 2)

    # Raw detection line
    cnt_str = str(raw_count) if raw_count is not None else "no hand"
    head_str = f"  head={head_dir}" if head_dir else ""
    cv2.putText(canvas,
                f"fingers detected: {cnt_str}{head_str}",
                (8, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (150, 150, 150), 1)

    # Ready banner
    if ready:
        cv2.rectangle(canvas, (0, 64), (W, 84), (20, 80, 20), -1)
        cv2.putText(canvas, ">>> READY — show fingers for next command <<<",
                    (8, 78), cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                    (80, 255, 80), 1)

    # Motor bars
    y = 92
    for grp, mids in _GROUPS:
        cv2.putText(canvas, grp, (8, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.37, (110, 170, 255), 1)
        y += 15
        for mid in mids:
            label  = MOTOR_MAP[mid][0]
            val    = angles.get(mid, 0.0)
            lo, hi = MOTOR_MAP[mid][1], MOTOR_MAP[mid][2]
            bw     = max(int(np.interp(val, [lo, hi], [0, W - 124])), 0)
            cv2.rectangle(canvas, (104, y - 10), (104 + bw, y + 2),
                          (45, 165, 90), -1)
            cv2.putText(canvas, f"{label:<14} {val:+7.2f}",
                        (4, y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.34, (210, 210, 210), 1)
            y += 16
        y += 4

    # Finger guide
    y += 4
    cv2.putText(canvas, "FINGER COMMAND GUIDE", (8, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.36, (80, 180, 255), 1)
    y += 14
    for fnum, fname, fdesc in _FINGER_GUIDE:
        cur_count = raw_count if raw_count is not None else -1
        active = False
        if fnum.isdigit() and int(fnum) == cur_count:
            active = True
        elif fnum == "-" and head_dir:
            active = True
        txt_col = (255, 240, 80) if active else (150, 150, 150)
        cv2.putText(canvas,
                    f"  [{fnum}] {fname:<12}  {fdesc}",
                    (4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.30, txt_col, 1)
        y += 13

    # Settings strip
    sy = H - 42
    cv2.rectangle(canvas, (0, sy), (W, H), (22, 22, 38), -1)
    cv2.putText(canvas,
                f"Wave pause {wave_pause:.2f}s [/]  "
                f"Speed {speed_scale:.2f}x ,/.  "
                f"Walk {walk_spd:.1f}/s +/-",
                (6, sy + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.31,
                (180, 220, 255), 1)
    cv2.putText(canvas,
                "S force stand    Q quit",
                (6, sy + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.31,
                (200, 200, 200), 1)

    # Camera thumbnail
    if frame is not None:
        th = cv2.resize(frame, (120, 90))
        canvas[sy - 95:sy - 5, W - 125:W - 5] = th

    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    dxl  = connect_dxl()
    live = dxl is not None

    # Pose model (for head tracking only)
    mp_pose    = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    pose_model = mp_pose.Pose(
        model_complexity=1,
        min_detection_confidence=0.60,
        min_tracking_confidence=0.55,
    )

    finger_det = FingerDetector()
    head_track = HeadTracker()
    controller = MotionController(dxl=dxl)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam")
        controller.shutdown()
        sys.exit(1)

    print("\n[INFO] Running — show hand to camera (~0.5–1 m)")
    print("  0 fist = stand/abort")
    print("  1-5, 8 fingers = commands (see HUD)")
    print("  head turn = head control (continuous via pose)")
    print("  S force stand   Q quit\n")

    prev_t     = time.time()
    # Track last fired head gesture to avoid re-firing same direction
    last_head_fired: Optional[str] = None   # ← FIX

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame     = cv2.flip(frame, 1)
            rgb       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            pose_res  = pose_model.process(rgb)
            rgb.flags.writeable = True
            annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            # ── Finger detection ──────────────────────────────────────────────
            rgb_hands = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            raw_count, confirmed_gesture = finger_det.update(rgb_hands)

            # ── Head tracking ─────────────────────────────────────────────────
            head_dir = head_track.update(pose_res.pose_landmarks)

            # ── Command routing ───────────────────────────────────────────────
            accepted = False

            if confirmed_gesture is not None:
                # Finger command — send to controller
                accepted = controller.request_gesture(confirmed_gesture)
                if accepted and confirmed_gesture not in ("HEAD_LEFT", "HEAD_RIGHT"):
                    last_head_fired = None  # reset head tracking on new command

            elif head_dir is not None and controller.ready:
                # Head tracking only fires when robot is ready and
                # the direction has changed (avoid re-firing same side)
                if head_dir != last_head_fired:
                    accepted = controller.request_gesture(head_dir)
                    if accepted:
                        last_head_fired = head_dir
            elif head_dir is None:
                last_head_fired = None

            # ── Pose skeleton overlay ─────────────────────────────────────────
            if pose_res.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated, pose_res.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(
                        color=(0, 200, 80), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(
                        color=(180, 180, 0), thickness=2),
                )

            # ── Camera feed overlays ──────────────────────────────────────────
            now    = time.time()
            fps    = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now

            cv2.putText(annotated, f"FPS {fps:.0f}", (8, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 200), 2)

            cur_gesture = controller.current_gesture
            g_col = GESTURE_COLORS.get(cur_gesture, (200, 200, 200))
            cv2.putText(annotated, f"{cur_gesture}",
                        (8, annotated.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, g_col, 2)

            phase_lbl = controller.phase.upper()
            cv2.putText(annotated, phase_lbl,
                        (8, annotated.shape[0] - 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.46, (130, 130, 200), 2)

            cnt_lbl = f"fingers: {raw_count}" if raw_count is not None \
                else "no hand"
            cv2.putText(annotated, cnt_lbl,
                        (8, annotated.shape[0] - 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.46, (200, 200, 200), 1)

            if controller.ready:
                cv2.putText(annotated, "READY",
                            (annotated.shape[1] - 110, 26),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (40, 255, 80), 2)

            if head_dir:
                cv2.putText(annotated, f"head: {head_dir}",
                            (8, annotated.shape[0] - 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.40,
                            (200, 140, 60), 1)

            # ── HUD ───────────────────────────────────────────────────────────
            hud = draw_hud(
                controller.current_angles,
                cur_gesture,
                controller.phase,
                live,
                controller.walk_speed,
                controller.is_executing,
                controller.ready,
                raw_count,
                head_dir,
                WAVE_PEAK_PAUSE,
                SPEED_SCALE,
                frame,
            )

            cv2.imshow("Gesture Input", annotated)
            cv2.imshow("Robot State",   hud)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("s"):
                controller.force_stand()
                print("[KEY] Forced stand")
            elif key in (ord("+"), ord("=")):
                controller.set_walk_speed(+0.2)
            elif key == ord("-"):
                controller.set_walk_speed(-0.2)
            elif key == ord("["):
                controller.adjust_wave_pause(+0.05)
            elif key == ord("]"):
                controller.adjust_wave_pause(-0.05)
            elif key == ord(","):
                controller.adjust_speed_scale(+0.1)
            elif key == ord("."):
                controller.adjust_speed_scale(-0.1)

    except KeyboardInterrupt:
        pass

    finally:
        print("[INFO] Returning to stand...")
        cap.release()
        cv2.destroyAllWindows()
        pose_model.close()
        finger_det.close()
        controller.force_stand()
        if live:
            dxl.set_positions(POSES["BALANCED_STAND"])
            time.sleep(1.5)
        controller.shutdown()
        print("[INFO] Done.")


if __name__ == "__main__":
    main()

[POSE CHECK] All poses within limits -- OK
[DXL] Opened COM4 @ 1000000 baud
[DXL] Active (25): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 51, 52, 53, 54]
[DXL] Torque ON

[INFO] Running — show hand to camera (~0.5–1 m)
  0 fist = stand/abort
  1-5, 8 fingers = commands (see HUD)
  head turn = head control (continuous via pose)
  S force stand   Q quit

